In [ ]:

import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
from dask.distributed import Client, wait
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import re
from matplotlib.pyplot import figure


plt.rcParams['axes.grid'] = False


xstart = 800
xstop = 1200
ystart = 800
ystop = 1200

def custom_scaled_heatmap(adata, layer = None, group_by = 'leiden', saveto = False, var_names = False, h = 20, w = 14, style = 'mean'):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    fig = plt.figure(figsize=(h, w))
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    matrix_data[group_by] = adata.obs[group_by]
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
    matrix_data_scaled = matrix_data_bulked/(np.max(matrix_data_bulked, axis=0))
    b = sns.heatmap(matrix_data_scaled, square = True, linewidths=0.06, linecolor='gray', cmap="viridis", cbar = True)

    
    
    plt.tick_params(axis='y', labelrotation=0)
    plt.tick_params(axis='x', labelrotation=90)
    
    b.set_yticklabels(b.get_yticklabels(), size = 17)
    b.set_xticklabels(b.get_xticklabels(), size = 17)
    #b.figure.axes[0].tick_params(axis="both", labelsize=10) 
    b.figure.axes[1].tick_params(axis="x", labelsize=0)

    

    plt.xlabel(None)
    plt.ylabel(None)

    plt.axhline(y=0, color='k',linewidth=1)
    plt.axhline(y=matrix_data_scaled.shape[0], color='k',linewidth=2)
    plt.axvline(x=0, color='k',linewidth=1)
    plt.axvline(x=matrix_data_scaled.shape[1], color='k',linewidth=2)
    plt.tight_layout()
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()
        
        
def custom_scaled_heatmap_Z(adata, layer = None, group_by = 'leiden', saveto = False, var_names = False, h = 20, w = 14, style = 'mean'):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    fig = plt.figure(figsize=(h, w))
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    matrix_data[group_by] = adata.obs[group_by]
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
    matrix_data_scaled = (matrix_data_bulked - np.mean(matrix_data_bulked))/np.std(matrix_data_bulked)
    matrix_data_scaled = matrix_data_scaled.clip(lower=-2, upper=2)
    b = sns.heatmap(matrix_data_scaled, square = True, linewidths=0.06, linecolor='gray', cmap="viridis", cbar = True)

    
    
    plt.tick_params(axis='y', labelrotation=0)
    plt.tick_params(axis='x', labelrotation=90)
    
    b.set_yticklabels(b.get_yticklabels(), size = 17)
    b.set_xticklabels(b.get_xticklabels(), size = 17)
    #b.figure.axes[0].tick_params(axis="both", labelsize=10) 
    b.figure.axes[1].tick_params(axis="x", labelsize=0)

    

    plt.xlabel(None)
    plt.ylabel(None)

    plt.axhline(y=0, color='k',linewidth=1)
    plt.axhline(y=matrix_data_scaled.shape[0], color='k',linewidth=2)
    plt.axvline(x=0, color='k',linewidth=1)
    plt.axvline(x=matrix_data_scaled.shape[1], color='k',linewidth=2)
    plt.tight_layout()
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()


def custom_scaled_heatmap_minmax(adata, layer = None, group_by = 'leiden', saveto = False, var_names = False, h = 20, w = 14, style = 'mean'):
    
    print("set layer equal to 'meta' if your variables are in the metadata")
    
    fig = plt.figure(figsize=(h, w))
    
    if layer == 'meta':
        matrix_data = adata.obs
    else:
        matrix_data = adata.to_df(layer = layer)
    
    if var_names:
        matrix_data = matrix_data[var_names]
    
    matrix_data[group_by] = adata.obs[group_by]
    if style == 'mean':
        matrix_data_bulked = matrix_data.groupby(group_by).mean()
    elif style == 'median':
        matrix_data_bulked = matrix_data.groupby(group_by).median()
    matrix_data_scaled = (matrix_data_bulked - np.min(matrix_data_bulked))/(np.max(matrix_data_bulked) - np.min(matrix_data_bulked))
    b = sns.heatmap(matrix_data_scaled, square = True, linewidths=0.06, linecolor='gray', cmap="viridis", cbar = True)

    
    
    plt.tick_params(axis='y', labelrotation=0)
    plt.tick_params(axis='x', labelrotation=90)
    
    b.set_yticklabels(b.get_yticklabels(), size = 17)
    b.set_xticklabels(b.get_xticklabels(), size = 17)
    #b.figure.axes[0].tick_params(axis="both", labelsize=10) 
    b.figure.axes[1].tick_params(axis="x", labelsize=0)

    

    plt.xlabel(None)
    plt.ylabel(None)

    plt.axhline(y=0, color='k',linewidth=1)
    plt.axhline(y=matrix_data_scaled.shape[0], color='k',linewidth=2)
    plt.axvline(x=0, color='k',linewidth=1)
    plt.axvline(x=matrix_data_scaled.shape[1], color='k',linewidth=2)
    plt.tight_layout()
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()   
        
def pval_print(pval, sci_thresh=1e-3, decimals=3):
    if pval == 0:
        return "0"
    if pval < sci_thresh:
        return f"{pval:.{decimals}e}"
    return f"{pval:.{decimals}f}"
        
import pandas as pd

def tukey_to_df(res, labels):
    

    ci = res.confidence_interval()
    rows = []

    for i in range(len(labels)):
        for j in range(i+1, len(labels)):
            rows.append({
                "group1": labels[i],
                "group2": labels[j],

                # Tukey q statistic
                "statistic": res.statistic[i, j],

                # mean difference
                "mean_diff": res.statistic[i, j],

                # adjusted p-value
                "p_adj": res.pvalue[i, j],

                # confidence intervals
                "ci_low": ci.low[i, j],
                "ci_high": ci.high[i, j]
            })

    return pd.DataFrame(rows)

# load adatas

In [ ]:
data_directory = "sample_adatas/"
adata_list = {}


for filename in os.listdir(data_directory):
    if filename.endswith(".h5ad"):
        file_path = os.path.join(data_directory, filename)
        adata = anndata.read_h5ad(file_path)

        file_name = re.sub('.h5ad', '', filename)
        adata.obs['origfile'] = file_name
        #adata.obs['file_leiden'] = [i + j for i, j in zip(adata.obs['origfile'], adata.obs['leiden'])]
        adata_list[file_name] = adata
        print(adata.var_names)
        print(filename)
        print('IBA1' in adata.var_names)





### fix mistaken names

In [ ]:

    
feature_dict = {
    'Claudin-5':'Claudin5',
    'a-Beta':'A-Beta',
    'DAPI_1':'DAPI'
}
features_to_remove = ['H2A.x', 'SMA - Claudin5 +', 'GFAP + Vimentin +']


# Define the custom order for feature names
custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV','PCNA', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'GFAP', 'DAPI']


    
for adata_name in adata_list.keys():
    adata = adata_list[adata_name]
    
    adata_vars = adata.var_names
    
    for var in adata_vars:
        if var in feature_dict.keys():
            adata.var_names = [feature_dict[name] if name in feature_dict.keys() else name for name in adata.var_names] 



    
    adata.var.drop(features_to_remove, inplace=False)







In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


merged_adata = anndata.concat(adata_list, axis=0, index_unique="-")

#custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'PCNA', 'GFAP', 'DAPI_1']
# Use NumPy's argsort to get the indices that would reorder the features
order_indices = [merged_adata.var_names.get_loc(feature) for feature in custom_feature_order]

# Reorder the features based on the custom order
merged_adata = merged_adata[:, order_indices]

merged_adata.obs['origfilenum'] = merged_adata.obs['origfile'].str.split('_', expand = True)[0]
#merged_adata.obs['AD_CNT'] = merged_adata.obs['origfile'].str.split('_', expand = True)[1]
merged_adata.obs['AD_CNT'] = [ad_cnt_dict[i] for i in merged_adata.obs['origfilenum']]



#sc.pl.violin(merged_adata_GM, 'CollagenIV', groupby='leiden', show=True)


# attach clustering metadata

In [ ]:

# alter the following below to adapt to new methods of clustering
clustering_metadata = anndata.read_h5ad('../final_h5ads/fabian_metadata_plus_clustering.h5ad')




cluster_column = 'leiden'
imageid_cellid_col = 'ImageID_CellID'


change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}




cluster_label_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2', 'default']
cluster_label_ticklabels = cluster_label_order

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs['leiden']]

clustering_metadata.obs['leiden'] = pd.Categorical(clustering_metadata.obs['leiden'], categories=cluster_label_order, ordered=False)

cluster_label_ticklabels.remove('default')







#--------------------------------------------------------------------------------------




mapping_dict = dict(zip(clustering_metadata.obs[imageid_cellid_col], clustering_metadata.obs[cluster_column]))
merged_adata.obs['leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['leiden']


# remove any meningeal macrophages

merged_adata = merged_adata[merged_adata.obs['parent'].isin(['Grey Matter', 'White Matter'])]

# remove any default clusters

merged_adata = merged_adata[merged_adata.obs['leiden'] != 'default']

merged_adata_WM = merged_adata[merged_adata.obs['parent'] == 'White Matter']
merged_adata_GM = merged_adata[merged_adata.obs['parent'] == 'Grey Matter']

merged_adata_WM = merged_adata_WM[:, merged_adata_WM.var_names.isin(['MAP2', 'NeuN']) == False]





merged_adata_sub = merged_adata[merged_adata.obs['id'].isin(clustering_metadata.obs['ImageID_CellID'])]
clustering_metadata.obs.index = clustering_metadata.obs['ImageID_CellID']

clustering_metadata = clustering_metadata[merged_adata_sub.obs['id'], :]

tsne_coordinates = clustering_metadata.obsm["X_tsne"]
merged_adata_sub.obsm['X_tsne'] = tsne_coordinates

#merged_adata.obsm.X_tsne = 

for feat in merged_adata_sub.var_names:
    df = merged_adata_sub.to_df()
    feat_upperbound = np.percentile(a=df[feat], q=[95])

    sc.pl.tsne(merged_adata_sub, color=feat, vmax=feat_upperbound)


# bulk data

In [ ]:
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90)
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_direct_overlap')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_all')


sc.pl.violin(merged_adata_GM, keys='NeuN',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='A-Beta',groupby='leiden', rotation=90)

sc.pl.matrixplot(merged_adata_GM, merged_adata_GM.var_names, groupby='leiden', dendrogram=False, standard_scale='var')


In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}



minmax_df_GM = merged_adata_GM.to_df(layer = 'zscore')
minmax_df_GM['sample_leiden'] = merged_adata_GM.obs['file_leiden']
minmax_df_GM_mean = minmax_df_GM.groupby(by=['sample_leiden']).mean()
minmax_df_GM_mean['sample_leiden'] = minmax_df_GM_mean.index
minmax_df_GM_mean['leiden'] = minmax_df_GM_mean['sample_leiden'].str.split('.', expand = True)[1]
minmax_df_GM_mean['sample'] = minmax_df_GM_mean['sample_leiden'].str.split('.', expand = True)[0]
minmax_df_GM_mean['sample_num'] = minmax_df_GM_mean['sample'].str.split('_', expand = True)[0]
minmax_df_GM_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_GM_mean['sample_num']]


minmax_df_WM = merged_adata_WM.to_df(layer = 'zscore')
minmax_df_WM['sample_leiden'] = merged_adata_WM.obs['file_leiden']
minmax_df_WM_mean = minmax_df_WM.groupby(by=['sample_leiden']).mean()
minmax_df_WM_mean['sample_leiden'] = minmax_df_WM_mean.index
minmax_df_WM_mean['leiden'] = minmax_df_WM_mean['sample_leiden'].str.split('.', expand = True)[1]
minmax_df_WM_mean['sample'] = minmax_df_WM_mean['sample_leiden'].str.split('.', expand = True)[0]
minmax_df_WM_mean['sample_num'] = minmax_df_WM_mean['sample'].str.split('_', expand = True)[0]
minmax_df_WM_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_WM_mean['sample_num']]






## box and whisker ANOVA

In [ ]:




from scipy.stats import f_oneway
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd

import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd



def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    
    fig, ax = plt.subplots()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True, whis=np.inf)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova
    
    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    
    print(anova_p_value)

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]
    print(combinations)
    

    for c in combinations:
        
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])

    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
            
        sig_symbol = sig_symbol + ' ' + pval_print(p)
        
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    else:
        return(fig, anova_f_statistic, anova_p_value, res)
    
    plt.show()
    
    
    
    
    



minmax_df_mean = minmax_df_GM_mean

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
    
    

    

minmax_df_mean = minmax_df_WM_mean

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    









anova_datax and whisker anova split by ad and cnt

### box and whisker anova ad

In [ ]:
minmax_df_mean = minmax_df_GM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter AD'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
    
    


minmax_df_mean = minmax_df_WM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter AD'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
    
    
    
   



### box and whisker anova cnt

In [ ]:
minmax_df_mean = minmax_df_GM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter Control'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
    
    


minmax_df_mean = minmax_df_WM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter Control'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
    
    


## dotplot GM

In [ ]:
minmax_df_mean = minmax_df_GM_mean

frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean[minmax_df_mean['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/GM_all_leiden_heatmap.csv')




def sillyscale(frame):
    m_frame = frame/np.mean(frame)
    m_frame = m_frame - np.min(m_frame)
    
    return(np.clip(m_frame, a_max = 1, a_min = 0))
    #return(np.log2(m_frame + 1))



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
#frame_z = sillyscale(frame)

pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''



# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = frame_z

# Sample additional values for bubble sizes
bubble_sizes = pframe_log.to_numpy() * 100  # Adjust the multiplier for bubble size

# Create a figure and axis
fig, ax = plt.subplots()

# Loop through the DataFrame and create bubbles
for i, row in enumerate(data.index):
    for j, col in enumerate(data.columns):
        value = data.at[row, col]  # Use DataFrame value for color
        size = bubble_sizes[i, j]  # Use additional value for bubble size
        color = plt.cm.viridis(value)  # Use DataFrame value for color
        ax.scatter(j, i, c=[color], s=size, alpha=0.7)

# Customize the plot
ax.set_xticks(np.arange(len(data.columns)))
ax.set_yticks(np.arange(len(data.index)))
ax.set_xticklabels(data.columns, rotation=90)
ax.set_yticklabels(data.index)
#plt.colorbar(label='Color Scale')
plt.title('Sig + Association bubblemap GM')

# Add grid lines
ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)

plt.yticks(plt.yticks()[0], [str.replace(i, 'GM', '') for i in data.index.tolist()])


ax.grid(which="minor", color="black", linestyle='-', linewidth=2)

# Show the plot
plt.show()

## heatmap GM

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])

# Customize the plot
plt.title('Sig + Relative association GM')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')


# Show the plot
plt.savefig('../figures/protein/GM_all_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/protein/GM_all_leiden_heatmap.svg', format='svg')

plt.show()

# heatmap ad and cnt seperate GM

In [ ]:





minmax_df_mean = minmax_df_GM_mean

minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/GM_AD_leiden_heatmap.csv')




frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
#frame_z = sillyscale(frame)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)
pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association GM AD')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/GM_AD_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/protein/GM_AD_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()

#------------------------------------------------------------------------


minmax_df_mean = minmax_df_GM_mean

minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]

minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/GM_Control_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
#frame_z = sillyscale(frame)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)



plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])



# Customize the plot
plt.title('Sig + Relative association GM CNT')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/GM_CONTROL_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/protein/GM_CONTROL_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()






## dotplot WM

In [ ]:
minmax_df_mean = minmax_df_WM_mean

frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean[minmax_df_mean['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/WM_all_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = frame_z

# Sample additional values for bubble sizes
bubble_sizes = pframe_log.to_numpy() * 100  # Adjust the multiplier for bubble size

# Create a figure and axis
fig, ax = plt.subplots()

# Loop through the DataFrame and create bubbles
for i, row in enumerate(data.index):
    for j, col in enumerate(data.columns):
        value = data.at[row, col]  # Use DataFrame value for color
        size = bubble_sizes[i, j]  # Use additional value for bubble size
        color = plt.cm.viridis(value)  # Use DataFrame value for color
        ax.scatter(j, i, c=[color], s=size, alpha=0.7)

# Customize the plot
ax.set_xticks(np.arange(len(data.columns)))
ax.set_yticks(np.arange(len(data.index)))
ax.set_xticklabels(data.columns, rotation=90)
ax.set_yticklabels(data.index)
#plt.colorbar(label='Color Scale')
plt.title('Sig + Association bubblemap WM')



# Add grid lines
ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)
ax.grid(which="minor", color="black", linestyle='-', linewidth=2)


plt.yticks(plt.yticks()[0], [str.replace(i, 'WM', '') for i in data.index.tolist()])


# Show the plot
plt.show()

## heatmap WM

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/WM_all_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/protein/WM_all_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()


# heatmap ad and cnt seperate WM

In [ ]:





minmax_df_mean = minmax_df_WM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/WM_AD_leiden_heatmap.csv')

frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM AD')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/WM_AD_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/protein/WM_AD_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()

#------------------------------------------------------------------------


minmax_df_mean = minmax_df_WM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/WM_Control_leiden_heatmap.csv')

frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)

plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM CNT')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/WM_CONTROL_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/protein/WM_CONTROL_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()






# protein x neighborhood correlations

In [ ]:

protein_data_orig = sc.read_h5ad('../final_h5ads/fabian_metadata_plus_clustering.h5ad')


protein_data_orig_sub = protein_data_orig[protein_data_orig.obs.ImageID_CellID.isin(merged_adata_GM.obs.id)]
merged_adata_GM_sub = merged_adata_GM[merged_adata_GM.obs.id.isin(protein_data_orig_sub.obs.ImageID_CellID)]
merged_adata_GM_sub.obs.index = merged_adata_GM_sub.obs.id.to_list()


matrix1 = protein_data_orig_sub.to_df()[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
matrix2 = merged_adata_GM_sub.to_df()
matrix2 = matrix2.loc[matrix1.index]

correlation_matrix = pd.DataFrame()

# Calculate correlations for all pairs of features
for col1 in matrix1.columns:
    for col2 in matrix2.columns:
        correlation = matrix1[col1].corr(matrix2[col2])
        correlation_matrix.at[col1, col2] = correlation

# Print the correlation matrix
print(correlation_matrix)


plt.figure(figsize=(12, 12))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', square=True)

# Customize the plot
plt.title('Morphology x Neighborhood correlation')
plt.tight_layout()

plt.savefig('../figures/morph_vs_neighborhood_heatmap.tiff')
plt.savefig('../figures/morph_vs_neighborhood_heatmap.svg', format = 'svg')

plt.show()



# morphology x neighborhood correlations

In [ ]:

morph_data_orig = sc.read_h5ad('../final_h5ads/clustering_and_morph_clustering.h5')
protein_data_matrix = morph_data_orig.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_data_matrix = morph_data_orig.to_df()

morph_data_orig_sub = morph_data_orig[morph_data_orig.obs.ImageID_CellID.isin(merged_adata_GM.obs.id)]
merged_adata_GM_sub = merged_adata_GM[merged_adata_GM.obs.id.isin(morph_data_orig_sub.obs.ImageID_CellID)]

merged_adata_GM_sub.obs.index = merged_adata_GM_sub.obs.id.to_list()


matrix1 = morph_data_orig_sub.to_df()
matrix2 = merged_adata_GM_sub.to_df()
matrix2 = matrix2.loc[matrix1.index]

correlation_matrix = pd.DataFrame()

# Calculate correlations for all pairs of features
for col1 in matrix1.columns:
    for col2 in matrix2.columns:
        correlation = matrix1[col1].corr(matrix2[col2])
        correlation_matrix.at[col1, col2] = correlation

# Print the correlation matrix
print(correlation_matrix)


plt.figure(figsize=(12, 12))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', square=True)

# Customize the plot
plt.title('Morphology x Neighborhood correlation')
plt.tight_layout()

plt.savefig('../figures/morph_vs_neighborhood_heatmap.tiff')
plt.savefig('../figures/morph_vs_neighborhood_heatmap.svg', format = 'svg')

plt.show()




## ad vs control bar chart

In [ ]:
import matplotlib.patches as mpatches



minmax_df_mean = minmax_df_GM_mean
ad_samples = [3196, 3155, 2997, 3280]
cnt_samples = [1873, 3628, 3026, 1796]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]


frame = pd.DataFrame()
pframe = pd.DataFrame()


for cluster in set(minmax_df_mean['leiden']):
    
    minmax_df_sub = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
    minmax_df_sub_AD = minmax_df_sub[minmax_df_sub['AD_CNT'] == 'AD']
    minmax_df_sub_CNT = minmax_df_sub[minmax_df_sub['AD_CNT'] == 'CNT']
    
    proteinlogs = []
    pvals = []
    
    proteinvals = []
    conditionvals = []
    protein_realvals = []
    
    for protein in minmax_df_sub.columns[:-5]:
        proteinvals.extend([protein]*8)
        conditionvals.extend(['AD']*4)
        conditionvals.extend(['CONTROL']*4)
        
        
        protein_AD_vals = minmax_df_sub_AD[protein]
        protein_CNT_vals = minmax_df_sub_CNT[protein]
        
        protein_realvals.extend(protein_AD_vals)
        protein_realvals.extend(protein_CNT_vals)
        
        
        # print(protein_AD_vals)
        # print(protein_CNT_vals)
        print(protein)
        U, p = stats.ttest_ind(protein_AD_vals, protein_CNT_vals, alternative='two-sided')
        
        # fc = np.mean(protein_AD_vals)/np.mean(protein_CNT_vals)
        # logfc = np.log2(fc)
        
        logfc = np.mean(protein_AD_vals) - np.mean(protein_CNT_vals)
        
        proteinlogs.append(logfc)
        pvals.append(p)
        
    #HERE
    
    
    # Data for the bar chart
    categories = minmax_df_sub.columns[:-5].tolist()
    
    values = proteinlogs

    # Significance values for each bar (you can replace these with your actual significance values)
    significance_values = pvals

    # Create a figure and axis
    fig, ax = plt.subplots()

    # Create the bar chart
    bars = ax.bar(categories, values, color=['blue' if v >= 0 else 'orange' for v in values])

    # Add values at the end of each bar
    for i, bar in enumerate(bars):
        significance_value = significance_values[i]
        if values[i] >= 0:
            ax.text(bar.get_x() + bar.get_width()/2, values[i] + 0.05, f'{significance_value:.3f}', va = 'center', ha = 'center', size = 7)
        else:
            ax.text(bar.get_x() + bar.get_width()/2, values[i] - 0.05, f'{significance_value:.3f}',  va = 'center', ha = 'center', size = 7)


            
    # Set the y-axis range from -1 to 1
    ax.set_xticklabels(categories, rotation=90)

    # Add labels and title
    ax.set_xlabel('Categories')
    ax.set_ylabel('Values')
    ax.set_title('Gray Matter ' + cluster)
    
    legend_patches = [mpatches.Patch(color='blue', label='Greater in AD'), mpatches.Patch(color='orange', label='Greater in CONTROL')]
    plt.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1, 1))
    


    # Show the plot
    plt.tight_layout()
    plt.show()
    
    #HERE

    # Sample data with more points
    np.random.seed(0)
    data = {
        'Protein': proteinvals,
        'Condition': conditionvals,
        'Values': protein_realvals
    }

    # Create a DataFrame
    df = pd.DataFrame(data)

    # Set the figure size
    plt.figure(figsize=(10, 6))

    # Create a grouped dot plot
    #sns.stripplot(x='Protein', y='Values', hue='Condition', data=df, jitter=False, dodge=True, palette='Set3', size=10)
    #sns.stripplot(x='Protein', y='Values', hue='Condition', data=df, jitter=False, dodge=True, size=10)
    sns.boxplot(x = 'Protein', y = 'Values', hue = 'Condition', data = df, dodge = True)

    
    # Customize the plot
    plt.xlabel('Protein')
    plt.ylabel('Values')
    plt.title('Gray Matter ' + cluster)
    
    
    
    plt.xticks(rotation=90)

    # Show the plot
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.legend(title='Condition', loc='upper right')
    plt.show()

    
    
    



        
        
        
     














# pseudotime inference MATRIX

## all cells

In [ ]:
import networkx
#import elpigraph
#import scFates as scf
import scFates as scf


path_to_subclustering = '../final_h5ads/fabian_metadata_plus_clustering.h5ad'
subclustering_obj_1 = anndata.read_h5ad(path_to_subclustering)


# subset and recalculate pca

proteins_to_cluster_with = ['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']
subclustering_obj_1 = subclustering_obj_1[:,proteins_to_cluster_with]

print(sc.pl.tsne(subclustering_obj_1, color=['leiden']))

print(np.unique(subclustering_obj_1.obs.ImageID))


print(np.unique(subclustering_obj_1.obs.ImageID))

print(sc.pl.tsne(subclustering_obj_1, color=['leiden']))



#sc.pp.pca(subclustering_obj_1)

#https://scfates.readthedocs.io/en/latest/Basic_Curved_trajectory_analysis.html


scf.tl.curve(subclustering_obj_1,Nodes=15,use_rep="X",ndims_rep=6, epg_lambda = 0.5)

scf.pl.graph(subclustering_obj_1,basis="tsne", show = False)
plt.savefig('../figures/protein/protein_stepwise_trajectory.tiff', format='tiff')
plt.savefig('../figures/protein/protein_stepwise_trajectory.svg', format='svg')


scf.tl.root(subclustering_obj_1,2)

scf.tl.pseudotime(subclustering_obj_1,n_jobs=20,n_map=100,seed=42)

sc.pl.tsne(subclustering_obj_1,color="t")

sc.pl.tsne(subclustering_obj_1, color=subclustering_obj_1.var_names,use_raw=False)


In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
sns.set(font="Arial")

sns.set_style('white')
plt.rcParams['axes.grid'] = False







scf.pl.trajectory(subclustering_obj_1,basis="tsne",arrows=False,arrow_offset=3, show = False)
plt.savefig('../figures/protein/protein_trajectory.tiff', format='tiff')
plt.savefig('../figures/protein/protein_trajectory.svg', format='svg')


In [ ]:
sc.pl.tsne(subclustering_obj_1,color="milestones")

scf.tl.rename_milestones(subclustering_obj_1,new={'0':'Cd163- Tmem+','2': "Cd163+ Tmem-"})
sc.pl.tsne(subclustering_obj_1,color="milestones")



In [ ]:
scf.pl.milestones(subclustering_obj_1,basis="tsne",annotate=True)


scf.tl.linearity_deviation(subclustering_obj_1,
                           start_milestone="Cd163+ Tmem-",
                           end_milestone="Cd163- Tmem+",
                           n_jobs=20,plot=True,basis="tsne")

In [ ]:
scf.pl.linearity_deviation(subclustering_obj_1,
                           start_milestone="Cd163+ Tmem-",
                           end_milestone="Cd163- Tmem+")

In [ ]:
# import rpy3
# scf.tl.test_association(subclustering_obj_1,n_jobs=20)
# scf.tl.test_association(subclustering_obj_1,reapply_filters=True,A_cut=.5)
# scf.pl.test_association(subclustering_obj_1)



### pseudotime by neighborhood plot

In [ ]:

merged_adata_GM_sub = merged_adata_GM[merged_adata_GM.obs['id'].isin(subclustering_obj_1.obs['ImageID_CellID'])]
merged_adata_GM_sub = merged_adata_GM_sub[merged_adata_GM_sub.obs.leiden != 'Monocytes']
subclustering_obj_1.obs.index = subclustering_obj_1.obs['ImageID_CellID']
merged_adata_GM_sub.obs['pseudotime'] = subclustering_obj_1.obs.loc[merged_adata_GM_sub.obs['id']]['t'].tolist()


subclustering_obj_1 = subclustering_obj_1[merged_adata_GM_sub.obs['id'], :]
pca_coordinates = subclustering_obj_1.obsm["X_tsne"]







In [ ]:
merged_adata_GM_sub.obsm['X_tsne'] = pca_coordinates
sc.pl.tsne(merged_adata_GM_sub,color="pseudotime", show=False)
sc.pl.tsne(merged_adata_GM_sub,color="SMA", show=False, layer='zscore')
sc.pl.tsne(merged_adata_GM_sub,color="A-Beta", show=False, layer='zscore')
sc.pl.tsne(merged_adata_GM_sub,color="NeuN", show=False, layer='zscore')
sc.pl.tsne(merged_adata_GM_sub,color="GFAP", show=False, layer='zscore')


In [ ]:
# split object by AD and CNT
merged_adata_GM_sub_AD = merged_adata_GM_sub[merged_adata_GM_sub.obs['AD_CNT'] == 'AD']
merged_adata_GM_sub_CNT = merged_adata_GM_sub[merged_adata_GM_sub.obs['AD_CNT'] == 'CNT']

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

# Load your AnnData object or create one
# For example, let's say you have an AnnData object with metadata and features
adata = merged_adata_GM_sub_AD

# Define the metadata column and features you want to plot
metadata_column = 'pseudotime'
feature_names = ['Claudin5', 'SMA', 'CollagenIV', 'A-Beta', 'NeuN']  # Replace with your feature names

# Extract the feature data from .X and metadata from .obs
feature_data = adata[:, feature_names].X
metadata = adata.obs[metadata_column]

# Create a DataFrame for the stacked bar plot
import pandas as pd
df = pd.DataFrame(data=feature_data, columns=feature_names)
df[metadata_column] = metadata.tolist()
df = df.sort_values(by='pseudotime', ascending=True)
# Melt the DataFrame to create a long-format DataFrame for Seaborn
df_long = pd.melt(df, id_vars=metadata_column, value_vars=feature_names, var_name='Features')





In [ ]:


import pandas as pd
import matplotlib.pyplot as plt

# Load your data into a DataFrame
# Replace 'your_data.csv' with the path to your data file

# Define the number of time bins
num_bins = 10

# Calculate the bin edges and bin labels
bin_edges = pd.cut(df['pseudotime'], bins=num_bins, labels=False)
df['bin'] = bin_edges

# Group by the 'bin' column and calculate the mean of feature columns
binned_data = df.groupby('bin').mean()

# Plot each feature with time on the x-axis and feature value on the y-axis
for feature in df.columns:
    if feature not in ['pseudotime', 'bin']:
        plt.plot(binned_data['pseudotime'], binned_data[feature], label=feature)

# Customize the plot
plt.title('Cd163+ GM AD Neighborhood vs Pseudotime')
plt.xlabel('pseudotime')
plt.ylabel('Feature Value')

plt.legend()
plt.tight_layout()

plt.savefig('../figures/protein/Pseudotime_lineplot_AD.tiff')
plt.savefig('../figures/protein/Pseudotime_lineplot_AD.svg')

# Show the plot
plt.show()







In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

# Load your AnnData object or create one
# For example, let's say you have an AnnData object with metadata and features
adata = merged_adata_GM_sub_CNT

# Define the metadata column and features you want to plot
metadata_column = 'pseudotime'
feature_names = ['Claudin5', 'SMA', 'CollagenIV', 'A-Beta', 'NeuN']  # Replace with your feature names

# Extract the feature data from .X and metadata from .obs
feature_data = adata[:, feature_names].X
metadata = adata.obs[metadata_column]

# Create a DataFrame for the stacked bar plot
import pandas as pd
df = pd.DataFrame(data=feature_data, columns=feature_names)
df[metadata_column] = metadata.tolist()
df = df.sort_values(by='pseudotime', ascending=True)
# Melt the DataFrame to create a long-format DataFrame for Seaborn
df_long = pd.melt(df, id_vars=metadata_column, value_vars=feature_names, var_name='Features')





In [ ]:


import pandas as pd
import matplotlib.pyplot as plt

# Load your data into a DataFrame
# Replace 'your_data.csv' with the path to your data file

# Define the number of time bins
num_bins = 10

# Calculate the bin edges and bin labels
bin_edges = pd.cut(df['pseudotime'], bins=num_bins, labels=False)
df['bin'] = bin_edges

# Group by the 'bin' column and calculate the mean of feature columns
binned_data = df.groupby('bin').mean()

# Plot each feature with time on the x-axis and feature value on the y-axis
for feature in df.columns:
    if feature not in ['pseudotime', 'bin']:
        plt.plot(binned_data['pseudotime'], binned_data[feature], label=feature)

# Customize the plot
plt.title('Cd163+ GM Ctl Neighborhood vs Pseudotime')

plt.xlabel('Time')
plt.ylabel('Feature Value')
plt.legend()

# Show the plot
plt.tight_layout()

plt.savefig('../figures/protein/Pseudotime_lineplot_control.tiff')
plt.savefig('../figures/protein/Pseudotime_lineplot_control.svg')


plt.show()







# DIRECT overlap

## load adatas direct overlap

In [ ]:
data_directory = "sample_adatas/"
adata_list = {}


for filename in os.listdir(data_directory):
    if filename.endswith(".h5ad"):
        file_path = os.path.join(data_directory, filename)
        adata = anndata.read_h5ad(file_path)

        file_name = re.sub('.h5ad', '', filename)
        adata.obs['origfile'] = file_name
        #adata.obs['file_leiden'] = [i + j for i, j in zip(adata.obs['origfile'], adata.obs['leiden'])]
        adata_list[file_name] = adata




### fix mistaken names

In [ ]:

    
feature_dict = {
    'Claudin-5':'Claudin5',
    'a-Beta':'A-Beta',
    'DAPI_1':'DAPI'
}
features_to_remove = ['H2A.x', 'SMA - Claudin5 +', 'GFAP + Vimentin +']


# Define the custom order for feature names
custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV','PCNA', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'GFAP', 'DAPI']


    
for adata_name in adata_list.keys():
    adata = adata_list[adata_name]
    
    adata_vars = adata.var_names
    
    for var in adata_vars:
        if var in feature_dict.keys():
            adata.var_names = [feature_dict[name] if name in feature_dict.keys() else name for name in adata.var_names] 



    
    adata.var.drop(features_to_remove, inplace=False)







In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


merged_adata = anndata.concat(adata_list, axis=0, index_unique="-")

#custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'PCNA', 'GFAP', 'DAPI_1']
# Use NumPy's argsort to get the indices that would reorder the features
order_indices = [merged_adata.var_names.get_loc(feature) for feature in custom_feature_order]

# Reorder the features based on the custom order
merged_adata = merged_adata[:, order_indices]

merged_adata.obs['origfilenum'] = merged_adata.obs['origfile'].str.split('_', expand = True)[0]
#merged_adata.obs['AD_CNT'] = merged_adata.obs['origfile'].str.split('_', expand = True)[1]
merged_adata.obs['AD_CNT'] = [ad_cnt_dict[i] for i in merged_adata.obs['origfilenum']]



#sc.pl.violin(merged_adata_GM, 'CollagenIV', groupby='leiden', show=True)


## attach clustering metadata

In [ ]:

# alter the following below to adapt to new methods of clustering
clustering_metadata = anndata.read_h5ad('../final_h5ads/fabian_metadata_plus_clustering.h5ad')




cluster_column = 'leiden'
imageid_cellid_col = 'ImageID_CellID'


change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}




cluster_label_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2', 'default']
cluster_label_ticklabels = cluster_label_order

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs['leiden']]

clustering_metadata.obs['leiden'] = pd.Categorical(clustering_metadata.obs['leiden'], categories=cluster_label_order, ordered=False)

cluster_label_ticklabels.remove('default')







#--------------------------------------------------------------------------------------




mapping_dict = dict(zip(clustering_metadata.obs[imageid_cellid_col], clustering_metadata.obs[cluster_column]))
merged_adata.obs['leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['leiden']


# remove any meningeal macrophages

merged_adata = merged_adata[merged_adata.obs['parent'].isin(['Grey Matter', 'White Matter'])]

# remove any default clusters

merged_adata = merged_adata[merged_adata.obs['leiden'] != 'default']

merged_adata_WM = merged_adata[merged_adata.obs['parent'] == 'White Matter']
merged_adata_GM = merged_adata[merged_adata.obs['parent'] == 'Grey Matter']

merged_adata_WM = merged_adata_WM[:, merged_adata_WM.var_names.isin(['MAP2', 'NeuN']) == False]




merged_adata_sub = merged_adata[merged_adata.obs['id'].isin(clustering_metadata.obs['ImageID_CellID'])]
clustering_metadata.obs.index = clustering_metadata.obs['ImageID_CellID']

clustering_metadata = clustering_metadata[merged_adata_sub.obs['id'], :]

tsne_coordinates = clustering_metadata.obsm["X_tsne"]
merged_adata_sub.obsm['X_tsne'] = tsne_coordinates

#merged_adata.obsm.X_tsne = 

for feat in merged_adata_sub.var_names:
    df = merged_adata_sub.to_df()
    feat_upperbound = np.percentile(a=df[feat], q=[95])

    sc.pl.tsne(merged_adata_sub, color=feat, vmax=feat_upperbound)


## bulk data

In [ ]:
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_direct_overlap')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_all')

In [ ]:
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90)
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='zscore_direct_overlap')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_all')


sc.pl.violin(merged_adata_GM, keys='NeuN',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='A-Beta',groupby='leiden', rotation=90)

sc.pl.matrixplot(merged_adata_GM, merged_adata_GM.var_names, groupby='leiden', dendrogram=False, standard_scale='var')


In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}



minmax_df_GM = merged_adata_GM.to_df(layer='zscore_direct_overlap')
minmax_df_GM['sample_leiden'] = merged_adata_GM.obs['file_leiden']
minmax_df_GM_mean = minmax_df_GM.groupby(by=['sample_leiden']).mean()
minmax_df_GM_mean['sample_leiden'] = minmax_df_GM_mean.index
minmax_df_GM_mean['leiden'] = minmax_df_GM_mean['sample_leiden'].str.split('.', expand = True)[1]
minmax_df_GM_mean['sample'] = minmax_df_GM_mean['sample_leiden'].str.split('.', expand = True)[0]
minmax_df_GM_mean['sample_num'] = minmax_df_GM_mean['sample'].str.split('_', expand = True)[0]
minmax_df_GM_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_GM_mean['sample_num']]


minmax_df_WM = merged_adata_WM.to_df(layer='zscore_direct_overlap')
minmax_df_WM['sample_leiden'] = merged_adata_WM.obs['file_leiden']
minmax_df_WM_mean = minmax_df_WM.groupby(by=['sample_leiden']).mean()
minmax_df_WM_mean['sample_leiden'] = minmax_df_WM_mean.index
minmax_df_WM_mean['leiden'] = minmax_df_WM_mean['sample_leiden'].str.split('.', expand = True)[1]
minmax_df_WM_mean['sample'] = minmax_df_WM_mean['sample_leiden'].str.split('.', expand = True)[0]
minmax_df_WM_mean['sample_num'] = minmax_df_WM_mean['sample'].str.split('_', expand = True)[0]
minmax_df_WM_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_WM_mean['sample_num']]






In [ ]:

import pandas as pd
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Step 3: Prepare Data
data = {
    'group1': [10, 12, 14, 16, 18],
    'group2': [20, 22, 24, 26, 28],
    'group3': [30, 32, 34, 36, 38]
}

df = pd.DataFrame(data)

# Step 4: Perform ANOVA
f_statistic, p_value = f_oneway(df['group1'], df['group2'], df['group3'])

# Output ANOVA results
print(f'ANOVA - F-statistic: {f_statistic}, P-value: {p_value}')

# Step 5: Perform Tukey Test
data_melted = pd.melt(df.reset_index(), id_vars=['index'], value_vars=['group1', 'group2', 'group3'])
tukey_results = pairwise_tukeyhsd(data_melted['value'], data_melted['variable'])

# Output Tukey results
print('\nTukey Test Results:')
print(tukey_results)









## box and whisker ANOVA

In [ ]:
from scipy.stats import f_oneway
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd

import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd



def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    
    fig, ax = plt.subplots()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True, whis=np.inf)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova
    
    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    
    print(anova_p_value)

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]
    print(combinations)
    

    for c in combinations:
        
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])

    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
            
        sig_symbol = sig_symbol + ' ' + pval_print(p)
        
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    else:
        return(fig, anova_f_statistic, anova_p_value, res)
    
    plt.show()
    
    
    
    
    



minmax_df_mean = minmax_df_GM_mean

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter direct overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/direct_overlap/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
    
    

    

minmax_df_mean = minmax_df_WM_mean

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter direct overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/direct_overlap/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    


## box and whisker anova split by ad and cnt

### box and whisker anova ad

In [ ]:
minmax_df_mean = minmax_df_GM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter AD direct overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/direct_overlap/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
minmax_df_mean = minmax_df_WM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter AD direct overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/direct_overlap/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')


### box and whisker anova cnt

In [ ]:
minmax_df_mean = minmax_df_GM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter Control direct overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/direct_overlap/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    

minmax_df_mean = minmax_df_WM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter Control direct overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/direct_overlap/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/direct_overlap/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')

## dotplot GM

In [ ]:
minmax_df_mean = minmax_df_GM_mean

frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean[minmax_df_mean['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/direct_overlap/GM_all_leiden_heatmap.csv')


frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = frame_z

# Sample additional values for bubble sizes
bubble_sizes = pframe_log.to_numpy() * 100  # Adjust the multiplier for bubble size

# Create a figure and axis
fig, ax = plt.subplots()

# Loop through the DataFrame and create bubbles
for i, row in enumerate(data.index):
    for j, col in enumerate(data.columns):
        value = data.at[row, col]  # Use DataFrame value for color
        size = bubble_sizes[i, j]  # Use additional value for bubble size
        color = plt.cm.viridis(value)  # Use DataFrame value for color
        ax.scatter(j, i, c=[color], s=size, alpha=0.7)

# Customize the plot
ax.set_xticks(np.arange(len(data.columns)))
ax.set_yticks(np.arange(len(data.index)))
ax.set_xticklabels(data.columns, rotation=90)
ax.set_yticklabels(data.index)
#plt.colorbar(label='Color Scale')
plt.title('Sig + Association bubblemap GM')

# Add grid lines
ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)

plt.yticks(plt.yticks()[0], [str.replace(i, 'GM', '') for i in data.index.tolist()])


ax.grid(which="minor", color="black", linestyle='-', linewidth=2)

# Show the plot
plt.show()

## heatmap GM

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])

# Customize the plot
plt.title('Sig + Relative association GM')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')


# Show the plot
plt.savefig('../figures/protein/direct_overlap/GM_all_leiden_heatmap_direct_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/direct_overlap/GM_all_leiden_heatmap_direct_overlap.svg', format='svg')

plt.show()

## heatmap ad and cnt seperate GM

In [ ]:





minmax_df_mean = minmax_df_GM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/direct_overlap/GM_AD_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association GM AD')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/direct_overlap/GM_AD_leiden_heatmap_direct_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/direct_overlap/GM_AD_leiden_heatmap_direct_overlap.svg', format='svg')

# Show the plot
plt.show()

#------------------------------------------------------------------------


minmax_df_mean = minmax_df_GM_mean

minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/direct_overlap/GM_Control_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)



plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])



# Customize the plot
plt.title('Sig + Relative association GM CNT')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/direct_overlap/GM_CONTROL_leiden_heatmap_direct_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/direct_overlap/GM_CONTROL_leiden_heatmap_direct_overlap.svg', format='svg')

# Show the plot
plt.show()






## dotplot WM

In [ ]:
minmax_df_mean = minmax_df_WM_mean

frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean[minmax_df_mean['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/direct_overlap/WM_all_leiden_heatmap.csv')

frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = frame_z

# Sample additional values for bubble sizes
bubble_sizes = pframe_log.to_numpy() * 100  # Adjust the multiplier for bubble size

# Create a figure and axis
fig, ax = plt.subplots()

# Loop through the DataFrame and create bubbles
for i, row in enumerate(data.index):
    for j, col in enumerate(data.columns):
        value = data.at[row, col]  # Use DataFrame value for color
        size = bubble_sizes[i, j]  # Use additional value for bubble size
        color = plt.cm.viridis(value)  # Use DataFrame value for color
        ax.scatter(j, i, c=[color], s=size, alpha=0.7)

# Customize the plot
ax.set_xticks(np.arange(len(data.columns)))
ax.set_yticks(np.arange(len(data.index)))
ax.set_xticklabels(data.columns, rotation=90)
ax.set_yticklabels(data.index)
#plt.colorbar(label='Color Scale')
plt.title('Sig + Association bubblemap WM')



# Add grid lines
ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)
ax.grid(which="minor", color="black", linestyle='-', linewidth=2)


plt.yticks(plt.yticks()[0], [str.replace(i, 'WM', '') for i in data.index.tolist()])


# Show the plot
plt.show()

## heatmap WM

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/direct_overlap/WM_all_leiden_heatmap_direct_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/direct_overlap/WM_all_leiden_heatmap_direct_overlap.svg', format='svg')

# Show the plot
plt.show()


## heatmap ad and cnt seperate WM

In [ ]:





minmax_df_mean = minmax_df_WM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/direct_overlap/WM_AD_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM AD')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/direct_overlap/WM_AD_leiden_heatmap_direct_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/direct_overlap/WM_AD_leiden_heatmap_direct_overlap.svg', format='svg')

# Show the plot
plt.show()

#------------------------------------------------------------------------


minmax_df_mean = minmax_df_WM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/direct_overlap/WM_Control_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)

plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM CNT')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/direct_overlap/WM_CONTROL_leiden_heatmap_direct_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/direct_overlap/WM_CONTROL_leiden_heatmap_direct_overlap.svg', format='svg')

# Show the plot
plt.show()






# WHOLE RADIUS overlap

## load adatas direct overlap

In [ ]:
data_directory = "sample_adatas/"
adata_list = {}


for filename in os.listdir(data_directory):
    if filename.endswith(".h5ad"):
        file_path = os.path.join(data_directory, filename)
        adata = anndata.read_h5ad(file_path)

        file_name = re.sub('.h5ad', '', filename)
        adata.obs['origfile'] = file_name
        #adata.obs['file_leiden'] = [i + j for i, j in zip(adata.obs['origfile'], adata.obs['leiden'])]
        adata_list[file_name] = adata




### fix mistaken names

In [ ]:

    
feature_dict = {
    'Claudin-5':'Claudin5',
    'a-Beta':'A-Beta',
    'DAPI_1':'DAPI'
}
features_to_remove = ['H2A.x', 'SMA - Claudin5 +', 'GFAP + Vimentin +']


# Define the custom order for feature names
custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV','PCNA', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'GFAP', 'DAPI']


    
for adata_name in adata_list.keys():
    adata = adata_list[adata_name]
    
    adata_vars = adata.var_names
    
    for var in adata_vars:
        if var in feature_dict.keys():
            adata.var_names = [feature_dict[name] if name in feature_dict.keys() else name for name in adata.var_names] 



    
    adata.var.drop(features_to_remove, inplace=False)







In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


merged_adata = anndata.concat(adata_list, axis=0, index_unique="-")

#custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'PCNA', 'GFAP', 'DAPI_1']
# Use NumPy's argsort to get the indices that would reorder the features
order_indices = [merged_adata.var_names.get_loc(feature) for feature in custom_feature_order]

# Reorder the features based on the custom order
merged_adata = merged_adata[:, order_indices]

merged_adata.obs['origfilenum'] = merged_adata.obs['origfile'].str.split('_', expand = True)[0]
#merged_adata.obs['AD_CNT'] = merged_adata.obs['origfile'].str.split('_', expand = True)[1]
merged_adata.obs['AD_CNT'] = [ad_cnt_dict[i] for i in merged_adata.obs['origfilenum']]



#sc.pl.violin(merged_adata_GM, 'CollagenIV', groupby='leiden', show=True)


## attach clustering metadata

In [ ]:

# alter the following below to adapt to new methods of clustering
clustering_metadata = anndata.read_h5ad('../final_h5ads/fabian_metadata_plus_clustering.h5ad')




cluster_column = 'leiden'
imageid_cellid_col = 'ImageID_CellID'


change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}




cluster_label_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2', 'default']
cluster_label_ticklabels = cluster_label_order

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs['leiden']]

clustering_metadata.obs['leiden'] = pd.Categorical(clustering_metadata.obs['leiden'], categories=cluster_label_order, ordered=False)

cluster_label_ticklabels.remove('default')







#--------------------------------------------------------------------------------------




mapping_dict = dict(zip(clustering_metadata.obs[imageid_cellid_col], clustering_metadata.obs[cluster_column]))
merged_adata.obs['leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['leiden']


# remove any meningeal macrophages

merged_adata = merged_adata[merged_adata.obs['parent'].isin(['Grey Matter', 'White Matter'])]

# remove any default clusters

merged_adata = merged_adata[merged_adata.obs['leiden'] != 'default']

merged_adata_WM = merged_adata[merged_adata.obs['parent'] == 'White Matter']
merged_adata_GM = merged_adata[merged_adata.obs['parent'] == 'Grey Matter']

merged_adata_WM = merged_adata_WM[:, merged_adata_WM.var_names.isin(['MAP2', 'NeuN']) == False]




merged_adata_sub = merged_adata[merged_adata.obs['id'].isin(clustering_metadata.obs['ImageID_CellID'])]
clustering_metadata.obs.index = clustering_metadata.obs['ImageID_CellID']

clustering_metadata = clustering_metadata[merged_adata_sub.obs['id'], :]

tsne_coordinates = clustering_metadata.obsm["X_tsne"]
merged_adata_sub.obsm['X_tsne'] = tsne_coordinates

#merged_adata.obsm.X_tsne = 

for feat in merged_adata_sub.var_names:
    df = merged_adata_sub.to_df()
    feat_upperbound = np.percentile(a=df[feat], q=[95])

    sc.pl.tsne(merged_adata_sub, color=feat, vmax=feat_upperbound)



## bulk data

In [ ]:
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90)
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='zscore_direct_overlap')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_all')


sc.pl.violin(merged_adata_GM, keys='NeuN',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='A-Beta',groupby='leiden', rotation=90)

sc.pl.matrixplot(merged_adata_GM, merged_adata_GM.var_names, groupby='leiden', dendrogram=False, standard_scale='var')


In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}



minmax_df_GM = merged_adata_GM.to_df(layer='zscore_all')
minmax_df_GM['sample_leiden'] = merged_adata_GM.obs['file_leiden']
minmax_df_GM_mean = minmax_df_GM.groupby(by=['sample_leiden']).mean()
minmax_df_GM_mean['sample_leiden'] = minmax_df_GM_mean.index
minmax_df_GM_mean['leiden'] = minmax_df_GM_mean['sample_leiden'].str.split('.', expand = True)[1]
minmax_df_GM_mean['sample'] = minmax_df_GM_mean['sample_leiden'].str.split('.', expand = True)[0]
minmax_df_GM_mean['sample_num'] = minmax_df_GM_mean['sample'].str.split('_', expand = True)[0]
minmax_df_GM_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_GM_mean['sample_num']]


minmax_df_WM = merged_adata_WM.to_df(layer='zscore_all')
minmax_df_WM['sample_leiden'] = merged_adata_WM.obs['file_leiden']
minmax_df_WM_mean = minmax_df_WM.groupby(by=['sample_leiden']).mean()
minmax_df_WM_mean['sample_leiden'] = minmax_df_WM_mean.index
minmax_df_WM_mean['leiden'] = minmax_df_WM_mean['sample_leiden'].str.split('.', expand = True)[1]
minmax_df_WM_mean['sample'] = minmax_df_WM_mean['sample_leiden'].str.split('.', expand = True)[0]
minmax_df_WM_mean['sample_num'] = minmax_df_WM_mean['sample'].str.split('_', expand = True)[0]
minmax_df_WM_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_WM_mean['sample_num']]






In [ ]:

import pandas as pd
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Step 3: Prepare Data
data = {
    'group1': [10, 12, 14, 16, 18],
    'group2': [20, 22, 24, 26, 28],
    'group3': [30, 32, 34, 36, 38]
}

df = pd.DataFrame(data)

# Step 4: Perform ANOVA
f_statistic, p_value = f_oneway(df['group1'], df['group2'], df['group3'])

# Output ANOVA results
print(f'ANOVA - F-statistic: {f_statistic}, P-value: {p_value}')

# Step 5: Perform Tukey Test
data_melted = pd.melt(df.reset_index(), id_vars=['index'], value_vars=['group1', 'group2', 'group3'])
tukey_results = pairwise_tukeyhsd(data_melted['value'], data_melted['variable'])

# Output Tukey results
print('\nTukey Test Results:')
print(tukey_results)









## box and whisker ANOVA

In [ ]:
from scipy.stats import f_oneway
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd

import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd



def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    
    fig, ax = plt.subplots()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True, whis=np.inf)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova
    
    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    
    print(anova_p_value)

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]
    print(combinations)
    

    for c in combinations:
        
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])

    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
            
        sig_symbol = sig_symbol + ' ' + pval_print(p)
        
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    else:
        return(fig, anova_f_statistic, anova_p_value, res)
    
    plt.show()
    
    
    
    
    



minmax_df_mean = minmax_df_GM_mean

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter whole overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/whole_radius/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
    
    

    

minmax_df_mean = minmax_df_WM_mean

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter whole overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/whole_radius/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    

## box and whisker anova split by ad and cnt

### box and whisker anova ad

In [ ]:
minmax_df_mean = minmax_df_GM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter AD whole overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/whole_radius/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
minmax_df_mean = minmax_df_WM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter AD whole overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/whole_radius/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')



### box and whisker anova cnt

In [ ]:
minmax_df_mean = minmax_df_GM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter Control whole overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/whole_radius/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')

minmax_df_mean = minmax_df_WM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter Control whole overlap'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/protein/whole_radius/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/protein/whole_radius/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')


## dotplot GM

In [ ]:
minmax_df_mean = minmax_df_GM_mean

frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean[minmax_df_mean['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/whole_radius/GM_all_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = frame_z

# Sample additional values for bubble sizes
bubble_sizes = pframe_log.to_numpy() * 100  # Adjust the multiplier for bubble size

# Create a figure and axis
fig, ax = plt.subplots()

# Loop through the DataFrame and create bubbles
for i, row in enumerate(data.index):
    for j, col in enumerate(data.columns):
        value = data.at[row, col]  # Use DataFrame value for color
        size = bubble_sizes[i, j]  # Use additional value for bubble size
        color = plt.cm.viridis(value)  # Use DataFrame value for color
        ax.scatter(j, i, c=[color], s=size, alpha=0.7)

# Customize the plot
ax.set_xticks(np.arange(len(data.columns)))
ax.set_yticks(np.arange(len(data.index)))
ax.set_xticklabels(data.columns, rotation=90)
ax.set_yticklabels(data.index)
#plt.colorbar(label='Color Scale')
plt.title('Sig + Association bubblemap GM')

# Add grid lines
ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)

plt.yticks(plt.yticks()[0], [str.replace(i, 'GM', '') for i in data.index.tolist()])


ax.grid(which="minor", color="black", linestyle='-', linewidth=2)

# Show the plot
plt.show()

## heatmap GM

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])

# Customize the plot
plt.title('Sig + Relative association GM')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')


# Show the plot
plt.savefig('../figures/protein/whole_radius/GM_all_leiden_heatmap_whole_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/whole_radius/GM_all_leiden_heatmap_whole_overlap.svg', format='svg')

plt.show()

## heatmap ad and cnt seperate GM

In [ ]:





minmax_df_mean = minmax_df_GM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/whole_radius/GM_AD_leiden_heatmap.csv')


frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association GM AD')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/whole_radius/GM_AD_leiden_heatmap_whole_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/whole_radius/GM_AD_leiden_heatmap_whole_overlap.svg', format='svg')

# Show the plot
plt.show()

#------------------------------------------------------------------------


minmax_df_mean = minmax_df_GM_mean

minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/whole_radius/GM_Control_leiden_heatmap.csv')


frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)



plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])



# Customize the plot
plt.title('Sig + Relative association GM CNT')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/whole_radius/GM_CONTROL_leiden_heatmap_whole_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/whole_radius/GM_CONTROL_leiden_heatmap_whole_overlap.svg', format='svg')

# Show the plot
plt.show()






## dotplot WM

In [ ]:
minmax_df_mean = minmax_df_WM_mean

frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean[minmax_df_mean['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/whole_radius/WM_all_leiden_heatmap.csv')


frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = frame_z

# Sample additional values for bubble sizes
bubble_sizes = pframe_log.to_numpy() * 100  # Adjust the multiplier for bubble size

# Create a figure and axis
fig, ax = plt.subplots()

# Loop through the DataFrame and create bubbles
for i, row in enumerate(data.index):
    for j, col in enumerate(data.columns):
        value = data.at[row, col]  # Use DataFrame value for color
        size = bubble_sizes[i, j]  # Use additional value for bubble size
        color = plt.cm.viridis(value)  # Use DataFrame value for color
        ax.scatter(j, i, c=[color], s=size, alpha=0.7)

# Customize the plot
ax.set_xticks(np.arange(len(data.columns)))
ax.set_yticks(np.arange(len(data.index)))
ax.set_xticklabels(data.columns, rotation=90)
ax.set_yticklabels(data.index)
#plt.colorbar(label='Color Scale')
plt.title('Sig + Association bubblemap WM')



# Add grid lines
ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)
ax.grid(which="minor", color="black", linestyle='-', linewidth=2)


plt.yticks(plt.yticks()[0], [str.replace(i, 'WM', '') for i in data.index.tolist()])


# Show the plot
plt.show()

## heatmap WM

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/whole_radius/WM_all_leiden_heatmap_whole_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/whole_radius/WM_all_leiden_heatmap_whole_overlap.svg', format='svg')

# Show the plot
plt.show()


## heatmap ad and cnt seperate WM

In [ ]:





minmax_df_mean = minmax_df_WM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/whole_radius/WM_AD_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM AD')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/whole_radius/WM_AD_leiden_heatmap_whole_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/whole_radius/WM_AD_leiden_heatmap_whole_overlap.svg', format='svg')

# Show the plot
plt.show()

#------------------------------------------------------------------------


minmax_df_mean = minmax_df_WM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/protein/whole_radius/WM_Control_leiden_heatmap.csv')



frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)

plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM CNT')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/protein/whole_radius/WM_CONTROL_leiden_heatmap_whole_overlap.tiff', format='tiff')
plt.savefig('../figures/protein/whole_radius/WM_CONTROL_leiden_heatmap_whole_overlap.svg', format='svg')

# Show the plot
plt.show()






# MORPH

## load adatas

In [ ]:
data_directory = "sample_adatas/"
adata_list = {}


for filename in os.listdir(data_directory):
    if filename.endswith(".h5ad"):
        file_path = os.path.join(data_directory, filename)
        adata = anndata.read_h5ad(file_path)

        file_name = re.sub('.h5ad', '', filename)
        adata.obs['origfile'] = file_name
        #adata.obs['file_leiden'] = [i + j for i, j in zip(adata.obs['origfile'], adata.obs['leiden'])]
        adata_list[file_name] = adata




### fix mistaken names

In [ ]:

    
feature_dict = {
    'Claudin-5':'Claudin5',
    'a-Beta':'A-Beta',
    'DAPI_1':'DAPI'
}
features_to_remove = ['H2A.x', 'SMA - Claudin5 +', 'GFAP + Vimentin +']


# Define the custom order for feature names
custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV','PCNA', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'GFAP', 'DAPI']


    
for adata_name in adata_list.keys():
    adata = adata_list[adata_name]
    
    adata_vars = adata.var_names
    
    for var in adata_vars:
        if var in feature_dict.keys():
            adata.var_names = [feature_dict[name] if name in feature_dict.keys() else name for name in adata.var_names] 



    
    adata.var.drop(features_to_remove, inplace=False)







In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


merged_adata = anndata.concat(adata_list, axis=0, index_unique="-")

#custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'PCNA', 'GFAP', 'DAPI_1']
# Use NumPy's argsort to get the indices that would reorder the features
order_indices = [merged_adata.var_names.get_loc(feature) for feature in custom_feature_order]

# Reorder the features based on the custom order
merged_adata = merged_adata[:, order_indices]

merged_adata.obs['origfilenum'] = merged_adata.obs['origfile'].str.split('_', expand = True)[0]
#merged_adata.obs['AD_CNT'] = merged_adata.obs['origfile'].str.split('_', expand = True)[1]
merged_adata.obs['AD_CNT'] = [ad_cnt_dict[i] for i in merged_adata.obs['origfilenum']]



#sc.pl.violin(merged_adata_GM, 'CollagenIV', groupby='leiden', show=True)


## attach clustering metadata

In [ ]:

# alter the following below to adapt to new methods of clustering
clustering_metadata = anndata.read_h5ad('../final_h5ads/clustering_and_morph_clustering.h5')
merged_adata = merged_adata[merged_adata.obs.id.isin(clustering_metadata.obs.ImageID_CellID)]


cluster_column = 'leiden'
imageid_cellid_col = 'ImageID_CellID'


cluster_label_order = ['Rounded', 'Intermediate', 'Ramified 1', 'Ramified 2', 'Ramified 3']
cluster_label_ticklabels = cluster_label_order

change_cluster_names_dict = {
    '1': 'Rounded',
    '2': 'Intermediate',
    '0_ram':'Ramified 2',
    '1_ram':'Ramified 3',
    '2_ram':'Ramified 1'
}

cMapDict = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs.final_leiden_M1]
clustering_metadata.obs['leiden'] = pd.Categorical(clustering_metadata.obs['leiden'], categories=cluster_label_order, ordered=False)







#--------------------------------------------------------------------------------------




mapping_dict = dict(zip(clustering_metadata.obs[imageid_cellid_col], clustering_metadata.obs[cluster_column]))
merged_adata.obs['leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['leiden']


# remove any meningeal macrophages

merged_adata = merged_adata[merged_adata.obs['parent'].isin(['Grey Matter', 'White Matter'])]

# remove any default clusters

merged_adata = merged_adata[merged_adata.obs['leiden'] != 'default']

merged_adata_WM = merged_adata[merged_adata.obs['parent'] == 'White Matter']
merged_adata_GM = merged_adata[merged_adata.obs['parent'] == 'Grey Matter']

merged_adata_WM = merged_adata_WM[:, merged_adata_WM.var_names.isin(['MAP2', 'NeuN']) == False]





merged_adata_sub = merged_adata[merged_adata.obs['id'].isin(clustering_metadata.obs['ImageID_CellID'])]
clustering_metadata.obs.index = clustering_metadata.obs['ImageID_CellID']

clustering_metadata = clustering_metadata[merged_adata_sub.obs['id'], :]

tsne_coordinates = clustering_metadata.obsm["X_tsne"]
merged_adata_sub.obsm['X_tsne'] = tsne_coordinates

#merged_adata.obsm.X_tsne = 

for feat in merged_adata_sub.var_names:
    df = merged_adata_sub.to_df()
    feat_upperbound = np.percentile(a=df[feat], q=[95])

    sc.pl.tsne(merged_adata_sub, color=feat, vmax=feat_upperbound)
    


# save clustering metadata for mert

clustering_metadata_mert = clustering_metadata.obs[['leiden', 'Parent', 'marker_intensity_csv_filename', 'cell', 'ImageType', 'ImageIDOLD', 'spatial_X', 'spatial_Y']]
clustering_metadata_mert.to_csv('mert_morpho_metadata.csv')

## bulk data

In [ ]:
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_direct_overlap')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_all')

In [ ]:
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90)
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_direct_overlap')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='Claudin5',groupby='leiden', rotation=90, layer='minmax_all')


sc.pl.violin(merged_adata_GM, keys='NeuN',groupby='leiden', rotation=90, layer='minmax')
sc.pl.violin(merged_adata_GM, keys='A-Beta',groupby='leiden', rotation=90)

sc.pl.matrixplot(merged_adata_GM, merged_adata_GM.var_names, groupby='leiden', dendrogram=False, standard_scale='var')


In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}



minmax_df_GM = merged_adata_GM.to_df(layer='zscore')
minmax_df_GM['sample_leiden'] = merged_adata_GM.obs['file_leiden']
minmax_df_GM_mean = minmax_df_GM.groupby(by=['sample_leiden']).mean()
minmax_df_GM_mean['sample_leiden'] = minmax_df_GM_mean.index
minmax_df_GM_mean['leiden'] = minmax_df_GM_mean['sample_leiden'].str.split('.', expand = True)[1]
minmax_df_GM_mean['sample'] = minmax_df_GM_mean['sample_leiden'].str.split('.', expand = True)[0]
minmax_df_GM_mean['sample_num'] = minmax_df_GM_mean['sample'].str.split('_', expand = True)[0]
minmax_df_GM_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_GM_mean['sample_num']]


minmax_df_WM = merged_adata_WM.to_df(layer='zscore')
minmax_df_WM['sample_leiden'] = merged_adata_WM.obs['file_leiden']
minmax_df_WM_mean = minmax_df_WM.groupby(by=['sample_leiden']).mean()
minmax_df_WM_mean['sample_leiden'] = minmax_df_WM_mean.index
minmax_df_WM_mean['leiden'] = minmax_df_WM_mean['sample_leiden'].str.split('.', expand = True)[1]
minmax_df_WM_mean['sample'] = minmax_df_WM_mean['sample_leiden'].str.split('.', expand = True)[0]
minmax_df_WM_mean['sample_num'] = minmax_df_WM_mean['sample'].str.split('_', expand = True)[0]
minmax_df_WM_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_WM_mean['sample_num']]






In [ ]:

import pandas as pd
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Step 3: Prepare Data
data = {
    'group1': [10, 12, 14, 16, 18],
    'group2': [20, 22, 24, 26, 28],
    'group3': [30, 32, 34, 36, 38]
}

df = pd.DataFrame(data)

# Step 4: Perform ANOVA
f_statistic, p_value = f_oneway(df['group1'], df['group2'], df['group3'])

# Output ANOVA results
print(f'ANOVA - F-statistic: {f_statistic}, P-value: {p_value}')

# Step 5: Perform Tukey Test
data_melted = pd.melt(df.reset_index(), id_vars=['index'], value_vars=['group1', 'group2', 'group3'])
tukey_results = pairwise_tukeyhsd(data_melted['value'], data_melted['variable'])

# Output Tukey results
print('\nTukey Test Results:')
print(tukey_results)









## box and whisker ANOVA

In [ ]:
from scipy.stats import f_oneway
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd

import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd



def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    
    fig, ax = plt.subplots()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True, whis=np.inf)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova
    
    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    
    print(anova_p_value)

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]
    print(combinations)
    

    for c in combinations:
        
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])

    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
            
        sig_symbol = sig_symbol + ' ' + pval_print(p)
        
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    else:
        return(fig, anova_f_statistic, anova_p_value, res)
    
    plt.show()
    
    
    
    
    



minmax_df_mean = minmax_df_GM_mean

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter morph'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]

for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/morph/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    
    
    

    

minmax_df_mean = minmax_df_WM_mean

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter morph'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/morph/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    



## box and whisker anova split by ad and cnt

### box and whisker anova ad

In [ ]:
minmax_df_mean = minmax_df_GM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' Gray Matter AD morph'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/morph/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    
    

minmax_df_mean = minmax_df_WM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter AD morph'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/morph/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')


### box and whisker anova cnt

In [ ]:
minmax_df_mean = minmax_df_GM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter Control morph'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/morph/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')
    
    

minmax_df_mean = minmax_df_WM_mean
minmax_df_mean = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']

anova_plot_dict = {}

for protein in minmax_df_mean.columns[:-5]:
    print(protein)
    
    data = []
    title = protein + ' White Matter Control morph'
    ylabel = 'average normalized neighbor % composition'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)

    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    
    plot, fstat, pval, tukeyres = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  
    
    anova_plot_dict[title] = [plot, fstat, pval, title, tukeyres]

    
pvals = [anova_plot_dict[i][2] for i in anova_plot_dict]
adjusted_pvals = multipletests(pvals, method='fdr_bh')[1]


for i, p_adjust, p in zip(anova_plot_dict, adjusted_pvals, pvals):
    plot, fstat, pval, title, tukeyres = anova_plot_dict[i]
    plot.axes[0].set_title(title + ' (ANOVA p.adj = ' + pval_print(p_adjust) + ')' )
        
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.svg', format='svg')
    plot.savefig('../figures/morph/' + title + '_leiden_box&whisker_ANOVA.tiff', format='tiff')
    tukey_df = tukey_to_df(tukeyres, labels = cluster_label_order)
    tukey_df['anova_p_adjust'] = p_adjust
    tukey_df['anova_p'] = p
    tukey_df['title'] = title
    tukey_df.to_csv('../stat_tables/morph/' + title + '_ledien_box&whisker_ANOVA_test_table.csv')

## dotplot GM

In [ ]:
minmax_df_mean = minmax_df_GM_mean

frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean[minmax_df_mean['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/morph/GM_morph_leiden_heatmap.csv')


def sillyscale(frame):
    m_frame = frame/np.mean(frame)
    m_frame = m_frame - np.min(m_frame)
    
    return(np.clip(m_frame, a_max = 1, a_min = 0))
    #return(np.log2(m_frame + 1))

frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)

#frame_z = sillyscale(frame)


pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = frame_z

# Sample additional values for bubble sizes
bubble_sizes = pframe_log.to_numpy() * 100  # Adjust the multiplier for bubble size

# Create a figure and axis
fig, ax = plt.subplots()

# Loop through the DataFrame and create bubbles
for i, row in enumerate(data.index):
    for j, col in enumerate(data.columns):
        value = data.at[row, col]  # Use DataFrame value for color
        size = bubble_sizes[i, j]  # Use additional value for bubble size
        color = plt.cm.viridis(value)  # Use DataFrame value for color
        ax.scatter(j, i, c=[color], s=size, alpha=0.7)

# Customize the plot
ax.set_xticks(np.arange(len(data.columns)))
ax.set_yticks(np.arange(len(data.index)))
ax.set_xticklabels(data.columns, rotation=90)
ax.set_yticklabels(data.index)
#plt.colorbar(label='Color Scale')
plt.title('Sig + Association bubblemap GM')

# Add grid lines
ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)

plt.yticks(plt.yticks()[0], [str.replace(i, 'GM', '') for i in data.index.tolist()])


ax.grid(which="minor", color="black", linestyle='-', linewidth=2)

# Show the plot
plt.show()

## heatmap GM

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])

# Customize the plot
plt.title('Sig + Relative association GM')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')


# Show the plot
plt.savefig('../figures/morph/GM_morph_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/morph/GM_morph_leiden_heatmap.svg', format='svg')

plt.show()

## heatmap ad and cnt seperate GM

In [ ]:





minmax_df_mean = minmax_df_GM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/morph/GM_AD_leiden_heatmap.csv')




frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
#frame_z = sillyscale(frame)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association GM AD')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/morph/GM_AD_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/morph/GM_AD_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()

#------------------------------------------------------------------------


minmax_df_mean = minmax_df_GM_mean

minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/morph/GM_Control_leiden_heatmap.csv')




frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
#frame_z = sillyscale(frame)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)



plt.xticks(plt.xticks()[0], [str.replace(i, 'GM', '') for i in data.columns.tolist()])



# Customize the plot
plt.title('Sig + Relative association GM CNT')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/morph/GM_CONTROL_leiden_heatmap_morph.tiff', format='tiff')
plt.savefig('../figures/morph/GM_CONTROL_leiden_heatmap_morph.svg', format='svg')

# Show the plot
plt.show()






## dotplot WM

In [ ]:
minmax_df_mean = minmax_df_WM_mean

frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean[minmax_df_mean['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean[minmax_df_mean['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/morph/WM_morph_leiden_heatmap.csv')


frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = frame_z

# Sample additional values for bubble sizes
bubble_sizes = pframe_log.to_numpy() * 100  # Adjust the multiplier for bubble size

# Create a figure and axis
fig, ax = plt.subplots()

# Loop through the DataFrame and create bubbles
for i, row in enumerate(data.index):
    for j, col in enumerate(data.columns):
        value = data.at[row, col]  # Use DataFrame value for color
        size = bubble_sizes[i, j]  # Use additional value for bubble size
        color = plt.cm.viridis(value)  # Use DataFrame value for color
        ax.scatter(j, i, c=[color], s=size, alpha=0.7)

# Customize the plot
ax.set_xticks(np.arange(len(data.columns)))
ax.set_yticks(np.arange(len(data.index)))
ax.set_xticklabels(data.columns, rotation=90)
ax.set_yticklabels(data.index)
#plt.colorbar(label='Color Scale')
plt.title('Sig + Association bubblemap WM')



# Add grid lines
ax.set_xticks(np.arange(len(data.columns) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(data.index) + 1) - 0.5, minor=True)
ax.grid(which="minor", color="black", linestyle='-', linewidth=2)


plt.yticks(plt.yticks()[0], [str.replace(i, 'WM', '') for i in data.index.tolist()])


# Show the plot
plt.show()

## heatmap WM

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/morph/WM_morph_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/morph/WM_morph_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()


## heatmap ad and cnt seperate WM

In [ ]:





minmax_df_mean = minmax_df_WM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'AD']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/morph/WM_AD_leiden_heatmap.csv')


frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)


plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM AD')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/morph/WM_AD_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/morph/WM_AD_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()

#------------------------------------------------------------------------


minmax_df_mean = minmax_df_WM_mean

# minmax_df_mean['sample'] = minmax_df_mean['sample_leiden'].str.split('_', expand=True)[0]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]







minmax_df_mean_AD = minmax_df_mean[minmax_df_mean['AD_CNT'] == 'CNT']


frame = pd.DataFrame()
pframe = pd.DataFrame()
statframe = pd.DataFrame()

for protein in minmax_df_mean_AD.columns[:-5]:
    
    data = []
    
    
    
    means = []
    pvals = []
    test_stats = []
    
    
    for cluster in cluster_label_order:
    
        sub_df = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] == cluster][protein]
        
        sub_df_other = minmax_df_mean_AD[minmax_df_mean_AD['leiden'] != cluster][protein]
        U, p = stats.ttest_ind(sub_df, sub_df_other, alternative='two-sided')
        
        
        mean_ = np.mean(sub_df)
        
        
        means.append(mean_)
        pvals.append(p)
        test_stats.append(U)
        
        
        
        #data.append(sub_df[protein])
    frame[protein] = means
    pframe[protein] = pvals
    statframe[protein] = test_stats
    
    
    #print(data)
    

    

frame.index = cluster_label_order
pframe.index = cluster_label_order
statframe.index = cluster_label_order


statistics_info = pd.concat([frame, pframe, statframe])
statistics_info['category'] = ['plotted_value']*len(frame.index) + ['p_value']*len(pframe.index) + ['T_statistic']*len(statframe.index)
statistics_info.to_csv('../stat_tables/morph/WM_Control_leiden_heatmap.csv')


frame_z = frame.apply(zscore).clip(lower=-2, upper = 2)
pframe_log = (-np.log10(pframe)).clip(lower = 0, upper = 2)



# Function to convert p-values to asterisks
def p_to_asterisks(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# Apply the function to each element in the DataFrame
pframe.loc[:,:] = multipletests(pframe.values.flatten(), method = 'fdr_bh')[1].reshape(pframe.shape)

pframe_asterisks = pframe.applymap(p_to_asterisks)

# Sample DataFrame with custom annotations
data = frame_z.transpose()

# Custom annotations for each cell
custom_annotations = pframe_asterisks.transpose().values.tolist()

# Create a figure and axis
plt.figure(figsize=(6, 10))  # Set the figure size
ax = sns.heatmap(data, cmap='viridis', annot=False, fmt="", linewidths=2, square=True, cbar=True, linecolor='white')

# Add custom annotations to each cell
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j + 0.5, i + 0.5, custom_annotations[i][j], ha='center', va='center', fontsize=12)

plt.xticks(plt.xticks()[0], [str.replace(i, 'WM', '') for i in data.columns.tolist()])


# Customize the plot
plt.title('Sig + Relative association WM CNT')
plt.xlabel('Microglia sub-type')
plt.ylabel('relative protein association')

plt.savefig('../figures/morph/WM_CONTROL_leiden_heatmap.tiff', format='tiff')
plt.savefig('../figures/morph/WM_CONTROL_leiden_heatmap.svg', format='svg')

# Show the plot
plt.show()






## ad vs control bar chart

In [ ]:
import matplotlib.patches as mpatches



minmax_df_mean = minmax_df_GM_mean
ad_samples = [3196, 3155, 2997, 3280]
cnt_samples = [1873, 3628, 3026, 1796]

# ad_cnt_dict = {
# '3196':'AD',
# '3155':'AD',
# '2997':'AD',
# '3280':'AD',
# '1873':'CNT',
# '3628':'CNT',
# '3026':'CNT',
# '1796':'CNT'
# }

# minmax_df_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in minmax_df_mean['sample']]


frame = pd.DataFrame()
pframe = pd.DataFrame()


for cluster in set(minmax_df_mean['leiden']):
    
    minmax_df_sub = minmax_df_mean[minmax_df_mean['leiden'] == cluster]
    minmax_df_sub_AD = minmax_df_sub[minmax_df_sub['AD_CNT'] == 'AD']
    minmax_df_sub_CNT = minmax_df_sub[minmax_df_sub['AD_CNT'] == 'CNT']
    
    proteinlogs = []
    pvals = []
    
    proteinvals = []
    conditionvals = []
    protein_realvals = []
    
    for protein in minmax_df_sub.columns[:-5]:
        proteinvals.extend([protein]*8)
        conditionvals.extend(['AD']*4)
        conditionvals.extend(['CONTROL']*4)
        
        
        protein_AD_vals = minmax_df_sub_AD[protein]
        protein_CNT_vals = minmax_df_sub_CNT[protein]
        
        protein_realvals.extend(protein_AD_vals)
        protein_realvals.extend(protein_CNT_vals)
        
        
        # print(protein_AD_vals)
        # print(protein_CNT_vals)
        print(protein)
        U, p = stats.ttest_ind(protein_AD_vals, protein_CNT_vals, alternative='two-sided')
        
        # fc = np.mean(protein_AD_vals)/np.mean(protein_CNT_vals)
        # logfc = np.log2(fc)
        
        logfc = np.mean(protein_AD_vals) - np.mean(protein_CNT_vals)
        
        proteinlogs.append(logfc)
        pvals.append(p)
        
    #HERE
    
    
    # Data for the bar chart
    categories = minmax_df_sub.columns[:-5].tolist()
    
    values = proteinlogs

    # Significance values for each bar (you can replace these with your actual significance values)
    significance_values = pvals

    # Create a figure and axis
    fig, ax = plt.subplots()

    # Create the bar chart
    bars = ax.bar(categories, values, color=['blue' if v >= 0 else 'orange' for v in values])

    # Add values at the end of each bar
    for i, bar in enumerate(bars):
        significance_value = significance_values[i]
        if values[i] >= 0:
            ax.text(bar.get_x() + bar.get_width()/2, values[i] + 0.05, f'{significance_value:.3f}', va = 'center', ha = 'center', size = 7)
        else:
            ax.text(bar.get_x() + bar.get_width()/2, values[i] - 0.05, f'{significance_value:.3f}',  va = 'center', ha = 'center', size = 7)


            
    # Set the y-axis range from -1 to 1
    ax.set_xticklabels(categories, rotation=90)

    # Add labels and title
    ax.set_xlabel('Categories')
    ax.set_ylabel('Values')
    ax.set_title('Gray Matter ' + cluster)
    
    legend_patches = [mpatches.Patch(color='blue', label='Greater in AD'), mpatches.Patch(color='orange', label='Greater in CONTROL')]
    plt.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1, 1))
    


    # Show the plot
    plt.tight_layout()
    plt.show()
    
    #HERE

    # Sample data with more points
    np.random.seed(0)
    data = {
        'Protein': proteinvals,
        'Condition': conditionvals,
        'Values': protein_realvals
    }

    # Create a DataFrame
    df = pd.DataFrame(data)

    # Set the figure size
    plt.figure(figsize=(10, 6))

    # Create a grouped dot plot
    #sns.stripplot(x='Protein', y='Values', hue='Condition', data=df, jitter=False, dodge=True, palette='Set3', size=10)
    #sns.stripplot(x='Protein', y='Values', hue='Condition', data=df, jitter=False, dodge=True, size=10)
    sns.boxplot(x = 'Protein', y = 'Values', hue = 'Condition', data = df, dodge = True)

    
    # Customize the plot
    plt.xlabel('Protein')
    plt.ylabel('Values')
    plt.title('Gray Matter ' + cluster)
    
    
    
    plt.xticks(rotation=90)

    # Show the plot
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.legend(title='Condition', loc='upper right')
    plt.show()

    
    
    



        
        
        
     














## pseudotime inference MATRIX

### all cells

In [ ]:
import networkx
#import elpigraph
import scFates as scf

path_to_subclustering = '../final_h5ads/clustering_and_morph_clustering.h5'
subclustering_obj_1 = anndata.read_h5ad(path_to_subclustering)


# subset and recalculate pca

print(sc.pl.tsne(subclustering_obj_1, color=['final_leiden_M1']))

print(np.unique(subclustering_obj_1.obs.ImageID))


print(np.unique(subclustering_obj_1.obs.ImageID))

print(sc.pl.tsne(subclustering_obj_1, color=['final_leiden_M1']))



#sc.pp.pca(subclustering_obj_1)

#https://scfates.readthedocs.io/en/latest/Basic_Curved_trajectory_analysis.html


scf.tl.curve(subclustering_obj_1,Nodes=15,use_rep="X",ndims_rep=6, epg_lambda = 0.5)

scf.pl.graph(subclustering_obj_1,basis="tsne", show = False)
plt.savefig('../figures/morph/morph_stepwise_trajectory.tiff', format='tiff')
plt.savefig('../figures/morph/morph_stepwise_trajectory.svg', format='svg')


scf.tl.root(subclustering_obj_1,0)

scf.tl.pseudotime(subclustering_obj_1,n_jobs=20,n_map=100,seed=42)

sc.pl.tsne(subclustering_obj_1,color="t")

sc.pl.tsne(subclustering_obj_1, color=subclustering_obj_1.var_names,use_raw=False)


In [ ]:
scf.pl.trajectory(subclustering_obj_1,basis="tsne",arrows=False,arrow_offset=3, show = False)
plt.savefig('../figures/morph/morph_trajectory.tiff', format='tiff')
plt.savefig('../figures/morph/morph_trajectory.svg', format='svg')

In [ ]:
sc.pl.tsne(subclustering_obj_1,color="milestones")

scf.tl.rename_milestones(subclustering_obj_1,new={'0':'Rounded','2': "Ramified"})
sc.pl.tsne(subclustering_obj_1,color="milestones")



In [ ]:
scf.pl.milestones(subclustering_obj_1,basis="tsne",annotate=True)


scf.tl.linearity_deviation(subclustering_obj_1,
                           start_milestone="Rounded",
                           end_milestone="Ramified",
                           n_jobs=20,plot=True,basis="tsne")

In [ ]:
scf.pl.linearity_deviation(subclustering_obj_1,
                           start_milestone="Rounded",
                           end_milestone="Ramified")

In [ ]:
# import rpy3
# scf.tl.test_association(subclustering_obj_1,n_jobs=20)
# scf.tl.test_association(subclustering_obj_1,reapply_filters=True,A_cut=.5)
# scf.pl.test_association(subclustering_obj_1)



#### pseudotime by neighborhood plot

In [ ]:

merged_adata_GM_sub = merged_adata_GM[merged_adata_GM.obs['id'].isin(subclustering_obj_1.obs['ImageID_CellID'])]
merged_adata_GM_sub = merged_adata_GM_sub[merged_adata_GM_sub.obs.leiden != 'Monocytes']
subclustering_obj_1.obs.index = subclustering_obj_1.obs['ImageID_CellID']
merged_adata_GM_sub.obs['pseudotime'] = subclustering_obj_1.obs.loc[merged_adata_GM_sub.obs['id']]['t'].tolist()


subclustering_obj_1 = subclustering_obj_1[merged_adata_GM_sub.obs['id'], :]
pca_coordinates = subclustering_obj_1.obsm["X_tsne"]







In [ ]:
merged_adata_GM_sub.obsm['X_tsne'] = pca_coordinates
sc.pl.tsne(merged_adata_GM_sub,color="pseudotime", show=False)
sc.pl.tsne(merged_adata_GM_sub,color="SMA", show=False, layer='zscore')
sc.pl.tsne(merged_adata_GM_sub,color="A-Beta", show=False, layer='zscore')
sc.pl.tsne(merged_adata_GM_sub,color="NeuN", show=False, layer='zscore')
sc.pl.tsne(merged_adata_GM_sub,color="GFAP", show=False, layer='zscore')


In [ ]:
# split object by AD and CNT
merged_adata_GM_sub_AD = merged_adata_GM_sub[merged_adata_GM_sub.obs['AD_CNT'] == 'AD']
merged_adata_GM_sub_CNT = merged_adata_GM_sub[merged_adata_GM_sub.obs['AD_CNT'] == 'CNT']

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

# Load your AnnData object or create one
# For example, let's say you have an AnnData object with metadata and features
adata = merged_adata_GM_sub_AD

# Define the metadata column and features you want to plot
metadata_column = 'pseudotime'
feature_names = ['Claudin5', 'SMA', 'CollagenIV', 'A-Beta', 'NeuN']  # Replace with your feature names

# Extract the feature data from .X and metadata from .obs
feature_data = adata[:, feature_names].X
metadata = adata.obs[metadata_column]

# Create a DataFrame for the stacked bar plot
import pandas as pd
df = pd.DataFrame(data=feature_data, columns=feature_names)
df[metadata_column] = metadata.tolist()
df = df.sort_values(by='pseudotime', ascending=True)
# Melt the DataFrame to create a long-format DataFrame for Seaborn
df_long = pd.melt(df, id_vars=metadata_column, value_vars=feature_names, var_name='Features')





In [ ]:


import pandas as pd
import matplotlib.pyplot as plt

# Load your data into a DataFrame
# Replace 'your_data.csv' with the path to your data file

# Define the number of time bins
num_bins = 10

# Calculate the bin edges and bin labels
bin_edges = pd.cut(df['pseudotime'], bins=num_bins, labels=False)
df['bin'] = bin_edges

# Group by the 'bin' column and calculate the mean of feature columns
binned_data = df.groupby('bin').mean()

# Plot each feature with time on the x-axis and feature value on the y-axis
for feature in df.columns:
    if feature not in ['pseudotime', 'bin']:
        plt.plot(binned_data['pseudotime'], binned_data[feature], label=feature)

# Customize the plot
plt.title('Cd163+ GM AD Neighborhood vs Pseudotime')
plt.xlabel('pseudotime')
plt.ylabel('Feature Value')

plt.legend()
plt.tight_layout()

plt.savefig('../figures/morph/Pseudotime_lineplot_AD.tiff')
plt.savefig('../figures/morph/Pseudotime_lineplot_AD.svg')

# Show the plot
plt.show()







In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

# Load your AnnData object or create one
# For example, let's say you have an AnnData object with metadata and features
adata = merged_adata_GM_sub_CNT

# Define the metadata column and features you want to plot
metadata_column = 'pseudotime'
feature_names = ['Claudin5', 'SMA', 'CollagenIV', 'A-Beta', 'NeuN']  # Replace with your feature names

# Extract the feature data from .X and metadata from .obs
feature_data = adata[:, feature_names].X
metadata = adata.obs[metadata_column]

# Create a DataFrame for the stacked bar plot
import pandas as pd
df = pd.DataFrame(data=feature_data, columns=feature_names)
df[metadata_column] = metadata.tolist()
df = df.sort_values(by='pseudotime', ascending=True)
# Melt the DataFrame to create a long-format DataFrame for Seaborn
df_long = pd.melt(df, id_vars=metadata_column, value_vars=feature_names, var_name='Features')





In [ ]:


import pandas as pd
import matplotlib.pyplot as plt

# Load your data into a DataFrame
# Replace 'your_data.csv' with the path to your data file

# Define the number of time bins
num_bins = 10

# Calculate the bin edges and bin labels
bin_edges = pd.cut(df['pseudotime'], bins=num_bins, labels=False)
df['bin'] = bin_edges

# Group by the 'bin' column and calculate the mean of feature columns
binned_data = df.groupby('bin').mean()

# Plot each feature with time on the x-axis and feature value on the y-axis
for feature in df.columns:
    if feature not in ['pseudotime', 'bin']:
        plt.plot(binned_data['pseudotime'], binned_data[feature], label=feature)

# Customize the plot
plt.title('Cd163+ GM Ctl Neighborhood vs Pseudotime')

plt.xlabel('Time')
plt.ylabel('Feature Value')
plt.legend()

# Show the plot
plt.tight_layout()

plt.savefig('../figures/morph/Pseudotime_lineplot_control.tiff')
plt.savefig('../figures/morph/Pseudotime_lineplot_control.svg')


plt.show()







# MORPH X PROTEIN CONNECTION

In [ ]:

# load in morph data 


import copy




## load adatas

In [ ]:
data_directory = "sample_adatas/"
adata_list = {}


for filename in os.listdir(data_directory):
    if filename.endswith(".h5ad"):
        file_path = os.path.join(data_directory, filename)
        adata = anndata.read_h5ad(file_path)

        file_name = re.sub('.h5ad', '', filename)
        adata.obs['origfile'] = file_name
        #adata.obs['file_leiden'] = [i + j for i, j in zip(adata.obs['origfile'], adata.obs['leiden'])]
        adata_list[file_name] = adata




### fix mistaken names

In [ ]:

    
feature_dict = {
    'Claudin-5':'Claudin5',
    'a-Beta':'A-Beta',
    'DAPI_1':'DAPI'
}
features_to_remove = ['H2A.x', 'SMA - Claudin5 +', 'GFAP + Vimentin +']


# Define the custom order for feature names
custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV','PCNA', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'GFAP', 'DAPI']


    
for adata_name in adata_list.keys():
    adata = adata_list[adata_name]
    
    adata_vars = adata.var_names
    
    for var in adata_vars:
        if var in feature_dict.keys():
            adata.var_names = [feature_dict[name] if name in feature_dict.keys() else name for name in adata.var_names] 



    
    adata.var.drop(features_to_remove, inplace=False)







In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


merged_adata = anndata.concat(adata_list, axis=0, index_unique="-")

#custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'PCNA', 'GFAP', 'DAPI_1']
# Use NumPy's argsort to get the indices that would reorder the features
order_indices = [merged_adata.var_names.get_loc(feature) for feature in custom_feature_order]

# Reorder the features based on the custom order
merged_adata = merged_adata[:, order_indices]

merged_adata.obs['origfilenum'] = merged_adata.obs['origfile'].str.split('_', expand = True)[0]
#merged_adata.obs['AD_CNT'] = merged_adata.obs['origfile'].str.split('_', expand = True)[1]
merged_adata.obs['AD_CNT'] = [ad_cnt_dict[i] for i in merged_adata.obs['origfilenum']]



#sc.pl.violin(merged_adata_GM, 'CollagenIV', groupby='leiden', show=True)


## attach clustering metadata

In [ ]:

layer = 'zscore'


# alter the following below to adapt to new methods of clustering
#new
clustering_metadata = anndata.read_h5ad('../final_h5ads/clustering_and_morph_clustering.h5')

merged_adata = merged_adata[merged_adata.obs.id.isin(clustering_metadata.obs.ImageID_CellID)]





cluster_label_order = ['Rounded', 'Intermediate', 'Ramified 1', 'Ramified 2', 'Ramified 3']
cluster_label_ticklabels = cluster_label_order

change_cluster_names_dict = {
    '1': 'Rounded',
    '2': 'Intermediate',
    '0_ram':'Ramified 2',
    '1_ram':'Ramified 3',
    '2_ram':'Ramified 1'
}

cMapDict = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs.final_leiden_M1]
clustering_metadata.obs['leiden'] = pd.Categorical(clustering_metadata.obs['leiden'], categories=cluster_label_order, ordered=False)






# also update the protein clusters



change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}




cluster_label_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2', 'default']
cluster_label_ticklabels = cluster_label_order

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['protein_leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs['protein_leiden']]

clustering_metadata.obs['protein_leiden'] = pd.Categorical(clustering_metadata.obs['protein_leiden'], categories=cluster_label_order, ordered=False)



#--------------------------------------------------------------------------------------




mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs['leiden']))
merged_adata.obs['leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['leiden']

mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs['protein_leiden']))
merged_adata.obs['protein_leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_protein_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['protein_leiden']


# also map distance metadata for later use 

distance_metrics = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta',
       'fabseg_distance_to_nearest_Astrocytes',
       'fabseg_distance_to_nearest_Neurons',
       'fabseg_distance_to_nearest_Oligodendrocytes',
       'fabseg_distance_to_nearest_dense_aBeta',
       'fabseg_distance_to_nearest_diffuse_aBeta',
        'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']

for met in distance_metrics:
    mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs[met]))
    merged_adata.obs[met] = [mapping_dict[i] for i in merged_adata.obs.id]
#merged_adata.obs['file_protein_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['protein_leiden']



# remove any meningeal macrophages

merged_adata = merged_adata[merged_adata.obs['parent'].isin(['Grey Matter', 'White Matter'])]

# remove any default clusters

merged_adata = merged_adata[merged_adata.obs['leiden'] != 'default']

merged_adata_WM = merged_adata[merged_adata.obs['parent'] == 'White Matter']
merged_adata_GM = merged_adata[merged_adata.obs['parent'] == 'Grey Matter']






merged_adata_sub = merged_adata[merged_adata.obs['id'].isin(clustering_metadata.obs['ImageID_CellID'])]
clustering_metadata.obs.index = clustering_metadata.obs['ImageID_CellID']

clustering_metadata = clustering_metadata[merged_adata_sub.obs['id'], :]

tsne_coordinates = clustering_metadata.obsm["X_tsne"]
merged_adata_sub.obsm['X_tsne'] = tsne_coordinates
protein_tsne_coordinates = clustering_metadata.obsm['X_tsne_PROTEIN']
merged_adata_sub.obsm['X_tsne_PROTEIN'] = protein_tsne_coordinates

#merged_adata.obsm.X_tsne = 

for feat in merged_adata_sub.var_names:
    df = merged_adata_sub.to_df()
    #feat_upperbound = np.percentile(a=df[feat], q=[80])

    sc.pl.tsne(merged_adata_sub, color=feat, layer=layer)
    sc.pl.scatter(merged_adata_sub, color = feat, basis = 'tsne_PROTEIN', layers = layer)
    sc.pl.violin(merged_adata_sub, feat, groupby = 'leiden', layer=layer)
    sc.pl.violin(merged_adata_sub, feat, groupby = 'protein_leiden', layer = layer)
    
    



## add protein and morph data to object 

In [ ]:
# add morph data

merged_adata_sub.obs.index = merged_adata_sub.obs.id
morph_data_raw = clustering_metadata.to_df(layer='unscaled_morph_data').loc[merged_adata_sub.obs.id]


protein_data_orig = sc.read_h5ad('../final_h5ads/fabian_metadata_plus_clustering.h5ad')

protein_data_orig_sub = protein_data_orig[protein_data_orig.obs.ImageID_CellID.isin(merged_adata_sub.obs.id)]
protein_data_raw = protein_data_orig_sub.to_df()[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']].loc[merged_adata_sub.obs.index]

merged_adata_sub.obs = pd.concat([merged_adata_sub.obs, morph_data_raw, protein_data_raw], axis = 1)


#matrix1 = protein_data_orig_sub.to_df()[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]


#merged_adata_sub.X = pd.concat([merged_adata_sub.to_df(), clustering_metadata.to_df().loc[merged_adata_sub.obs.id]], axis=0)

sc.pl.scatter(merged_adata_sub, color = 'CD163', basis = 'tsne_PROTEIN')


morph_features = list(morph_data_raw.columns)
protein_features = list(protein_data_raw.columns)

In [ ]:

sc.pl.matrixplot(merged_adata_sub, groupby = 'protein_leiden', var_names=['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45'], standard_scale='var')

In [ ]:

for cluster in clustering_metadata.obs.leiden.unique():
    sub_obj = clustering_metadata[(clustering_metadata.obs.leiden == cluster) & (clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')]
    sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='leiden')
    
sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne_PROTEIN', color='leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne_PROTEIN', color='protein_leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne', color='leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne', color='protein_leiden', size=20)


In [ ]:
cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}


# among ramified 2 cells, which are associated with Abeta, and what protein clusters do they belong to?

Abeta_pct_cutoff = 0.5
layer = 'zscore'

# Abeta_pct_cutoff = 0.1
# layer = None
from sklearn.cluster import DBSCAN

merged_adata_sub.obs['ABETA_percent'] = merged_adata_sub.to_df(layer=layer)['A-Beta']
merged_adata_sub.obs['beats_abeta_thresh'] = merged_adata_sub.obs.ABETA_percent > Abeta_pct_cutoff

sub_obj = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter') & (merged_adata_sub.obs.AD_CNT == 'AD')]


plt.hist(sub_obj.obs['ABETA_percent'])
plt.show()
sns.kdeplot(sub_obj.obs['ABETA_percent'])
plt.show()

print('what percent of mircroglia in GM AD are nearby Abeta?')
print(sum(sub_obj.obs.ABETA_percent > Abeta_pct_cutoff)/len(sub_obj.obs.ABETA_percent))

#sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='beats_abeta_thresh', legend_loc='on data')


sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='leiden', show=False, color_map=cMapDict_morph)


plt.savefig('../figures/neighborhood_exploration/morph_clusters_on_protein_tsne.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/morph_clusters_on_protein_tsne.svg', format='svg')


for var in sub_obj.var_names:
    sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color=var, show = False, layers='zscore')
    
    plt.savefig('../figures/neighborhood_exploration/' + var + '_scaled_featureplot.tiff', format='tiff')
    plt.savefig('../figures/neighborhood_exploration/' + var + '_scaled_featureplot.svg', format='svg')


coordinates = sub_obj.obsm['X_tsne_PROTEIN']
coordsx = [i[0] for i in coordinates]
coordsy = [i[1] for i in coordinates]

coordinates_dframe = pd.DataFrame({'x':coordsx, 'y':coordsy})
#hdb = DBSCAN(min_samples=20, eps = 3)
hdb = DBSCAN(min_samples=80, eps=3.6)
hdb.fit(coordinates_dframe)

sub_obj.obs['dblabels'] = hdb.labels_
sub_obj.obs['dblabels'] = pd.Categorical(sub_obj.obs['dblabels'])


sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='dblabels', legend_loc='on data')








# coordinates = only_nearby_abeta.obsm['X_tsne_PROTEIN']
# coordsx = [i[0] for i in coordinates]
# coordsy = [i[1] for i in coordinates]

# coordinates_dframe = pd.DataFrame({'x':coordsx, 'y':coordsy})
# #hdb = DBSCAN(min_samples=20, eps = 3)
# hdb = DBSCAN(min_samples=20, eps = 3.5)
# hdb.fit(coordinates_dframe)

# only_nearby_abeta.obs['dblabels'] = hdb.labels_
# only_nearby_abeta.obs['dblabels'] = pd.Categorical(only_nearby_abeta.obs['dblabels'])


# sc.pl.scatter(only_nearby_abeta, basis='tsne_PROTEIN', color='dblabels', legend_loc='on data')





In [ ]:
sub_obj

### cluster neighbor data

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
plt.rcParams['axes.grid'] = False



cmapDict_spatial  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
    }




from scipy.stats import zscore
import umap
import umap.plot

orig_df = sub_obj.to_df()
zscore_within_samples = sub_obj.to_df().groupby(sub_obj.obs['origfilenum']).transform(func = zscore, axis = 0)

zscore_all = zscore_within_samples.transform(func = zscore, axis = 0)


# perform umap dimreduction
embedding = umap.UMAP().fit(zscore_all)
print(umap.plot.points(embedding))

sub_obj.obsm['X_neighbor_umap'] = coordinates = embedding.embedding_
sub_obj.layers['neighbor_zscore_new'] = zscore_all
sub_obj.obsm['zscore_all'] = zscore_all
plt.show()


for var in sub_obj.var_names:
    

    sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, layers='zscore', show = False)
    plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '.tiff', format='tiff')
    plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '.svg', format='svg')
    
    sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, show = False)
    # plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '_unscaled.tiff', format='tiff')
    # plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '_unscaled.svg', format='svg')
    
    #sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, layers='neighbor_zscore_new')

    
    
sc.pp.neighbors(sub_obj, random_state=10, use_rep='zscore_all')

sc.tl.leiden(sub_obj, key_added='spatial_clustering')
sub_obj.obs['spatial_clustering'] = sub_obj.obs['spatial_clustering'].astype('category')









# for clus in sub_obj.obs.spatial_clustering.unique():
#     sub_obj_clus = sub_obj[sub_obj.obs.spatial_clustering == clus]
#     sc.pl.scatter(sub_obj_clus, basis = 'tsne_PROTEIN', color = 'spatial_clustering', legend_loc='on data', show=False)
#     plt.axis([-100, 100, -100, 100])
#     plt.show()

# hm = sc.pl.matrixplot(sub_obj, groupby = 'spatial_clustering', layer='zscore', var_names=sub_obj.var_names, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.svg', format='svg')

# hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=protein_features, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.svg', format='svg')

# hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=morph_features, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.svg', format='svg')



# remove artefactual cluster

sub_obj = sub_obj[sub_obj.obs.spatial_clustering != '16']
sub_obj.obs.spatial_clustering.cat.categories = list(cmapDict_spatial.keys())

#miniclus = copy.deepcopy(sub_obj)
miniclus = sub_obj.copy()


sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = 'spatial_clustering', legend_loc='on data', show = False, palette=[cmapDict_spatial[i] for i in list(sub_obj.obs.spatial_clustering.cat.categories)])
#sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = 'spatial_clustering', legend_loc='on data', show = False)

plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap.svg', format='svg')


sc.pl.scatter(sub_obj, basis = 'tsne_PROTEIN', color = 'spatial_clustering', show = False)
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_protein.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_protein.svg', format='svg')





# make custom plot:



sc.pl.scatter(sub_obj, basis = 'tsne', color = 'spatial_clustering', show = False)
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_morph.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_morph.svg', format='svg')


In [ ]:
    sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color='spatial_clustering', layers='zscore', show = False)


In [ ]:

sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = 'spatial_clustering', legend_loc='on data', show = False, palette=[cmapDict_spatial[i] for i in list(sub_obj.obs.spatial_clustering.cat.categories)])

fab_data = sub_obj.obs[['id', 'X', 'Y', 'parent', 'AD_CNT', 'origfilenum', 'spatial_clustering']]
fab_data['cellnum'] = [i[1] for i in fab_data['id'].str.split('_')]
#fab_data.to_csv('fab_data_nhood_clusters.csv')

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
plt.rcParams['axes.grid'] = False





sub_obj_2 = sub_obj.copy()

colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)


neighbor_percents = sub_obj_2.to_df()

upperlims = []

for i,n in enumerate(neighbor_percents.columns):
    nvals = neighbor_percents[n]
    upperlim = np.percentile(nvals, 99)
    
    upperlims.append(upperlim)

sub_obj_2.obsm['X_umap'] = sub_obj_2.obsm['X_neighbor_umap']
sc.pl.umap(sub_obj_2, color=sub_obj_2.var_names, vmax=upperlims, show=False, color_map=cmap)

plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling_EASYEDIT.tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling_EASYEDIT.svg', format = 'svg')

In [ ]:
colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

xvals = [i[0] for i in sub_obj.obsm['X_neighbor_umap']]
yvals = [i[1] for i in sub_obj.obsm['X_neighbor_umap']]
neighbor_percents = sub_obj.to_df()

fig, axs = plt.subplots(3, 4, figsize = (21,10))


for i,n in enumerate(neighbor_percents.columns):
    nvals = neighbor_percents[n]
    upperlim = np.percentile(nvals, 99)
    
    xpos = i//4
    ypos = i%4

    im = axs[xpos,ypos].scatter(x = xvals, y = yvals, c=neighbor_percents[n], cmap=cmap, s=1, vmax=upperlim)
    plt.colorbar(im, ax = axs[xpos,ypos])
    axs[xpos,ypos].set_title(n)
    axs[xpos,ypos].set_xticks([])
    axs[xpos,ypos].set_yticks([])
    axs[xpos,ypos].set_xlabel('UMAP1')
    axs[xpos,ypos].set_ylabel('UMAP2')


    # axs[0,i].xlabel('tSNE1')
    # plt.ylabel('tSNE2')
    # plt.title(n)
    # plt.show()
plt.tight_layout()
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.svg', format = 'svg')
plt.show()

# sc.pl.tsne(clustering_adata, color=adataScaled.var_names,use_raw=False, return_fig = True, color_map=cmap, show = False)
# plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.tiff')
# plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.svg', format = 'svg')
# sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = sub_obj.var_names)


In [ ]:

hm = sc.pl.matrixplot(sub_obj, groupby = 'spatial_clustering', layer='zscore', var_names=sub_obj.var_names, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.svg', format='svg')

hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=protein_features, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.svg', format='svg')

hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=morph_features, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.svg', format='svg')

#KILLME




        
# test = custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING.tiff')
# fjdlf

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING.svg')

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_ALT_SCALING.svg')

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_ALT_SCALING.svg')



custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_Z_SCALING.tiff')
custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_Z_SCALING.svg')

custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_Z_SCALING.tiff')
custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_Z_SCALING.svg')

custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_Z_SCALING.tiff')
custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_Z_SCALING.svg')









In [ ]:
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd
from scipy.stats import f_oneway


import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd



def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    fig = plt.figure(figsize=(10, 5))  # width=10 inches, height=5 inches

    ax = plt.axes()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova

    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]

    for c in combinations:
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])
    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    
    plt.show()
    
    
    
cmapDict_spatial  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
    }


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


proteins_to_profile = protein_features

sub_obj.obs['file_spatial_clustering'] = sub_obj.obs['origfilenum'].astype(str) + '_' +  sub_obj.obs['spatial_clustering'].astype(str)

#protdata = sub_obj.obs[proteins_to_profile]

protdata = sub_obj[sub_obj.obs['spatial_clustering'] != '12'].obs[proteins_to_profile]
protdata['sample_spatial_clustering'] = sub_obj.obs['file_spatial_clustering']
protdata_mean = protdata.groupby(by=['sample_spatial_clustering']).median()
protdata_mean['sample_spatial_clustering'] = protdata_mean.index
protdata_mean['spatial_clustering'] = protdata_mean['sample_spatial_clustering'].str.split('_', expand = True)[1]
protdata_mean['sample_num'] = protdata_mean['sample_spatial_clustering'].str.split('_', expand = True)[0]
protdata_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in protdata_mean['sample_num']]

cluster_label_order = [str(i) for i in range(16)]
cluster_label_order.remove('12')

cluster_label_ticklabels = cluster_label_order



minmax_df_mean = protdata_mean
for protein in ['CD163']:
    print(protein)
    
    data = []
    title = protein + 'Gray Matter'
    ylabel = 'average protein expression'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['spatial_clustering'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_spatial_clustering.to_list()
        data_anova_df_samplecol += sub_df['sample_num'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = [str(i) for i in range(16)], ordered = True)

    lel = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cmapDict_spatial.values())  











In [ ]:

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_ALT_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_ALT_SCALING_MEDIAN.svg', style = 'median')

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING_MEDIAN.svg', style = 'median')

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_ALT_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_ALT_SCALING_MEDIAN.svg', style = 'median')


In [ ]:

custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_Z_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_Z_SCALING_MEDIAN.svg', style = 'median')


custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_Z_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_Z_SCALING_MEDIAN.svg', style = 'median')

custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_Z_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap_Z(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_Z_SCALING_MEDIAN.svg', style = 'median')



In [ ]:
custom_scaled_heatmap_minmax(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_minmax_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap_minmax(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_minmax_SCALING_MEDIAN.svg', style = 'median')


custom_scaled_heatmap_minmax(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_minmax_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap_minmax(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_minmax_SCALING_MEDIAN.svg', style = 'median')

custom_scaled_heatmap_minmax(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_minmax_SCALING_MEDIAN.tiff', style = 'median')
custom_scaled_heatmap_minmax(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_minmax_SCALING_MEDIAN.svg', style = 'median')


In [ ]:
sc.pl.violin(sub_obj, 'CD163', 'spatial_clustering')
sc.pl.violin(sub_obj, 'TMEM119', 'spatial_clustering')
sc.pl.violin(sub_obj, 'HLA-DR', 'spatial_clustering')
sc.pl.violin(sub_obj, 'CD11c', 'spatial_clustering')

In [ ]:
sc.pl.violin(sub_obj, 'CD163', 'protein_leiden')
sc.pl.violin(sub_obj, 'TMEM119', 'protein_leiden')
sc.pl.violin(sub_obj, 'HLA-DR', 'protein_leiden')
sc.pl.violin(sub_obj, 'CD11c', 'protein_leiden')

In [ ]:

for group in set(sub_obj.obs.protein_leiden):
    
    if group == 'BLM':
        continue
    
    import numpy as np
    from scipy.stats import ttest_ind

    distribution1 = sub_obj[sub_obj.obs.protein_leiden == 'BLM'].obs['CD163']
    distribution2 = sub_obj[sub_obj.obs.protein_leiden == group].obs['CD163']

    # Perform t-test
    print('---------------------------------')

    print(group)
    t_statistic, p_value = ttest_ind(distribution1, distribution2)
    print(f"T-statistic: {t_statistic}")
    print(f"P-value: {p_value}")











## cd163 and tmem119 expression

In [ ]:
# load in files
def compare_cells_simple(obj, groupby, sample_col, col, saveto = False, title='', pal = None):
    from scipy.stats import ttest_ind
    from decimal import Decimal

    
    samples = []
    props = []
    groups = []
    
    
    for i in obj.obs[groupby].cat.categories:
    
        group_obj = obj[obj.obs[groupby] == i]
        
        for samp in set(group_obj.obs[sample_col]):
            prop = sum(group_obj[group_obj.obs[sample_col] == samp].obs[col] == 1)/len(group_obj[group_obj.obs[sample_col] == samp].obs[col])
            
            samples.append(samp)
            props.append(prop)
            groups.append(i)
    
    samp_prop_dframe = pd.DataFrame({'group':groups, 'sample':samples, 'proportion':props}) 
    plt.figure(figsize = [6.4, 4.8])
    ax = sns.barplot(samp_prop_dframe, x = 'group', y = 'proportion', palette=pal)
    sns.stripplot(samp_prop_dframe, x = 'group', y = 'proportion', jitter=False, color = 'black', size = 6, marker="D")
    ax.set(xlabel=None)
    ax.set_title(title)
    
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()


cmapDict_spatial  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
        "16": "#f7b6d2"
    }

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}




dir_path = '../binary_marker_expression/13_06_2024_new_marker_detections/'
marker_dframes = []


for file in os.listdir(dir_path):
    
    marker_dframe = pd.read_csv(dir_path + file)
    
    filenum = file.split('_')[0]
    marker_dframe.index = filenum + '_' + marker_dframe['cell'].astype(str)
    marker_dframes.append(marker_dframe)

all_marker_dframes = pd.concat(marker_dframes)

# add data to object

cols_to_analyze = ['CD163_positive', 'TMEM119_positive']

for col in all_marker_dframes:
    merged_adata_sub.obs[col] = all_marker_dframes.loc[merged_adata_sub.obs.index][col]

for col in all_marker_dframes:
    sub_obj.obs[col] = all_marker_dframes.loc[sub_obj.obs.index][col]

# analyze nhood clusters

obj = sub_obj.copy()
groupby = 'spatial_clustering'
sample_col = 'origfilenum'

for col in cols_to_analyze:
    
    clusters = obj.obs[groupby].cat.categories
    # Generate unique colors for each bar
    
    title = groupby + col + '_sample_mean_proportions'
    
    compare_cells_simple(obj, groupby, sample_col, col, '../figures/protein_binary_expression/' + title + '.svg', title, cmapDict_spatial)
    compare_cells_simple(obj, groupby, sample_col, col, '../figures/protein_binary_expression/' + title + '.tiff', title, cmapDict_spatial)
    

obj = merged_adata_sub.copy()
groupby = 'protein_leiden'
sample_col = 'origfilenum'

for col in cols_to_analyze:
    
    clusters = obj.obs[groupby].cat.categories
    # Generate unique colors for each bar
    
    title = groupby + col + '_sample_mean_proportions'
    
    compare_cells_simple(obj, groupby, sample_col, col, '../figures/protein_binary_expression/' + title + '.svg', title, cMapDict)
    compare_cells_simple(obj, groupby, sample_col, col, '../figures/protein_binary_expression/' + title + '.tiff', title, cMapDict)
    

    
    

def compare_cells(obj, groupby, sample_col, col, saveto = False, title=''):
    from scipy.stats import ttest_ind
    from decimal import Decimal

    
    samples = []
    props = []
    groups = []
    
    
    for i in obj.obs[groupby].cat.categories:
    
        group_obj = obj[obj.obs[groupby] == i]
        
        for samp in set(group_obj.obs[sample_col]):
            prop = sum(group_obj[group_obj.obs[sample_col] == samp].obs[col] == 1)/len(group_obj[group_obj.obs[sample_col] == samp].obs[col])
            
            samples.append(samp)
            props.append(prop)
            groups.append(i)
    
    samp_prop_dframe = pd.DataFrame({'group':groups, 'sample':samples, 'proportion':props}) 
    plt.figure()
    ax = sns.barplot(samp_prop_dframe, x = 'group', y = 'proportion', color = 'lightblue')
    sns.stripplot(samp_prop_dframe, x = 'group', y = 'proportion', jitter=False, color = 'black', size = 8, marker="D")
    ax.set(xlabel=None)
    ax.set_title(title)
    
    total_height = max(props) * 1.1
    cats = obj.obs[groupby].cat.categories
    
    #plt.plot(['AD', 'AD', 'CNT', 'CNT'], [total_height, total_height * 1.1, total_height * 1.1, total_height], color = 'black')
    plt.plot([cats[0], cats[0], cats[1], cats[1]], [total_height, total_height * 1.1, total_height * 1.1, total_height], color = 'black')

    
    
    
    
    
    
    
    
    
    group1_props = samp_prop_dframe[samp_prop_dframe['group'] == cats[0]]['proportion']
    group2_props = samp_prop_dframe[samp_prop_dframe['group'] == cats[1]]['proportion']
    
    t_statistic, p_value = ttest_ind(group1_props, group2_props)
    
    if p_value < 0.001:
        ast = '***'
    elif p_value < 0.01:
        ast = '**'
    elif p_value < 0.05:
        ast = '*'
    else:
        ast = 'ns'
    
    plt.text((0 + 1)*.5, total_height * 1.2, 'p = ' + '%.2E' % Decimal(str(p_value)) + ' (' + ast + ')', ha='center', va='bottom')
    
    plt.ylim(top = total_height * 1.3)
    
    #plt.text((x1+x2)*.5, y+h, "ns", ha='center', va='bottom', color=col)
    
    if saveto:
        plt.savefig(saveto)
    else:
        plt.show()




regs = [['Grey Matter', 'White Matter'], ['Grey Matter'], ['White Matter']]
regnames = ['GM_and_WM', 'GM', 'WM']

clusters_i = [[i] for i in list(set(merged_adata_sub.obs['protein_leiden']))]
all_clusters_i = list(set(merged_adata_sub.obs['protein_leiden']))
clusters_i.append(all_clusters_i)

clusters_i_names = [i for i in list(set(merged_adata_sub.obs['protein_leiden']))]
clusters_i_names.append('all_clusters')

for col in cols_to_analyze:
    for regint, reg in enumerate(regs):
        print(reg)

        regname = regnames[regint]

        reg_subobj = merged_adata_sub[merged_adata_sub.obs['parent'].isin(reg)]


        for clusint, clus in enumerate(clusters_i):
            print(clus)

            clusname = clusters_i_names[clusint]

            clus_subobj = reg_subobj[reg_subobj.obs['protein_leiden'].isin(clus)]
        
            titlei = str(regname) + '_' +  str(clusname) + '_' + str(col) + '_AD_CNT_comparison'
            print(titlei)
            
            
            
            
            compare_cells(clus_subobj, 'AD_CNT', 'origfilenum', col, '../figures/protein_binary_expression/' + titlei + '.tiff', titlei)
            compare_cells(clus_subobj, 'AD_CNT', 'origfilenum', col, '../figures/protein_binary_expression/' + titlei + '.svg', titlei)

            
        





    
# show proportion of positive cells in each nhood cluster

# obj = copy.deepcopy(sub_obj)
# groupby = 'spatial_clustering'
# sample_col = 'origfilenum'

# samples = []
# props = []
# clusters = []

# for col in cols_to_analyze:
#     for samp in set(obj.obs[sample_col]):
        
#         samp_spec_obj = obj[obj.obs[sample_col] == samp]
#         samp_total_positive = sum(samp_spec_obj.obs[col] == 1)
#         total_cells_in_sample = sum(samp_spec_obj.obs[col] == 1) + sum(samp_spec_obj.obs[col] == 0)
        
        
#         for i in samp_spec_obj.obs[groupby].cat.categories:
        
#             prop = sum(samp_spec_obj[samp_spec_obj.obs[groupby] == i].obs[col] == 1)/samp_total_positive
#             total_cells_in_sample_cluster = sum(samp_spec_obj.obs[groupby] == i)
#             prop_adj = prop/(total_cells_in_sample_cluster/total_cells_in_sample)
            
            
            
            
            
#             samples.append(samp)
#             props.append(prop_adj)
#             clusters.append(i)
    
#     samp_prop_dframe = pd.DataFrame({'cluster':clusters, 'sample':samples, 'proportion':props})
#     sns.barplot(samp_prop_dframe, x = 'cluster', y = 'proportion')
#     plt.show()
#     dhfddjfld
        
        
#     # Extracting keys and values
#     clusters = list(prop_dict.keys())
#     proportions = list(prop_dict.values())

#     # Generate unique colors for each bar
#     colors = [cmapDict_spatial[i] for i in clusters]

#     # Creating the bar chart
#     plt.figure(figsize=(10, 6))
#     plt.bar(clusters, proportions, color=colors)

#     # Adding titles and labels
#     plt.title('Proportions of Each Cluster')
#     plt.xlabel('Clusters')
#     plt.ylabel('Proportion')

#     # Display the plot
#     plt.savefig('../figures/protein_binary_expression/' + col + '_nhood_clusters.tiff')
#     plt.savefig('../figures/protein_binary_expression/' + col + '_nhood_clusters.svg')

#     plt.show()
    

In [ ]:
# show proportion of positive cells in each cluster



In [ ]:
## cutoff remnant

In [ ]:
## cutoff remnant

In [ ]:
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd
from scipy.stats import f_oneway


import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    fig = plt.figure(figsize=(7, 5))  # width=10 inches, height=5 inches

    ax = plt.axes()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova

    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]

    for c in combinations:
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])
    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    
    plt.show()
    
    


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e'
}







proteins_to_profile = protein_features

merged_adata_sub.obs['file_protein_leiden'] = merged_adata_sub.obs['origfilenum'].astype(str) + '_' +  merged_adata_sub.obs['protein_leiden'].astype(str)
protdata = merged_adata_sub.obs[proteins_to_profile]
protdata['sample_protein_leiden'] = merged_adata_sub.obs['file_protein_leiden']
protdata_mean = protdata.groupby(by=['sample_protein_leiden']).mean()
protdata_mean['sample_protein_leiden'] = protdata_mean.index
protdata_mean['protein_leiden'] = protdata_mean['sample_protein_leiden'].str.split('_', expand = True)[1]
protdata_mean['sample_num'] = protdata_mean['sample_protein_leiden'].str.split('_', expand = True)[0]
protdata_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in protdata_mean['sample_num']]

cluster_label_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2']
cluster_label_ticklabels = cluster_label_order



minmax_df_mean = protdata_mean
for protein in ['CD163']:
    print(protein)
    
    data = []
    title = protein 
    ylabel = 'average protein expression'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['protein_leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_protein_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample_num'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_protein_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_protein_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)
    box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= '../figures/protein/' + protein + '_protein_leiden_box&whisker_ANOVA.tiff')  
    






In [ ]:

anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
print(res)


In [ ]:
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd
from scipy.stats import f_oneway


import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    fig = plt.figure(figsize=(10, 5))  # width=10 inches, height=5 inches

    ax = plt.axes()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova

    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]

    for c in combinations:
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])
    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    
    plt.show()
    
    
    
cmapDict_spatial  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e"
    }


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


proteins_to_profile = protein_features

sub_obj.obs['file_spatial_clustering'] = sub_obj.obs['origfilenum'].astype(str) + '_' +  sub_obj.obs['spatial_clustering'].astype(str)

protdata = sub_obj.obs[proteins_to_profile]
protdata['sample_spatial_clustering'] = sub_obj.obs['file_spatial_clustering']
protdata_mean = protdata.groupby(by=['sample_spatial_clustering']).mean()
protdata_mean['sample_spatial_clustering'] = protdata_mean.index
protdata_mean['spatial_clustering'] = protdata_mean['sample_spatial_clustering'].str.split('_', expand = True)[1]
protdata_mean['sample_num'] = protdata_mean['sample_spatial_clustering'].str.split('_', expand = True)[0]
protdata_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in protdata_mean['sample_num']]

cluster_label_order = [str(i) for i in range(16)]
cluster_label_ticklabels = cluster_label_order



minmax_df_mean = protdata_mean
for protein in ['CD163']:
    print(protein)
    
    data = []
    title = protein + 'Gray Matter'
    ylabel = 'average protein expression'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['spatial_clustering'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_spatial_clustering.to_list()
        data_anova_df_samplecol += sub_df['sample_num'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = [str(i) for i in range(16)], ordered = True)

    lel = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cmapDict_spatial.values())  




In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
plt.rcParams['axes.grid'] = False


ax = sc.pl.scatter(sub_obj, color=["spatial_clustering"], groups=["0", "9", "15"], show=False, basis = 'tsne_PROTEIN', size = 7, legend_loc='none')
ax.set_rasterized(True)
# We can change the 'NA' in the legend that represents all cells outside of the
# specified groups
# legend_texts = ax.get_legend().get_texts()
# # Find legend object whose text is "NA" and change it
# for legend_text in legend_texts:
#     if legend_text.get_text() == "NA":
#         legend_text.set_text("other cell types")
        
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_only_3clusters.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_only_3clusters.svg', format='svg')


sc.pl.scatter(merged_adata_sub[(merged_adata_sub.obs.AD_CNT == 'AD') & (merged_adata_sub.obs.parent == 'Grey Matter')], color = 'protein_leiden', basis = 'tsne_PROTEIN')
sc.pl.scatter(merged_adata_sub[(merged_adata_sub.obs.AD_CNT == 'CNT') & (merged_adata_sub.obs.parent == 'Grey Matter')], color = 'protein_leiden', basis = 'tsne_PROTEIN')



import matplotlib.colors as mcolors

# Get the colormap
viridis = plt.cm.get_cmap('viridis')

last_color = viridis(255)
first_color = viridis(0)

last_color_hex = mcolors.to_hex(last_color)
first_color_hex = mcolors.to_hex(first_color)


# manually select region
sc.pl.scatter(merged_adata_sub, color = 'protein_leiden', basis = 'tsne_PROTEIN')

posframe = pd.DataFrame({'x':[i[0] for i in merged_adata_sub.obsm['X_tsne_PROTEIN']], 'y':[i[1] for i in merged_adata_sub.obsm['X_tsne_PROTEIN']], 'cell':merged_adata_sub.obs['id'], 'protclus':merged_adata_sub.obs['protein_leiden']})

selected_cells = posframe[(posframe['y'] > -11) & (posframe['y'] < -2) & (posframe['x'] < -61)]['cell']

merged_adata_sub.obs['selected'] = merged_adata_sub.obs['id'].isin(selected_cells)
merged_adata_sub.obs['selected'] = merged_adata_sub.obs['selected'].astype('category')



sc.pl.scatter(merged_adata_sub, color = 'selected', basis = 'tsne_PROTEIN', color_map = 'viridis', size = 7, legend_loc='none', palette=[first_color_hex, last_color_hex])

sc.pl.scatter(merged_adata_sub[(merged_adata_sub.obs.AD_CNT == 'AD') & (merged_adata_sub.obs.parent == 'Grey Matter')], color = 'selected', basis = 'tsne_PROTEIN', show=False, size = 7, legend_loc='none')
plt.savefig('../figures/neighborhood_exploration/SELECTED_POP_AD.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/SELECTED_POP_AD.svg', format='svg')
sc.pl.scatter(merged_adata_sub[(merged_adata_sub.obs.AD_CNT == 'CNT') & (merged_adata_sub.obs.parent == 'Grey Matter')], color = 'selected', basis = 'tsne_PROTEIN', show = False, size = 7, legend_loc='none')
plt.savefig('../figures/neighborhood_exploration/SELECTED_POP_CONTROL.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/SELECTED_POP_CONTROL.svg', format='svg')
sc.pl.scatter(merged_adata_sub, color = 'CD11c', basis = 'tsne_PROTEIN')
sc.pl.scatter(merged_adata_sub, color = 'A-Beta', basis = 'tsne_PROTEIN', layers='zscore')
sc.pl.scatter(merged_adata_sub, color = 'HLA-DR', basis = 'tsne_PROTEIN')
sc.pl.scatter(merged_adata_sub, color = 'CD68', basis = 'tsne_PROTEIN', layers='zscore')
sc.pl.scatter(merged_adata_sub, color = 'CD45', basis = 'tsne_PROTEIN', layers='zscore')


In [ ]:




# makde data frame 
plt.figure()
X = [i[0] for i in sub_obj.obsm['X_tsne_PROTEIN']]
Y = [i[1] for i in sub_obj.obsm['X_tsne_PROTEIN']]
cat = [i if i in ['0', '9', '15'] else 'none' for i in sub_obj.obs.spatial_clustering]
preferred_order = {'15': 1, '9': 2, '0': 3, 'none':4}

sorted_indices = sorted(range(len(cat)), key=lambda x: preferred_order[cat[x]])[::-1]
Xnew = [X[i] for i in sorted_indices]
Ynew = [Y[i] for i in sorted_indices]
catnew = [cat[i] for i in sorted_indices]

cmapDict_stupid  = {
        "0": "#1f77b4",
        "9": "#2ca02c",
        "15": "#ff7f0e",
        "none":'#D3D3D3'
    }

colorcat = [cmapDict_stupid[i] for i in catnew]

plt.scatter(x = Xnew, y = Ynew, c=colorcat, rasterized = True, s = 0.6)
plt.axis('off')
plt.savefig('test1.svg')



#------------------------------------------------



import matplotlib.colors as mcolors

# Get the colormap
viridis = plt.cm.get_cmap('viridis')

last_color = viridis(255)
first_color = viridis(0)

last_color_hex = mcolors.to_hex(last_color)
first_color_hex = mcolors.to_hex(first_color)


# manually select region
posframe = pd.DataFrame({'x':[i[0] for i in merged_adata_sub.obsm['X_tsne_PROTEIN']], 'y':[i[1] for i in merged_adata_sub.obsm['X_tsne_PROTEIN']], 'cell':merged_adata_sub.obs['id'], 'protclus':merged_adata_sub.obs['protein_leiden']})

selected_cells = posframe[(posframe['y'] > -11) & (posframe['y'] < -2) & (posframe['x'] < -61)]['cell']

merged_adata_sub.obs['selected'] = merged_adata_sub.obs['id'].isin(selected_cells)
merged_adata_sub.obs['selected'] = merged_adata_sub.obs['selected'].astype('category')

plt.figure()

miniobj = merged_adata_sub[(merged_adata_sub.obs.AD_CNT == 'AD') & (merged_adata_sub.obs.parent == 'Grey Matter')]
print(collections.Counter(miniobj.obs.AD_CNT))
print(collections.Counter(miniobj.obs.parent))


X = [i[0] for i in miniobj.obsm['X_tsne_PROTEIN']]
Y = [i[1] for i in miniobj.obsm['X_tsne_PROTEIN']]
cat = miniobj.obs.selected
preferred_order = {True: 1, False: 2}

sorted_indices = sorted(range(len(cat)), key=lambda x: preferred_order[cat[x]])[::-1]
Xnew = [X[i] for i in sorted_indices]
Ynew = [Y[i] for i in sorted_indices]
catnew = [cat[i] for i in sorted_indices]

cmapDict_stupid  = {
        True: last_color_hex,
        False: first_color_hex
    }

colorcat = [cmapDict_stupid[i] for i in catnew]

plt.scatter(x = Xnew, y = Ynew, c=colorcat, rasterized = True, s = 0.6)
plt.axis('off')
plt.savefig('test2.svg')


plt.figure()

miniobj = merged_adata_sub[(merged_adata_sub.obs.AD_CNT == 'CNT') & (merged_adata_sub.obs.parent == 'Grey Matter')]
print(collections.Counter(miniobj.obs.AD_CNT))
print(collections.Counter(miniobj.obs.parent))


X = [i[0] for i in miniobj.obsm['X_tsne_PROTEIN']]
Y = [i[1] for i in miniobj.obsm['X_tsne_PROTEIN']]
cat = miniobj.obs.selected
preferred_order = {True: 1, False: 2}

sorted_indices = sorted(range(len(cat)), key=lambda x: preferred_order[cat[x]])[::-1]
Xnew = [X[i] for i in sorted_indices]
Ynew = [Y[i] for i in sorted_indices]
catnew = [cat[i] for i in sorted_indices]

cmapDict_stupid  = {
        True: last_color_hex,
        False: first_color_hex
    }

colorcat = [cmapDict_stupid[i] for i in catnew]


plt.scatter(x = Xnew, y = Ynew, c=colorcat, rasterized = True, s = 0.6)
plt.axis('off')
plt.savefig('test3.svg')






sc.pl.scatter(merged_adata_sub, color = 'selected', basis = 'tsne_PROTEIN', color_map = 'viridis', size = 7, legend_loc='none', palette=[first_color_hex, last_color_hex])

sc.pl.scatter(merged_adata_sub[(merged_adata_sub.obs.AD_CNT == 'AD') & (merged_adata_sub.obs.parent == 'Grey Matter')], color = 'selected', basis = 'tsne_PROTEIN', show=False, size = 7, legend_loc='none')
plt.savefig('../figures/neighborhood_exploration/SELECTED_POP_AD.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/SELECTED_POP_AD.svg', format='svg')
sc.pl.scatter(merged_adata_sub[(merged_adata_sub.obs.AD_CNT == 'CNT') & (merged_adata_sub.obs.parent == 'Grey Matter')], color = 'selected', basis = 'tsne_PROTEIN', show = False, size = 7, legend_loc='none')
plt.savefig('../figures/neighborhood_exploration/SELECTED_POP_CONTROL.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/SELECTED_POP_CONTROL.svg', format='svg')











In [ ]:
cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}

sub_obj_withcontrol = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter')]
sub_obj_withcontrol.obs['combined_column'] = sub_obj_withcontrol.obs['AD_CNT'].astype(str) + sub_obj_withcontrol.obs['beats_abeta_thresh'].astype(str)

translator_dict = {
    'CNTTrue':'Control',
    'CNTFalse':'Control',
    'ADTrue':'AD + high A-beta',
    'ADFalse':'AD + low A-beta'

}


sub_obj_withcontrol.obs['combined_column_translated'] = [translator_dict[i] for i in sub_obj_withcontrol.obs['combined_column']]


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': sub_obj_withcontrol.obs['combined_column_translated'],
    'Category2': sub_obj_withcontrol.obs['leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. Protein Expression clustering')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.savefig('../figures/neighborhood_exploration/adlevels_morph_barchart.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/adlevels_morph_barchart.svg', format='svg')

print(normalized_data)


# also split by population

for protein_cluster in sub_obj_withcontrol.obs.protein_leiden.unique():
    
    

    # Sample data as a pandas DataFrame
    data = pd.DataFrame({
        'Category1': sub_obj_withcontrol[sub_obj_withcontrol.obs.protein_leiden == protein_cluster].obs['combined_column_translated'],
        'Category2': sub_obj_withcontrol[sub_obj_withcontrol.obs.protein_leiden == protein_cluster].obs['leiden']
    })

    # Group the data by Category1 and count occurrences of Category2
    grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

    # Normalize the data to make it proportional
    normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

    # Create the proportional stacked bar chart
    ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

    # Set labels and title
    plt.xlabel('Proportion')
    plt.ylabel('Protein expression clusters')
    plt.title('Morphology vs. Protein Expression clustering_' + protein_cluster)
    
    

    # Show the legend
    plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

    print('---------------')
    print(protein_cluster)
    print(normalized_data)



In [ ]:
# order mini clusters by abeta neighbors

only_abeta_scores = miniclus.to_df(layer='zscore')['A-Beta']
abeta_score_df = pd.DataFrame({'cluster': miniclus.obs.spatial_clustering, 'abeta':only_abeta_scores})
abeta_score_df_bulked = abeta_score_df.groupby('cluster').mean()
abeta_score_df_bulked = abeta_score_df_bulked.sort_values(by = 'abeta', ascending=False)

#miniclus.obs['spatial_clustering'] = pd.Categorical(miniclus.obs['spatial_clustering'], categories=list(abeta_score_df_bulked.index))


sc.pl.matrixplot(miniclus, groupby='spatial_clustering', var_names=miniclus.var_names, layer='zscore', standard_scale='var')
sc.pl.matrixplot(miniclus, groupby='spatial_clustering', var_names=miniclus.var_names, layer='zscore')

sc.pl.matrixplot(miniclus, groupby='spatial_clustering', var_names=protein_features, standard_scale='var')
sc.pl.matrixplot(miniclus, groupby='spatial_clustering', var_names=morph_features, standard_scale='var')



In [ ]:
cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': miniclus.obs['spatial_clustering'],
    'Category2': miniclus.obs['leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)
normalized_data = normalized_data.loc[normalized_data.index[::-1]]

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. sub-clusters')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.savefig('../figures/neighborhood_exploration/nhood_clusters_morph_barchart.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_morph_barchart.svg', format='svg')






In [ ]:
# is there a correlation between ram 2 and abeta percent? 




for morph_type in normalized_data.columns:
    print(morph_type)
    
    
    data1 = abeta_score_df_bulked['abeta']
    data2 = normalized_data.loc[abeta_score_df_bulked.index][morph_type]
    
    coefficients1 = np.polyfit(data1, data2, 1)
    poly1 = np.poly1d(coefficients1)

    plt.scatter(data1, data2)
    plt.xlabel('Abeta zscore')
    plt.ylabel(morph_type + ' proportion')
    plt.plot(data1, poly1(data1), color='blue', linestyle='-', label='Line of Best Fit (Dataset 1)')

    
    plt.show()
    
    print(data1.corr(data2, method='spearman'))
    print('-------------------')

In [ ]:
cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}

sub_obj_withcontrol = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter')]
sub_obj_withcontrol.obs['combined_column'] = sub_obj_withcontrol.obs['AD_CNT'].astype(str) + sub_obj_withcontrol.obs['beats_abeta_thresh'].astype(str)

translator_dict = {
    'CNTTrue':'Control',
    'CNTFalse':'Control',
    'ADTrue':'AD + high A-beta',
    'ADFalse':'AD + low A-beta'

}


sub_obj_withcontrol.obs['combined_column_translated'] = [translator_dict[i] for i in sub_obj_withcontrol.obs['combined_column']]


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': sub_obj_withcontrol.obs['combined_column_translated'],
    'Category2': sub_obj_withcontrol.obs['protein_leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. Protein Expression clustering')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.savefig('../figures/neighborhood_exploration/adlevels_protein_barchart.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/adlevels_protein_barchart.svg', format='svg')

normalized_data

In [ ]:
cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': miniclus.obs['spatial_clustering'],
    'Category2': miniclus.obs['protein_leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)
normalized_data = normalized_data.loc[normalized_data.index[::-1]]

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Protein vs. sub-clusters')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')


plt.savefig('../figures/neighborhood_exploration/nhood_clusters_protein_barchart.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_protein_barchart.svg', format='svg')

In [ ]:

for protein_type in normalized_data.columns:
    print(protein_type)
    
    
    data1 = abeta_score_df_bulked['abeta']
    data2 = normalized_data.loc[abeta_score_df_bulked.index][protein_type]
    
    coefficients1 = np.polyfit(data1, data2, 1)
    poly1 = np.poly1d(coefficients1)

    plt.scatter(data1, data2)
    plt.xlabel('Abeta zscore')
    plt.ylabel(protein_type + ' proportion')
    plt.plot(data1, poly1(data1), color='blue', linestyle='-', label='Line of Best Fit (Dataset 1)')

    plt.show()
    
    print(data1.corr(data2, method='spearman'))
    print('-------------------')





#### save object with neighborhood clusters

In [ ]:
sub_obj.write_h5ad('../final_h5ads/sub_obj_with_spatial_clustering.h5ad')


## ranking all features by prediction power over abeta (via simple correlation)


In [ ]:

# first create a data frame

merged_adata_spec = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter') & (merged_adata_sub.obs.AD_CNT == 'AD')]

Abeta_nhood = merged_adata_spec.to_df(layer = 'zscore')['A-Beta']
nhood_frame = merged_adata_spec.to_df(layer = 'zscore')[['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'PCNA', 'MAP2','Neurofilament', 'NeuN', 'GFAP', 'DAPI']]
nhood_frame = nhood_frame.add_prefix('nhood_')
expression_frame = merged_adata_spec.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_frame = merged_adata_spec.obs[['area µm²', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um',
       'bi_um']]





combined_frame = pd.concat([nhood_frame, expression_frame, morph_frame], axis = 1)

cat_dict_keys = combined_frame.columns
cat_dict_vals = pd.Series(['nhood']*len(nhood_frame.columns) + ['prot']*len(expression_frame.columns) + ['morph']*len(morph_frame.columns))
cat_dictionary = dict(zip(cat_dict_keys, cat_dict_vals))




# correlate with abeta nhood

corr_res_pearson = combined_frame.corrwith(Abeta_nhood, method = 'pearson')
corr_res_spearman = combined_frame.corrwith(Abeta_nhood, method = 'spearman')


cor_res_results = [corr_res_pearson, corr_res_spearman]
cor_res_names = ['pearson', 'spearman']

for i,cor in enumerate(cor_res_results):
    feats = cor.index.to_list()
    cats = [cat_dictionary[i] for i in feats]
    corrs = cor.values
    abs_corrs = np.abs(cor.values)
    
    adcorr_df = pd.DataFrame({'feature':feats, 'category':cats, 'correlation with Abeta in nhood':corrs, 'abs_corr':abs_corrs})
    adcorr_df['category'] = adcorr_df['category'].astype('category', )
    adcorr_df = adcorr_df.sort_values(by='abs_corr', ascending = False)
    #adcorr_df['feature'] = adcorr_df['feature'].astype('category')
    plt.figure(figsize=[20,10])
    sns.barplot(data = adcorr_df, y = 'feature', x = 'correlation with Abeta in nhood', hue = 'category', width = 1, dodge=False)
    plt.tight_layout()
    
    plt.savefig('../figures/ML_exploration/' + cor_res_names[i] + '_feature_correlations_ALL_ABETA.svg')
    plt.savefig('../figures/ML_exploration/' + cor_res_names[i] + '_feature_correlations_ALL_ABETA.tiff')

    
    
# which modality, or combination of madailties, allows for the best prediction power over 
    



    
    
    

### how accurate is a predictor built on all features?

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
sns.set(font="Arial")

sns.set_style('white')

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


merged_adata_spec = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter') & (merged_adata_sub.obs.AD_CNT == 'AD')]

Abeta_nhood = merged_adata_spec.to_df(layer = 'minmax')['A-Beta']
nhood_frame = merged_adata_spec.to_df(layer = 'minmax')[['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'PCNA', 'MAP2','Neurofilament', 'NeuN', 'GFAP', 'DAPI']]
nhood_frame = nhood_frame.add_prefix('nhood_')
expression_frame = merged_adata_spec.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_frame = merged_adata_spec.obs[['area µm²', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um',
       'bi_um']]



combined_frame = pd.concat([nhood_frame, expression_frame, morph_frame], axis = 1)





# Split the data into features (X) and target (y)
X = combined_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Standardize the features if necessary
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)




# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual')
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction')

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('Normalized Abeta')
plt.legend()

plt.savefig('../figures/ML_exploration/all_features_ML_ALL_ABETA.svg')
plt.savefig('../figures/ML_exploration/all_features_ML_ALL_ABETA.tiff')



def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual')
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction')

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()


plt.savefig('../figures/ML_exploration/all_features_ML_ALL_ABETA_averaged.svg')
plt.savefig('../figures/ML_exploration/all_features_ML_ALL_ABETA_averaged.tiff')



### how accurate is a predictor built on only protein features?

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = expression_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual')
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction')

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/protein_features_ML_ALL_ABETA.svg')
plt.savefig('../figures/ML_exploration/protein_features_ML_ALL_ABETA.tiff')



def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual')
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction')

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/protein_features_ML_ALL_ABETA_averaged.svg')
plt.savefig('../figures/ML_exploration/protein_features_ML_ALL_ABETA_averaged.tiff')


### how accurate is a predictor built on only morph features?

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = morph_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual')
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction')

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/morph_features_ML_ALL_ABETA.svg')
plt.savefig('../figures/ML_exploration/morph_features_ML_ALL_ABETA.tiff')


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual')
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction')

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/morph_features_ML_ALL_ABETA_averaged.svg')
plt.savefig('../figures/ML_exploration/morph_features_ML_ALL_ABETA_averaged.tiff')


### how accurate is a predictor built on only neighborhood_features?

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = nhood_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual')
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction')

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/nhood_features_ML_ALL_ABETA.svg')
plt.savefig('../figures/ML_exploration/mhood_features_ML_ALL_ABETA.tiff')


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual')
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction')

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/nhood_features_ML_ALL_ABETA_averaged.svg')
plt.savefig('../figures/ML_exploration/nhood_features_ML_ALL_ABETA_averaged.tiff')



### how accurate is a binary classifier built on all features?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report



merged_adata_spec = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter') & (merged_adata_sub.obs.AD_CNT == 'AD')]

Abeta_nhood = merged_adata_spec.to_df()['A-Beta']
nhood_frame = merged_adata_spec.to_df()[['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'PCNA', 'MAP2','Neurofilament', 'NeuN', 'GFAP', 'DAPI']]
nhood_frame = nhood_frame.add_prefix('nhood_')
expression_frame = merged_adata_spec.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_frame = merged_adata_spec.obs[['area µm²', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um',
       'bi_um']]


combined_frame = pd.concat([nhood_frame, expression_frame, morph_frame], axis = 1)

Abeta_nhood = Abeta_nhood > 0.05



# Split the data into features (X) and target (y)
X = combined_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Standardize the features if necessary
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)




model = RandomForestClassifier(n_estimators=200, random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)
print(collections.Counter(y_pred))
print(collections.Counter(y_test))

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy}')
print('Confusion Matrix:')
print(conf_matrix)
print('Classification Report:')
print(class_report)


# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['False', 'True'], yticklabels=['False', 'True'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()




### how accurate is a binary classifier built on protein features?

In [ ]:
# Split the data into features (X) and target (y)
X = expression_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Standardize the features if necessary
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)




model = RandomForestClassifier(n_estimators=200, random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)
print(collections.Counter(y_pred))
print(collections.Counter(y_test))

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy}')
print('Confusion Matrix:')
print(conf_matrix)
print('Classification Report:')
print(class_report)


# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['False', 'True'], yticklabels=['False', 'True'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


### how accurate is a binary classifier built on morph features?

In [ ]:
# Split the data into features (X) and target (y)
X = morph_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Standardize the features if necessary
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)




model = RandomForestClassifier(n_estimators=200, random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)
print(collections.Counter(y_pred))
print(collections.Counter(y_test))

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy}')
print('Confusion Matrix:')
print(conf_matrix)
print('Classification Report:')
print(class_report)


# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['False', 'True'], yticklabels=['False', 'True'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


### how accurate is a binary classifier built on nhood features?

In [ ]:
# Split the data into features (X) and target (y)
X = nhood_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Standardize the features if necessary
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)




model = RandomForestClassifier(n_estimators=200, random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)
print(collections.Counter(y_pred))
print(collections.Counter(y_test))

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy}')
print('Confusion Matrix:')
print(conf_matrix)
print('Classification Report:')
print(class_report)


# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['False', 'True'], yticklabels=['False', 'True'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


## splitting analysis by dense and diffuse

### adding dense vs diffuse neighbors to object

In [ ]:
sns.set_style('white')
merged_adata_spec = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter') & (merged_adata_sub.obs.AD_CNT == 'AD')]



data_directory = "sample_adatas_segsurround/"
adata_list = {}


for filename in os.listdir(data_directory):
    if filename.endswith(".h5ad"):
        file_path = os.path.join(data_directory, filename)
        adata = anndata.read_h5ad(file_path)

        file_name = re.sub('.h5ad', '', filename)
        adata.obs['origfile'] = file_name
        #adata.obs['file_leiden'] = [i + j for i, j in zip(adata.obs['origfile'], adata.obs['leiden'])]
        adata_list[file_name] = adata
        print(adata.to_df().columns)

merged_adata_seg = anndata.concat(adata_list, axis=0, index_unique="-", join='outer')
merged_adata_seg.obs.index = merged_adata_seg.obs['id'].to_list()

for i in merged_adata_seg.layers.keys():
    print(i)
    
    merged_adata_spec.obsm['seg_' + i] = merged_adata_seg.to_df(layer=i).loc[merged_adata_spec.obs.index]

merged_adata_spec.obsm['seg_' + 'X'] = merged_adata_seg.to_df().loc[merged_adata_spec.obs.index]
    
    
plt.scatter(merged_adata_spec.to_df()['GFAP'], merged_adata_spec.obsm['seg_X']['Astrocyte'])



### ranking all features by prediction power over dense abeta (via simple correlation)

In [ ]:
# first create a data frame


Abeta_nhood = merged_adata_spec.obsm['seg_zscore']['dense_Abeta']
nhood_frame = merged_adata_spec.to_df(layer = 'zscore')[['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'PCNA', 'MAP2','Neurofilament', 'NeuN', 'GFAP', 'DAPI']]
nhood_frame = nhood_frame.add_prefix('nhood_')
expression_frame = merged_adata_spec.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_frame = merged_adata_spec.obs[['area µm²', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um',
       'bi_um']]





combined_frame = pd.concat([nhood_frame, expression_frame, morph_frame], axis = 1)

cat_dict_keys = combined_frame.columns
cat_dict_vals = pd.Series(['nhood']*len(nhood_frame.columns) + ['prot']*len(expression_frame.columns) + ['morph']*len(morph_frame.columns))
cat_dictionary = dict(zip(cat_dict_keys, cat_dict_vals))




# correlate with abeta nhood

corr_res_pearson = combined_frame.corrwith(Abeta_nhood, method = 'pearson')
corr_res_spearman = combined_frame.corrwith(Abeta_nhood, method = 'spearman')


cor_res_results = [corr_res_pearson, corr_res_spearman]
cor_res_names = ['pearson', 'spearman']


for i,cor in enumerate(cor_res_results):
    feats = cor.index.to_list()
    cats = [cat_dictionary[i] for i in feats]
    corrs = cor.values
    abs_corrs = np.abs(cor.values)
    
    adcorr_df = pd.DataFrame({'feature':feats, 'category':cats, 'correlation with Abeta in nhood':corrs, 'abs_corr':abs_corrs})
    adcorr_df['category'] = adcorr_df['category'].astype('category', )
    adcorr_df = adcorr_df.sort_values(by='abs_corr', ascending = False)
    #adcorr_df['feature'] = adcorr_df['feature'].astype('category')
    plt.figure(figsize=[20,10])
    sns.barplot(data = adcorr_df, y = 'feature', x = 'correlation with Abeta in nhood', hue = 'category', width = 1, dodge=False)
    plt.tight_layout()
    plt.savefig('../figures/ML_exploration/' + cor_res_names[i] + '_feature_correlations_dense_ABETA.svg')
    plt.savefig('../figures/ML_exploration/' + cor_res_names[i] + '_feature_correlations_dense_ABETA.tiff')
# which modality, or combination of madailties, allows for the best prediction power over 
    




### how accurate is a predictor built on all features? (dense abeta)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor



Abeta_nhood = merged_adata_spec.obsm['seg_minmax']['dense_Abeta']
nhood_frame = merged_adata_spec.to_df(layer = 'minmax')[['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'PCNA', 'MAP2','Neurofilament', 'NeuN', 'GFAP', 'DAPI']]
nhood_frame = nhood_frame.add_prefix('nhood_')
expression_frame = merged_adata_spec.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_frame = merged_adata_spec.obs[['area µm²', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um',
       'bi_um']]



combined_frame = pd.concat([nhood_frame, expression_frame, morph_frame], axis = 1)





# Split the data into features (X) and target (y)
X = combined_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Standardize the features if necessary
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)




# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/ALL_features_ML_dense_ABETA.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/ALL_features_ML_dense_ABETA.tiff', dpi = 200)


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/ALL_features_ML_dense_ABETA_averaged.svg', dpi = 200)

plt.savefig('../figures/ML_exploration/ALL_features_ML_dense_ABETA_averaged.tiff', dpi = 200)


### how accurate is a predictor built on only protein features? (dense abeta)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = expression_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/protein_features_ML_dense_ABETA.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/protein_features_ML_dense_ABETA.tiff', dpi = 200)


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/protein_features_ML_dense_ABETA_averaged.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/protein_features_ML_dense_ABETA_averaged.tiff', dpi = 200)


### how accurate is a predictor built on only morph features? (dense abeta)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = morph_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/morph_features_ML_dense_ABETA.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/morph_features_ML_dense_ABETA.tiff', dpi = 200)


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/morph_features_ML_dense_ABETA_averaged.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/morph_features_ML_dense_ABETA_averaged.tiff', dpi = 200)


### how accurate is a predictor built on only neighborhood_features? (dense abeta)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = nhood_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/nhood_features_ML_dense_ABETA.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/nhood_features_ML_dense_ABETA.tiff', dpi = 200)


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/nhood_features_ML_dense_ABETA_averaged.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/nhood_features_ML_dense_ABETA_averaged.tiff', dpi = 200)


### ranking all features by prediction power over diffuse abeta (via simple correlation)

In [ ]:
# first create a data frame


Abeta_nhood = merged_adata_spec.obsm['seg_zscore']['diffuse_Abeta']
nhood_frame = merged_adata_spec.to_df(layer = 'zscore')[['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'PCNA', 'MAP2','Neurofilament', 'NeuN', 'GFAP', 'DAPI']]
nhood_frame = nhood_frame.add_prefix('nhood_')
expression_frame = merged_adata_spec.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_frame = merged_adata_spec.obs[['area µm²', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um',
       'bi_um']]





combined_frame = pd.concat([nhood_frame, expression_frame, morph_frame], axis = 1)

cat_dict_keys = combined_frame.columns
cat_dict_vals = pd.Series(['nhood']*len(nhood_frame.columns) + ['prot']*len(expression_frame.columns) + ['morph']*len(morph_frame.columns))
cat_dictionary = dict(zip(cat_dict_keys, cat_dict_vals))




# correlate with abeta nhood

corr_res_pearson = combined_frame.corrwith(Abeta_nhood, method = 'pearson')
corr_res_spearman = combined_frame.corrwith(Abeta_nhood, method = 'spearman')


cor_res_results = [corr_res_pearson, corr_res_spearman]
cor_res_names = ['pearson', 'spearman']

for i,cor in enumerate(cor_res_results):
    feats = cor.index.to_list()
    cats = [cat_dictionary[i] for i in feats]
    corrs = cor.values
    abs_corrs = np.abs(cor.values)
    
    adcorr_df = pd.DataFrame({'feature':feats, 'category':cats, 'correlation with Abeta in nhood':corrs, 'abs_corr':abs_corrs})
    adcorr_df['category'] = adcorr_df['category'].astype('category', )
    adcorr_df = adcorr_df.sort_values(by='abs_corr', ascending = False)
    #adcorr_df['feature'] = adcorr_df['feature'].astype('category')
    plt.figure(figsize=[20,10])
    sns.barplot(data = adcorr_df, y = 'feature', x = 'correlation with Abeta in nhood', hue = 'category', width = 1, dodge=False)
    plt.tight_layout()
    plt.savefig('../figures/ML_exploration/' + cor_res_names[i] + '_feature_correlations_diffuse_ABETA.svg')
    plt.savefig('../figures/ML_exploration/' + cor_res_names[i] + '_feature_correlations_diffuse_ABETA.tiff')
    
# which modality, or combination of madailties, allows for the best prediction power over 
    




### how accurate is a predictor built on all features? (diffuse abeta)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor



Abeta_nhood = merged_adata_spec.obsm['seg_minmax']['diffuse_Abeta']
nhood_frame = merged_adata_spec.to_df(layer = 'minmax')[['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'PCNA', 'MAP2','Neurofilament', 'NeuN', 'GFAP', 'DAPI']]
nhood_frame = nhood_frame.add_prefix('nhood_')
expression_frame = merged_adata_spec.obs[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]
morph_frame = merged_adata_spec.obs[['area µm²', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um',
       'bi_um']]



combined_frame = pd.concat([nhood_frame, expression_frame, morph_frame], axis = 1)





# Split the data into features (X) and target (y)
X = combined_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#Standardize the features if necessary
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)




# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/ALL_features_ML_diffuse_ABETA.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/ALL_features_ML_diffuse_ABETA.tiff', dpi = 200)


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/ALL_features_ML_diffuse_ABETA_averaged.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/ALL_features_ML_diffuse_ABETA_averaged.tiff', dpi = 200)


### how accurate is a predictor built on only protein features? (diffuse abeta)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = expression_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/protein_features_ML_diffuse_ABETA.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/protein_features_ML_diffuse_ABETA.tiff', dpi = 200)


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/protein_features_ML_diffuse_ABETA_averaged.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/protein_features_ML_diffuse_ABETA_averaged.tiff', dpi = 200)


### how accurate is a predictor built on only morph features? (diffuse abeta)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = morph_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/morph_features_ML_diffuse_ABETA.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/morph_features_ML_diffuse_ABETA.tiff', dpi = 200)


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/morph_features_ML_diffuse_ABETA_averaged.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/morph_features_ML_diffuse_ABETA_averaged.tiff', dpi = 200)


### how accurate is a predictor built on only neighborhood_features? (diffuse abeta)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression #*
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor #*
from sklearn.ensemble import GradientBoostingRegressor #*
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor


# Split the data into features (X) and target (y)
X = nhood_frame
y = Abeta_nhood

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features if necessary
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize the model
model = RandomForestRegressor(n_estimators = 200,random_state=42)

# Train the model
model.fit(X_train, y_train)



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'R^2: {r2}')


abeta_df = pd.DataFrame({'prediction':y_pred, 'actual':y_test})
abeta_df = abeta_df.sort_values(by='actual', ascending = False)
abeta_df['id'] = range(len(abeta_df['actual']))[::-1]


plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_df['prediction'], color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/nhood_features_ML_diffuse_ABETA.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/nhood_features_ML_diffuse_ABETA.tiff', dpi = 200)


def average_nearest_points(points, window_size=10):
    if window_size > len(points):
        raise ValueError("Window size must be less than or equal to the number of points")
    
    new_points = []
    half_window = window_size // 2
    
    for i in range(len(points)):
        # Determine the range of indices to average
        start_index = max(0, i - half_window)
        end_index = min(len(points), i + half_window + 1)
        
        # Handle the case where there are less than 10 points at the start or end
        if end_index - start_index < window_size:
            if start_index == 0:
                end_index = min(len(points), window_size)
            elif end_index == len(points):
                start_index = max(0, len(points) - window_size)
        
        # Compute the average of the nearest points
        nearest_points = points[start_index:end_index]
        average_point = sum(nearest_points) / len(nearest_points)
        
        new_points.append(average_point)
    
    return new_points

abeta_prediction_smooth = average_nearest_points(abeta_df['prediction'])

plt.figure(figsize=(10, 6))

# Scatter plots
plt.scatter(abeta_df['id'], abeta_df['actual'], color='blue', label='actual', rasterized = True)
plt.scatter(abeta_df['id'], abeta_prediction_smooth, color='green', label='prediction', rasterized = True)

# Line plots
# plt.plot(abeta_df['id'], abeta_df['actual'], color='blue')
# plt.plot(abeta_df['id'], abeta_df['prediction'], color='green')

# Labels and title
plt.xlabel('')
plt.ylabel('normalized Abeta')
plt.legend()
plt.savefig('../figures/ML_exploration/nhood_features_ML_diffuse_ABETA_averaged.svg', dpi = 200)
plt.savefig('../figures/ML_exploration/nhood_features_ML_diffuse_ABETA_averaged.tiff', dpi = 200)


# MORPH X PROTEIN CONNECTION (only RAM2)

In [ ]:

# load in morph data 


import copy




## load adatas

In [ ]:
data_directory = "sample_adatas/"
adata_list = {}


for filename in os.listdir(data_directory):
    if filename.endswith(".h5ad"):
        file_path = os.path.join(data_directory, filename)
        adata = anndata.read_h5ad(file_path)

        file_name = re.sub('.h5ad', '', filename)
        adata.obs['origfile'] = file_name
        #adata.obs['file_leiden'] = [i + j for i, j in zip(adata.obs['origfile'], adata.obs['leiden'])]
        adata_list[file_name] = adata




### fix mistaken names

In [ ]:

    
feature_dict = {
    'Claudin-5':'Claudin5',
    'a-Beta':'A-Beta',
    'DAPI_1':'DAPI'
}
features_to_remove = ['H2A.x', 'SMA - Claudin5 +', 'GFAP + Vimentin +']


# Define the custom order for feature names
custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV','PCNA', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'GFAP', 'DAPI']


    
for adata_name in adata_list.keys():
    adata = adata_list[adata_name]
    
    adata_vars = adata.var_names
    
    for var in adata_vars:
        if var in feature_dict.keys():
            adata.var_names = [feature_dict[name] if name in feature_dict.keys() else name for name in adata.var_names] 



    
    adata.var.drop(features_to_remove, inplace=False)







In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


merged_adata = anndata.concat(adata_list, axis=0, index_unique="-")

#custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'PCNA', 'GFAP', 'DAPI_1']
# Use NumPy's argsort to get the indices that would reorder the features
order_indices = [merged_adata.var_names.get_loc(feature) for feature in custom_feature_order]

# Reorder the features based on the custom order
merged_adata = merged_adata[:, order_indices]

merged_adata.obs['origfilenum'] = merged_adata.obs['origfile'].str.split('_', expand = True)[0]
#merged_adata.obs['AD_CNT'] = merged_adata.obs['origfile'].str.split('_', expand = True)[1]
merged_adata.obs['AD_CNT'] = [ad_cnt_dict[i] for i in merged_adata.obs['origfilenum']]



#sc.pl.violin(merged_adata_GM, 'CollagenIV', groupby='leiden', show=True)


## attach clustering metadata

In [ ]:

layer = 'zscore'


# alter the following below to adapt to new methods of clustering
clustering_metadata = anndata.read_h5ad('../final_h5ads/clustering_and_morph_clustering.h5')

merged_adata = merged_adata[merged_adata.obs.id.isin(clustering_metadata.obs.ImageID_CellID)]





cluster_label_order = ['Rounded', 'Intermediate', 'Ramified 1', 'Ramified 2', 'Ramified 3']
cluster_label_ticklabels = cluster_label_order

change_cluster_names_dict = {
    '1': 'Rounded',
    '2': 'Intermediate',
    '0_ram':'Ramified 2',
    '1_ram':'Ramified 3',
    '2_ram':'Ramified 1'
}

cMapDict = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs.final_leiden_M1]
clustering_metadata.obs['leiden'] = pd.Categorical(clustering_metadata.obs['leiden'], categories=cluster_label_order, ordered=False)






# also update the protein clusters



change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}




cluster_label_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2', 'default']
cluster_label_ticklabels = cluster_label_order

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['protein_leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs['protein_leiden']]

clustering_metadata.obs['protein_leiden'] = pd.Categorical(clustering_metadata.obs['protein_leiden'], categories=cluster_label_order, ordered=False)



#--------------------------------------------------------------------------------------




mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs['leiden']))
merged_adata.obs['leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['leiden']

mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs['protein_leiden']))
merged_adata.obs['protein_leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_protein_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['protein_leiden']


# also map distance metadata for later use 

distance_metrics = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta',
       'fabseg_distance_to_nearest_Astrocytes',
       'fabseg_distance_to_nearest_Neurons',
       'fabseg_distance_to_nearest_Oligodendrocytes',
       'fabseg_distance_to_nearest_dense_aBeta',
       'fabseg_distance_to_nearest_diffuse_aBeta',
        'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']

for met in distance_metrics:
    mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs[met]))
    merged_adata.obs[met] = [mapping_dict[i] for i in merged_adata.obs.id]
#merged_adata.obs['file_protein_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['protein_leiden']



# remove any meningeal macrophages

merged_adata = merged_adata[merged_adata.obs['parent'].isin(['Grey Matter', 'White Matter'])]

# remove any default clusters

merged_adata = merged_adata[merged_adata.obs['leiden'] != 'default']

merged_adata_WM = merged_adata[merged_adata.obs['parent'] == 'White Matter']
merged_adata_GM = merged_adata[merged_adata.obs['parent'] == 'Grey Matter']






merged_adata_sub = merged_adata[merged_adata.obs['id'].isin(clustering_metadata.obs['ImageID_CellID'])]
clustering_metadata.obs.index = clustering_metadata.obs['ImageID_CellID']

clustering_metadata = clustering_metadata[merged_adata_sub.obs['id'], :]

tsne_coordinates = clustering_metadata.obsm["X_tsne"]
merged_adata_sub.obsm['X_tsne'] = tsne_coordinates
protein_tsne_coordinates = clustering_metadata.obsm['X_tsne_PROTEIN']
merged_adata_sub.obsm['X_tsne_PROTEIN'] = protein_tsne_coordinates

#merged_adata.obsm.X_tsne = 

for feat in merged_adata_sub.var_names:
    df = merged_adata_sub.to_df()
    #feat_upperbound = np.percentile(a=df[feat], q=[80])

    sc.pl.tsne(merged_adata_sub, color=feat, layer=layer)
    sc.pl.scatter(merged_adata_sub, color = feat, basis = 'tsne_PROTEIN', layers = layer)
    sc.pl.violin(merged_adata_sub, feat, groupby = 'leiden', layer=layer)
    sc.pl.violin(merged_adata_sub, feat, groupby = 'protein_leiden', layer = layer)
    
    



## add protein and morph data to object 

In [ ]:
# add morph data

merged_adata_sub.obs.index = merged_adata_sub.obs.id
morph_data_raw = clustering_metadata.to_df(layer='unscaled_morph_data').loc[merged_adata_sub.obs.id]


protein_data_orig = sc.read_h5ad('../final_h5ads/fabian_metadata_plus_clustering.h5ad')

protein_data_orig_sub = protein_data_orig[protein_data_orig.obs.ImageID_CellID.isin(merged_adata_sub.obs.id)]
protein_data_raw = protein_data_orig_sub.to_df()[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']].loc[merged_adata_sub.obs.index]

merged_adata_sub.obs = pd.concat([merged_adata_sub.obs, morph_data_raw, protein_data_raw], axis = 1)


#matrix1 = protein_data_orig_sub.to_df()[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]


#merged_adata_sub.X = pd.concat([merged_adata_sub.to_df(), clustering_metadata.to_df().loc[merged_adata_sub.obs.id]], axis=0)

sc.pl.scatter(merged_adata_sub, color = 'CD163', basis = 'tsne_PROTEIN')


morph_features = list(morph_data_raw.columns)
protein_features = list(protein_data_raw.columns)

In [ ]:

sc.pl.matrixplot(merged_adata_sub, groupby = 'protein_leiden', var_names=['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45'], standard_scale='var')

In [ ]:

for cluster in clustering_metadata.obs.leiden.unique():
    sub_obj = clustering_metadata[(clustering_metadata.obs.leiden == cluster) & (clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')]
    sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='leiden')
    
sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne_PROTEIN', color='leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne_PROTEIN', color='protein_leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne', color='leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne', color='protein_leiden', size=20)


In [ ]:
cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}


# among ramified 2 cells, which are associated with Abeta, and what protein clusters do they belong to?

Abeta_pct_cutoff = 0.5
layer = 'zscore'

# Abeta_pct_cutoff = 0.1
# layer = None
from sklearn.cluster import DBSCAN

merged_adata_sub.obs['ABETA_percent'] = merged_adata_sub.to_df(layer=layer)['A-Beta']
merged_adata_sub.obs['beats_abeta_thresh'] = merged_adata_sub.obs.ABETA_percent > Abeta_pct_cutoff

sub_obj = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter') & (merged_adata_sub.obs.AD_CNT == 'AD') & (merged_adata_sub.obs.leiden == 'Ramified 2')]



plt.hist(sub_obj.obs['ABETA_percent'])
plt.show()
sns.kdeplot(sub_obj.obs['ABETA_percent'])
plt.show()

print('what percent of mircroglia in GM AD are nearby Abeta?')
print(sum(sub_obj.obs.ABETA_percent > Abeta_pct_cutoff)/len(sub_obj.obs.ABETA_percent))

#sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='beats_abeta_thresh', legend_loc='on data')


sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='leiden', show=False, color_map=cMapDict_morph)


plt.savefig('../figures/neighborhood_exploration/morph_clusters_on_protein_tsne.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/morph_clusters_on_protein_tsne.svg', format='svg')


for var in sub_obj.var_names:
    sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color=var, show = False, layers='zscore')
    
    plt.savefig('../figures/neighborhood_exploration/' + var + '_scaled_featureplot.tiff', format='tiff')
    plt.savefig('../figures/neighborhood_exploration/' + var + '_scaled_featureplot.svg', format='svg')


coordinates = sub_obj.obsm['X_tsne_PROTEIN']
coordsx = [i[0] for i in coordinates]
coordsy = [i[1] for i in coordinates]

coordinates_dframe = pd.DataFrame({'x':coordsx, 'y':coordsy})
#hdb = DBSCAN(min_samples=20, eps = 3)
hdb = DBSCAN(min_samples=80, eps=3.6)
hdb.fit(coordinates_dframe)

sub_obj.obs['dblabels'] = hdb.labels_
sub_obj.obs['dblabels'] = pd.Categorical(sub_obj.obs['dblabels'])


sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='dblabels', legend_loc='on data')








# coordinates = only_nearby_abeta.obsm['X_tsne_PROTEIN']
# coordsx = [i[0] for i in coordinates]
# coordsy = [i[1] for i in coordinates]

# coordinates_dframe = pd.DataFrame({'x':coordsx, 'y':coordsy})
# #hdb = DBSCAN(min_samples=20, eps = 3)
# hdb = DBSCAN(min_samples=20, eps = 3.5)
# hdb.fit(coordinates_dframe)

# only_nearby_abeta.obs['dblabels'] = hdb.labels_
# only_nearby_abeta.obs['dblabels'] = pd.Categorical(only_nearby_abeta.obs['dblabels'])


# sc.pl.scatter(only_nearby_abeta, basis='tsne_PROTEIN', color='dblabels', legend_loc='on data')





### cluster neighbor data

In [ ]:


cmapDict_spatial  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
        "16": "#f7b6d2"
    }




from scipy.stats import zscore
import umap
import umap.plot

orig_df = sub_obj.to_df()
zscore_within_samples = sub_obj.to_df().groupby(sub_obj.obs['origfilenum']).transform(func = zscore, axis = 0)

zscore_all = zscore_within_samples.transform(func = zscore, axis = 0)


# perform umap dimreduction
embedding = umap.UMAP().fit(zscore_all)
print(umap.plot.points(embedding))

sub_obj.obsm['X_neighbor_umap'] = coordinates = embedding.embedding_
sub_obj.layers['neighbor_zscore_new'] = zscore_all
sub_obj.obsm['zscore_all'] = zscore_all
plt.show()


for var in sub_obj.var_names:
    

    sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, layers='zscore', show = False)
    plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '.tiff', format='tiff')
    plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '.svg', format='svg')
    
    sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, show = False)
    # plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '_unscaled.tiff', format='tiff')
    # plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '_unscaled.svg', format='svg')
    
    #sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, layers='neighbor_zscore_new')

    
    
sc.pp.neighbors(sub_obj, random_state=10, use_rep='zscore_all')

sc.tl.leiden(sub_obj, key_added='spatial_clustering')
sub_obj.obs['spatial_clustering'] = sub_obj.obs['spatial_clustering'].astype('category')

sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = 'spatial_clustering', legend_loc='on data', show = False, palette=[cmapDict_spatial[i] for i in list(sub_obj.obs.spatial_clustering.cat.categories)])
#sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = 'spatial_clustering', legend_loc='on data', show = False)

plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap.svg', format='svg')


sc.pl.scatter(sub_obj, basis = 'tsne_PROTEIN', color = 'spatial_clustering', show = False)
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_protein.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_protein.svg', format='svg')





# make custom plot:



sc.pl.scatter(sub_obj, basis = 'tsne', color = 'spatial_clustering', show = False)
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_morph.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_morph.svg', format='svg')







# for clus in sub_obj.obs.spatial_clustering.unique():
#     sub_obj_clus = sub_obj[sub_obj.obs.spatial_clustering == clus]
#     sc.pl.scatter(sub_obj_clus, basis = 'tsne_PROTEIN', color = 'spatial_clustering', legend_loc='on data', show=False)
#     plt.axis([-100, 100, -100, 100])
#     plt.show()

# hm = sc.pl.matrixplot(sub_obj, groupby = 'spatial_clustering', layer='zscore', var_names=sub_obj.var_names, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.svg', format='svg')

# hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=protein_features, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.svg', format='svg')

# hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=morph_features, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.svg', format='svg')






miniclus = copy.deepcopy(sub_obj)


In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
sns.set(font="Arial")

sns.set_style('white')




sub_obj_2 = sub_obj.copy()

colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)


neighbor_percents = sub_obj_2.to_df()

upperlims = []

for i,n in enumerate(neighbor_percents.columns):
    nvals = neighbor_percents[n]
    upperlim = np.percentile(nvals, 99)
    
    upperlims.append(upperlim)

sub_obj_2.obsm['X_umap'] = sub_obj_2.obsm['X_neighbor_umap']
sc.pl.umap(sub_obj_2, color=sub_obj_2.var_names, vmax=upperlims, show=False, color_map=cmap)

plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_onlyRAM2_bluewhite_noscaling_EASYEDIT.tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_onlyRAM2_bluewhite_noscaling_EASYEDIT.svg', format = 'svg')

In [ ]:
colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

xvals = [i[0] for i in sub_obj.obsm['X_neighbor_umap']]
yvals = [i[1] for i in sub_obj.obsm['X_neighbor_umap']]
neighbor_percents = sub_obj.to_df()

fig, axs = plt.subplots(3, 4, figsize = (21,10))


for i,n in enumerate(neighbor_percents.columns):
    nvals = neighbor_percents[n]
    upperlim = np.percentile(nvals, 99)
    
    xpos = i//4
    ypos = i%4

    im = axs[xpos,ypos].scatter(x = xvals, y = yvals, c=neighbor_percents[n], cmap=cmap, s=1, vmax=upperlim)
    plt.colorbar(im, ax = axs[xpos,ypos])
    axs[xpos,ypos].set_title(n)
    axs[xpos,ypos].set_xticks([])
    axs[xpos,ypos].set_yticks([])
    axs[xpos,ypos].set_xlabel('UMAP1')
    axs[xpos,ypos].set_ylabel('UMAP2')


    # axs[0,i].xlabel('tSNE1')
    # plt.ylabel('tSNE2')
    # plt.title(n)
    # plt.show()
plt.tight_layout()
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_onlyRAM2_bluewhite_noscaling.tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_onlyRAM2_bluewhite_noscaling.svg', format = 'svg')
plt.show()

# sc.pl.tsne(clustering_adata, color=adataScaled.var_names,use_raw=False, return_fig = True, color_map=cmap, show = False)
# plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.tiff')
# plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.svg', format = 'svg')
# sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = sub_obj.var_names)


In [ ]:

hm = sc.pl.matrixplot(sub_obj, groupby = 'spatial_clustering', layer='zscore', var_names=sub_obj.var_names, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_onlyRAM2.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_onlyRAM2.svg', format='svg')

hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=protein_features, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_onlyRAM2.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_onlyRAM2.svg', format='svg')

hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=morph_features, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_onlyRAM2.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_onlyRAM2.svg', format='svg')

#KILLME




        
# test = custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING.tiff')
# fjdlf

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_onlyRAM2_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_onlyRAM2_ALT_SCALING.svg')

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_onlyRAM2_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_onlyRAM2_ALT_SCALING.svg')

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_onlyRAM2_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_onlyRAM2_ALT_SCALING.svg')









In [ ]:
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd
from scipy.stats import f_oneway


import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    fig = plt.figure(figsize=(10, 5))  # width=10 inches, height=5 inches

    ax = plt.axes()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova

    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]

    for c in combinations:
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])
    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    
    plt.show()
    
    
    
cmapDict_spatial  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
        "16": "#f7b6d2"
    }


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


proteins_to_profile = protein_features

sub_obj.obs['file_spatial_clustering'] = sub_obj.obs['origfilenum'].astype(str) + '_' +  sub_obj.obs['spatial_clustering'].astype(str)

protdata = sub_obj.obs[proteins_to_profile]
protdata['sample_spatial_clustering'] = sub_obj.obs['file_spatial_clustering']
protdata_mean = protdata.groupby(by=['sample_spatial_clustering']).mean()
protdata_mean['sample_spatial_clustering'] = protdata_mean.index
protdata_mean['spatial_clustering'] = protdata_mean['sample_spatial_clustering'].str.split('_', expand = True)[1]
protdata_mean['sample_num'] = protdata_mean['sample_spatial_clustering'].str.split('_', expand = True)[0]
protdata_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in protdata_mean['sample_num']]

cluster_label_order = [str(i) for i in range(12)]
cluster_label_ticklabels = cluster_label_order



minmax_df_mean = protdata_mean
for protein in ['CD163']:
    print(protein)
    
    data = []
    title = protein + 'Gray Matter'
    ylabel = 'average protein expression'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['spatial_clustering'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_spatial_clustering.to_list()
        data_anova_df_samplecol += sub_df['sample_num'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = [str(i) for i in range(17)], ordered = True)

    lel = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cmapDict_spatial.values())  











In [ ]:
sc.pl.violin(sub_obj, 'CD163', 'spatial_clustering')
sc.pl.violin(sub_obj, 'TMEM119', 'spatial_clustering')
sc.pl.violin(sub_obj, 'HLA-DR', 'spatial_clustering')
sc.pl.violin(sub_obj, 'CD11c', 'spatial_clustering')

In [ ]:
sc.pl.violin(sub_obj, 'CD163', 'protein_leiden')
sc.pl.violin(sub_obj, 'TMEM119', 'protein_leiden')
sc.pl.violin(sub_obj, 'HLA-DR', 'protein_leiden')
sc.pl.violin(sub_obj, 'CD11c', 'protein_leiden')

In [ ]:

for group in set(sub_obj.obs.protein_leiden):
    
    if group == 'BLM':
        continue
    
    import numpy as np
    from scipy.stats import ttest_ind

    distribution1 = sub_obj[sub_obj.obs.protein_leiden == 'BLM'].obs['CD163']
    distribution2 = sub_obj[sub_obj.obs.protein_leiden == group].obs['CD163']

    # Perform t-test
    print('---------------------------------')

    print(group)
    t_statistic, p_value = ttest_ind(distribution1, distribution2)
    print(f"T-statistic: {t_statistic}")
    print(f"P-value: {p_value}")











In [ ]:
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd
from scipy.stats import f_oneway


import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    fig = plt.figure(figsize=(7, 5))  # width=10 inches, height=5 inches

    ax = plt.axes()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova

    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]

    for c in combinations:
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])
    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    
    plt.show()
    
    


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e'
}







proteins_to_profile = protein_features

sub_obj.obs['file_protein_leiden'] = sub_obj.obs['origfilenum'].astype(str) + '_' +  sub_obj.obs['protein_leiden'].astype(str)
protdata = sub_obj.obs[proteins_to_profile]
protdata['sample_protein_leiden'] = sub_obj.obs['file_protein_leiden']
protdata_mean = protdata.groupby(by=['sample_protein_leiden']).mean()
protdata_mean['sample_protein_leiden'] = protdata_mean.index
protdata_mean['protein_leiden'] = protdata_mean['sample_protein_leiden'].str.split('_', expand = True)[1]
protdata_mean['sample_num'] = protdata_mean['sample_protein_leiden'].str.split('_', expand = True)[0]
protdata_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in protdata_mean['sample_num']]

cluster_label_order = ['PVM', 'BLM', 'Microglia 1', 'Microglia 2']
cluster_label_ticklabels = cluster_label_order



minmax_df_mean = protdata_mean
for protein in ['CD163']:
    print(protein)
    
    data = []
    title = protein + 'Gray Matter'
    ylabel = 'average protein expression'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['protein_leiden'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_protein_leiden.to_list()
        data_anova_df_samplecol += sub_df['sample_num'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_protein_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_protein_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = cluster_label_order, ordered = True)
    lel = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values())  







In [ ]:
from scipy import stats
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from scipy.stats import tukey_hsd
from scipy.stats import f_oneway


import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice = sns.color_palette('pastel'), savepath = False, saveformat = 'tiff'):
    """
    Create a box-and-whisker plot with significance bars.
    """
    
    
    list_of_lists = data

    # Create a dictionary with 'category' and 'values' keys
    result_dict = {'category': [], 'values': []}
    
    for sublist_i in range(len(list_of_lists)):
        
        result_dict['category'].extend([sublist_i]*len(list_of_lists[sublist_i]))
        result_dict['values'].extend(list_of_lists[sublist_i])


    result_dict = pd.DataFrame(result_dict)
    
    
    
    
    fig = plt.figure(figsize=(10, 5))  # width=10 inches, height=5 inches

    ax = plt.axes()
    bp = ax.boxplot(data, widths=0.6, patch_artist=True)
    ax.scatter(x = result_dict['category'] + 1, y=result_dict['values'], s=10, zorder = 2, c = 'black')
    # Graph title
    ax.set_title(title, fontsize=14)
    # Label y-axis
    ax.set_ylabel(ylabel)
    # Label x-axis ticks
    ax.set_xticklabels(xticklabels)
    # Hide x-axis major ticks
    ax.tick_params(axis='x', which='major', length=0)
    # Show x-axis minor ticks
    xticks = [0.5] + [x + 0.5 for x in ax.get_xticks()]
    ax.set_xticks(xticks, minor=True)
    # Clean up the appearance
    ax.tick_params(axis='x', which='minor', length=3, width=1)

    # Change the colour of the boxes to Seaborn's 'pastel' palette
    colors = colorschoice
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    # Colour of the median lines
    plt.setp(bp['medians'], color='k')
    
    # pivot data
    
    anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    
    #impute missing values as mean of other samples
    for col in anova_data_pivoted.columns:
        if any(pd.isna(anova_data_pivoted[col])):
            anova_data_pivoted[col][pd.isna(anova_data_pivoted[col])] = np.mean(anova_data_pivoted[col])

    # perform anova

    anova_f_statistic, anova_p_value = f_oneway(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])

    # perform tukey post-hoc
    
    res = tukey_hsd(*[anova_data_pivoted[i] for i in anova_data_pivoted.columns.to_list()])
    print(res)
    
    # Check for statistical significance
    significant_combinations = []
    # Check from the outside pairs of boxes inwards
    ls = list(range(0, len(data)))
    combinations = [(ls[x], ls[x + y]) for y in reversed(ls) for x in range((len(ls) - y))]

    for c in combinations:
        stat = res.statistic[c[0],c[1]]
        p = res.pvalue[c[0],c[1]]
        
        
        if p < 0.05:
            significant_combinations.append([[c[0] + 1, c[1] + 1], p])
    # Get info about y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom

    
    
    
    
    for i, significant_combination in enumerate(significant_combinations):
        # Columns corresponding to the datasets of interest
        x1 = significant_combination[0][0]
        x2 = significant_combination[0][1]
        # What level is this bar among the bars above the plot?
        level = len(significant_combinations) - i
        # Plot the bar
        bar_height = (yrange * 0.08 * level) + top
        bar_tips = bar_height - (yrange * 0.02)
        plt.plot(
            [x1, x1, x2, x2],
            [bar_tips, bar_height, bar_height, bar_tips], lw=1, c='k')
        # Significance level
        p = significant_combination[1]
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
        text_height = bar_height + (yrange * 0.01)
        plt.text((x1 + x2) * 0.5, text_height, sig_symbol, ha='center', c='k')
    
    # Adjust y-axis
    bottom, top = ax.get_ylim()
    yrange = top - bottom
    ax.set_ylim(bottom - 0.02 * yrange, top)

    # Annotate sample size below each box
    for i, dataset in enumerate(data):
        sample_size = len(dataset)
        ax.text(i + 1, bottom, fr'n = {sample_size}', ha='center', size='x-small')

    if savepath:
        plt.savefig(savepath, format=saveformat)
    
    plt.show()
    
    
    
cmapDict_spatial  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
        "16": "#f7b6d2"
    }


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


proteins_to_profile = protein_features

sub_obj.obs['file_spatial_clustering'] = sub_obj.obs['origfilenum'].astype(str) + '_' +  sub_obj.obs['spatial_clustering'].astype(str)

protdata = sub_obj.obs[proteins_to_profile]
protdata['sample_spatial_clustering'] = sub_obj.obs['file_spatial_clustering']
protdata_mean = protdata.groupby(by=['sample_spatial_clustering']).mean()
protdata_mean['sample_spatial_clustering'] = protdata_mean.index
protdata_mean['spatial_clustering'] = protdata_mean['sample_spatial_clustering'].str.split('_', expand = True)[1]
protdata_mean['sample_num'] = protdata_mean['sample_spatial_clustering'].str.split('_', expand = True)[0]
protdata_mean['AD_CNT'] = [ad_cnt_dict[str(i)] for i in protdata_mean['sample_num']]

cluster_label_order = [str(i) for i in range(12)]
cluster_label_ticklabels = cluster_label_order



minmax_df_mean = protdata_mean
for protein in ['CD163']:
    print(protein)
    
    data = []
    title = protein + 'Gray Matter'
    ylabel = 'average protein expression'
    
    data_anova_df_groupcol = []
    data_anova_df_valuecol = []
    data_anova_df_namecol = []
    data_anova_df_samplecol = []
    
    xticklabels = cluster_label_ticklabels

    
    for cluster in cluster_label_order:

    
        sub_df = minmax_df_mean[minmax_df_mean['spatial_clustering'] == cluster]
        
        
        data.append(sub_df[protein])
        
        data_anova_df_groupcol += [cluster] * sub_df.shape[0]
        data_anova_df_valuecol += sub_df[protein].to_list()
        data_anova_df_namecol += sub_df.sample_spatial_clustering.to_list()
        data_anova_df_samplecol += sub_df['sample_num'].to_list()
        
    anova_data = df = pd.DataFrame({
    'name': data_anova_df_namecol,
    'group': data_anova_df_groupcol,
    'value': data_anova_df_valuecol,
    'sample': data_anova_df_samplecol
        
    
})

    #anova_data_pivoted = anova_data.pivot(index = 'sample', columns = 'group', values = 'value')
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.tiff'))  
    # box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cMapDict.values(), savepath= ('../figures/protein/' + title + '_leiden_box&whisker_ANOVA.svg'), saveformat='svg')  
    anova_data['group'] = pd.Categorical(anova_data['group'], categories = [str(i) for i in range(17)], ordered = True)

    lel = box_and_whisker_anova(data, anova_data, title, ylabel, xticklabels, colorschoice=cmapDict_spatial.values())  




In [ ]:

ax = sc.pl.scatter(sub_obj, color=["spatial_clustering"], groups=["5", "2", "8"], show=False, basis = 'tsne_PROTEIN')

# We can change the 'NA' in the legend that represents all cells outside of the
# specified groups
legend_texts = ax.get_legend().get_texts()
# Find legend object whose text is "NA" and change it
for legend_text in legend_texts:
    if legend_text.get_text() == "NA":
        legend_text.set_text("other cell types")
        
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_only_3clusters_onlyRAM2.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_only_3clusters_onlyRAM2.svg', format='svg')


In [ ]:
cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': miniclus.obs['spatial_clustering'],
    'Category2': miniclus.obs['leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)
normalized_data = normalized_data.loc[normalized_data.index[::-1]]

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. sub-clusters')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.savefig('../figures/neighborhood_exploration/nhood_clusters_morph_barchart_onlyRAM2.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_morph_barchart_onlyRAM2.svg', format='svg')






In [ ]:
cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': miniclus.obs['spatial_clustering'],
    'Category2': miniclus.obs['protein_leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)
normalized_data = normalized_data.loc[normalized_data.index[::-1]]

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Protein vs. sub-clusters')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')


plt.savefig('../figures/neighborhood_exploration/nhood_clusters_protein_barchart_onlyRAM2.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_protein_barchart_onlyRAM2.svg', format='svg')

#### save object with neighborhood clusters

In [ ]:
#sub_obj.write_h5ad('sub_obj_with_spatial_clustering.h5ad')

# MORPH X PROTEIN CONNECTION (ALL)

In [ ]:

# load in morph data 


import copy




## load adatas

In [ ]:
data_directory = "sample_adatas/"
adata_list = {}


for filename in os.listdir(data_directory):
    if filename.endswith(".h5ad"):
        file_path = os.path.join(data_directory, filename)
        adata = anndata.read_h5ad(file_path)

        file_name = re.sub('.h5ad', '', filename)
        adata.obs['origfile'] = file_name
        #adata.obs['file_leiden'] = [i + j for i, j in zip(adata.obs['origfile'], adata.obs['leiden'])]
        adata_list[file_name] = adata




### fix mistaken names

In [ ]:

    
feature_dict = {
    'Claudin-5':'Claudin5',
    'a-Beta':'A-Beta',
    'DAPI_1':'DAPI'
}
features_to_remove = ['H2A.x', 'SMA - Claudin5 +', 'GFAP + Vimentin +']


# Define the custom order for feature names
custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV','PCNA', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'GFAP', 'DAPI']


    
for adata_name in adata_list.keys():
    adata = adata_list[adata_name]
    
    adata_vars = adata.var_names
    
    for var in adata_vars:
        if var in feature_dict.keys():
            adata.var_names = [feature_dict[name] if name in feature_dict.keys() else name for name in adata.var_names] 



    
    adata.var.drop(features_to_remove, inplace=False)







In [ ]:
ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}


merged_adata = anndata.concat(adata_list, axis=0, index_unique="-")

#custom_feature_order = ['Claudin5', 'SMA', 'Vimentin', 'CollagenIV', 'MAP2', 'Neurofilament', 'NeuN', 'A-Beta', 'ApoE', 'PCNA', 'GFAP', 'DAPI_1']
# Use NumPy's argsort to get the indices that would reorder the features
order_indices = [merged_adata.var_names.get_loc(feature) for feature in custom_feature_order]

# Reorder the features based on the custom order
merged_adata = merged_adata[:, order_indices]

merged_adata.obs['origfilenum'] = merged_adata.obs['origfile'].str.split('_', expand = True)[0]
#merged_adata.obs['AD_CNT'] = merged_adata.obs['origfile'].str.split('_', expand = True)[1]
merged_adata.obs['AD_CNT'] = [ad_cnt_dict[i] for i in merged_adata.obs['origfilenum']]



#sc.pl.violin(merged_adata_GM, 'CollagenIV', groupby='leiden', show=True)


## attach clustering metadata

In [ ]:

layer = 'zscore'


# alter the following below to adapt to new methods of clustering
clustering_metadata = anndata.read_h5ad('../final_h5ads/clustering_and_morph_clustering.h5')

merged_adata = merged_adata[merged_adata.obs.id.isin(clustering_metadata.obs.ImageID_CellID)]





cluster_label_order = ['Rounded', 'Intermediate', 'Ramified 1', 'Ramified 2', 'Ramified 3']
cluster_label_ticklabels = cluster_label_order

change_cluster_names_dict = {
    '1': 'Rounded',
    '2': 'Intermediate',
    '0_ram':'Ramified 2',
    '1_ram':'Ramified 3',
    '2_ram':'Ramified 1'
}

cMapDict = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs.final_leiden_M1]
clustering_metadata.obs['leiden'] = pd.Categorical(clustering_metadata.obs['leiden'], categories=cluster_label_order, ordered=False)






# also update the protein clusters



change_cluster_names_dict = {'MG_0': 'Microglia 1',
         'MG_1': 'Microglia 2',
         'MO_MAC_0': 'Monocytes',
         'MO_MAC_1': 'PVM',
         'MO_MAC_2': 'BLM',
         'default': 'default'
}

cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}




cluster_label_order = ['Monocytes', 'PVM', 'BLM', 'Microglia 1', 'Microglia 2', 'default']
cluster_label_ticklabels = cluster_label_order

colors_in_order = [cMapDict[i] for i in cluster_label_order]
cMapDict = dict(zip(cluster_label_order, colors_in_order))

clustering_metadata.obs['protein_leiden'] = [change_cluster_names_dict[i] for i in clustering_metadata.obs['protein_leiden']]

clustering_metadata.obs['protein_leiden'] = pd.Categorical(clustering_metadata.obs['protein_leiden'], categories=cluster_label_order, ordered=False)



#--------------------------------------------------------------------------------------




mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs['leiden']))
merged_adata.obs['leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['leiden']

mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs['protein_leiden']))
merged_adata.obs['protein_leiden'] = [mapping_dict[i] for i in merged_adata.obs.id]
merged_adata.obs['file_protein_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['protein_leiden']


# also map distance metadata for later use 

distance_metrics = ['custom_distance_to_nearest_vasculature', 'custom_distance_to_nearest_ApoE', 'custom_distance_to_nearest_aBeta',
       'fabseg_distance_to_nearest_Astrocytes',
       'fabseg_distance_to_nearest_Neurons',
       'fabseg_distance_to_nearest_Oligodendrocytes',
       'fabseg_distance_to_nearest_dense_aBeta',
       'fabseg_distance_to_nearest_diffuse_aBeta',
        'fabseg_distance_to_nearest_dense_aBetadiffuse_aBeta']

for met in distance_metrics:
    mapping_dict = dict(zip(clustering_metadata.obs['ImageID_CellID'], clustering_metadata.obs[met]))
    merged_adata.obs[met] = [mapping_dict[i] for i in merged_adata.obs.id]
#merged_adata.obs['file_protein_leiden'] = merged_adata.obs['origfile'] + '.' + merged_adata.obs['protein_leiden']



# remove any meningeal macrophages

merged_adata = merged_adata[merged_adata.obs['parent'].isin(['Grey Matter', 'White Matter'])]

# remove any default clusters

merged_adata = merged_adata[merged_adata.obs['leiden'] != 'default']

merged_adata_WM = merged_adata[merged_adata.obs['parent'] == 'White Matter']
merged_adata_GM = merged_adata[merged_adata.obs['parent'] == 'Grey Matter']






merged_adata_sub = merged_adata[merged_adata.obs['id'].isin(clustering_metadata.obs['ImageID_CellID'])]
clustering_metadata.obs.index = clustering_metadata.obs['ImageID_CellID']

clustering_metadata = clustering_metadata[merged_adata_sub.obs['id'], :]

tsne_coordinates = clustering_metadata.obsm["X_tsne"]
merged_adata_sub.obsm['X_tsne'] = tsne_coordinates
protein_tsne_coordinates = clustering_metadata.obsm['X_tsne_PROTEIN']
merged_adata_sub.obsm['X_tsne_PROTEIN'] = protein_tsne_coordinates

#merged_adata.obsm.X_tsne = 

for feat in merged_adata_sub.var_names:
    df = merged_adata_sub.to_df()
    #feat_upperbound = np.percentile(a=df[feat], q=[80])

    sc.pl.tsne(merged_adata_sub, color=feat, layer=layer)
    sc.pl.scatter(merged_adata_sub, color = feat, basis = 'tsne_PROTEIN', layers = layer)
    sc.pl.violin(merged_adata_sub, feat, groupby = 'leiden', layer=layer)
    sc.pl.violin(merged_adata_sub, feat, groupby = 'protein_leiden', layer = layer)
    
    



## add protein and morph data to object 

In [ ]:
# add morph data

merged_adata_sub.obs.index = merged_adata_sub.obs.id
morph_data_raw = clustering_metadata.to_df(layer='unscaled_morph_data').loc[merged_adata_sub.obs.id]


protein_data_orig = sc.read_h5ad('../final_h5ads/fabian_metadata_plus_clustering.h5ad')

protein_data_orig_sub = protein_data_orig[protein_data_orig.obs.ImageID_CellID.isin(merged_adata_sub.obs.id)]
protein_data_raw = protein_data_orig_sub.to_df()[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']].loc[merged_adata_sub.obs.index]

merged_adata_sub.obs = pd.concat([merged_adata_sub.obs, morph_data_raw, protein_data_raw], axis = 1)


#matrix1 = protein_data_orig_sub.to_df()[['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']]


#merged_adata_sub.X = pd.concat([merged_adata_sub.to_df(), clustering_metadata.to_df().loc[merged_adata_sub.obs.id]], axis=0)

sc.pl.scatter(merged_adata_sub, color = 'CD163', basis = 'tsne_PROTEIN')


morph_features = list(morph_data_raw.columns)
protein_features = list(protein_data_raw.columns)

In [ ]:

sc.pl.matrixplot(merged_adata_sub, groupby = 'protein_leiden', var_names=['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45'], standard_scale='var')

In [ ]:

for cluster in clustering_metadata.obs.leiden.unique():
    sub_obj = clustering_metadata[(clustering_metadata.obs.leiden == cluster) & (clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')]
    sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='leiden')
    
sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne_PROTEIN', color='leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne_PROTEIN', color='protein_leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne', color='leiden')

sc.pl.scatter(clustering_metadata[(clustering_metadata.obs.Parent == 'Grey Matter') & (clustering_metadata.obs.ImageType == 'AD')], basis='tsne', color='protein_leiden', size=20)


In [ ]:
cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}


# among ramified 2 cells, which are associated with Abeta, and what protein clusters do they belong to?

Abeta_pct_cutoff = 0.5
layer = 'zscore'

# Abeta_pct_cutoff = 0.1
# layer = None
from sklearn.cluster import DBSCAN

merged_adata_sub.obs['ABETA_percent'] = merged_adata_sub.to_df(layer=layer)['A-Beta']
merged_adata_sub.obs['beats_abeta_thresh'] = merged_adata_sub.obs.ABETA_percent > Abeta_pct_cutoff

sub_obj = merged_adata_sub


plt.hist(sub_obj.obs['ABETA_percent'])
plt.show()
sns.kdeplot(sub_obj.obs['ABETA_percent'])
plt.show()

print('what percent of mircroglia in GM AD are nearby Abeta?')
print(sum(sub_obj.obs.ABETA_percent > Abeta_pct_cutoff)/len(sub_obj.obs.ABETA_percent))

#sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='beats_abeta_thresh', legend_loc='on data')


sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='leiden', show=False, color_map=cMapDict_morph)


plt.savefig('../figures/neighborhood_exploration/morph_clusters_on_protein_tsne.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/morph_clusters_on_protein_tsne.svg', format='svg')


for var in sub_obj.var_names:
    sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color=var, show = False, layers='zscore')
    
    plt.savefig('../figures/neighborhood_exploration/' + var + '_scaled_featureplot.tiff', format='tiff')
    plt.savefig('../figures/neighborhood_exploration/' + var + '_scaled_featureplot.svg', format='svg')


coordinates = sub_obj.obsm['X_tsne_PROTEIN']
coordsx = [i[0] for i in coordinates]
coordsy = [i[1] for i in coordinates]

coordinates_dframe = pd.DataFrame({'x':coordsx, 'y':coordsy})
#hdb = DBSCAN(min_samples=20, eps = 3)
hdb = DBSCAN(min_samples=80, eps=3.6)
hdb.fit(coordinates_dframe)

sub_obj.obs['dblabels'] = hdb.labels_
sub_obj.obs['dblabels'] = pd.Categorical(sub_obj.obs['dblabels'])


sc.pl.scatter(sub_obj, basis='tsne_PROTEIN', color='dblabels', legend_loc='on data')








# coordinates = only_nearby_abeta.obsm['X_tsne_PROTEIN']
# coordsx = [i[0] for i in coordinates]
# coordsy = [i[1] for i in coordinates]

# coordinates_dframe = pd.DataFrame({'x':coordsx, 'y':coordsy})
# #hdb = DBSCAN(min_samples=20, eps = 3)
# hdb = DBSCAN(min_samples=20, eps = 3.5)
# hdb.fit(coordinates_dframe)

# only_nearby_abeta.obs['dblabels'] = hdb.labels_
# only_nearby_abeta.obs['dblabels'] = pd.Categorical(only_nearby_abeta.obs['dblabels'])


# sc.pl.scatter(only_nearby_abeta, basis='tsne_PROTEIN', color='dblabels', legend_loc='on data')





### cluster neighbor data

In [ ]:


cmapDict_spatial  = {
        "0": "#1f77b4",
        "1": "#c49c94",
        "2": "#17becf",
        "3": "#d62728",
        "4": "#9467bd",
        "5": "#8c564b",
        "6": "#e377c2",
        "7": "#7f7f7f",
        "8": "#bcbd22",
        "9": "#2ca02c",
        "10": "#aec7e8",
        "11": "#ffbb78",
        "12": "#98df8a",
        "13": "#ff9896",
        "14": "#c5b0d5",
        "15": "#ff7f0e",
        "16": "#f7b6d2"
    }




from scipy.stats import zscore
import umap
import umap.plot

orig_df = sub_obj.to_df()
zscore_within_samples = sub_obj.to_df().groupby(sub_obj.obs['origfilenum']).transform(func = zscore, axis = 0)

zscore_all = zscore_within_samples.transform(func = zscore, axis = 0)


# perform umap dimreduction
embedding = umap.UMAP().fit(zscore_all)
print(umap.plot.points(embedding))

sub_obj.obsm['X_neighbor_umap'] = coordinates = embedding.embedding_
sub_obj.layers['neighbor_zscore_new'] = zscore_all
sub_obj.obsm['zscore_all'] = zscore_all
plt.show()


for var in sub_obj.var_names:
    

    sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, layers='zscore', show = False)
    plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '.tiff', format='tiff')
    plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '.svg', format='svg')
    
    sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, show = False)
    # plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '_unscaled.tiff', format='tiff')
    # plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_' + var + '_unscaled.svg', format='svg')
    
    #sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color=var, layers='neighbor_zscore_new')

    
    
sc.pp.neighbors(sub_obj, random_state=10, use_rep='zscore_all')

sc.tl.leiden(sub_obj, key_added='spatial_clustering', resolution=1.5)
sub_obj.obs['spatial_clustering'] = sub_obj.obs['spatial_clustering'].astype('category')

sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = 'spatial_clustering', legend_loc='on data', show = False)
#sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = 'spatial_clustering', legend_loc='on data', show = False)

plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap.svg', format='svg')


sc.pl.scatter(sub_obj, basis = 'tsne_PROTEIN', color = 'spatial_clustering', show = False)
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_protein.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_protein.svg', format='svg')





# make custom plot:



sc.pl.scatter(sub_obj, basis = 'tsne', color = 'spatial_clustering', show = False)
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_morph.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_morph.svg', format='svg')







# for clus in sub_obj.obs.spatial_clustering.unique():
#     sub_obj_clus = sub_obj[sub_obj.obs.spatial_clustering == clus]
#     sc.pl.scatter(sub_obj_clus, basis = 'tsne_PROTEIN', color = 'spatial_clustering', legend_loc='on data', show=False)
#     plt.axis([-100, 100, -100, 100])
#     plt.show()

# hm = sc.pl.matrixplot(sub_obj, groupby = 'spatial_clustering', layer='zscore', var_names=sub_obj.var_names, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.svg', format='svg')

# hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=protein_features, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.svg', format='svg')

# hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=morph_features, standard_scale='var', return_fig=True)
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.tiff', format='tiff')
# hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.svg', format='svg')






miniclus = copy.deepcopy(sub_obj)


In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

sc.settings.set_figure_params(dpi_save=300, facecolor='white',figsize=(5,4))
sns.set(font="Arial")

sns.set_style('white')




sub_obj_2 = sub_obj.copy()

colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)


neighbor_percents = sub_obj_2.to_df()

upperlims = []

for i,n in enumerate(neighbor_percents.columns):
    nvals = neighbor_percents[n]
    upperlim = np.percentile(nvals, 99)
    
    upperlims.append(upperlim)

sub_obj_2.obsm['X_umap'] = sub_obj_2.obsm['X_neighbor_umap']
sc.pl.umap(sub_obj_2, color=sub_obj_2.var_names, vmax=upperlims, show=False, color_map=cmap)

plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling_EASYEDIT.tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling_EASYEDIT.svg', format = 'svg')

In [ ]:
colors = [(0.9, 0.9, 0.9), (0, 0, 1)]  # White to blue
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

xvals = [i[0] for i in sub_obj.obsm['X_neighbor_umap']]
yvals = [i[1] for i in sub_obj.obsm['X_neighbor_umap']]
neighbor_percents = sub_obj.to_df()

fig, axs = plt.subplots(3, 4, figsize = (21,10))


for i,n in enumerate(neighbor_percents.columns):
    nvals = neighbor_percents[n]
    upperlim = np.percentile(nvals, 99)
    
    xpos = i//4
    ypos = i%4

    im = axs[xpos,ypos].scatter(x = xvals, y = yvals, c=neighbor_percents[n], cmap=cmap, s=1, vmax=upperlim)
    plt.colorbar(im, ax = axs[xpos,ypos])
    axs[xpos,ypos].set_title(n)
    axs[xpos,ypos].set_xticks([])
    axs[xpos,ypos].set_yticks([])
    axs[xpos,ypos].set_xlabel('UMAP1')
    axs[xpos,ypos].set_ylabel('UMAP2')


    # axs[0,i].xlabel('tSNE1')
    # plt.ylabel('tSNE2')
    # plt.title(n)
    # plt.show()
plt.tight_layout()
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.svg', format = 'svg')
plt.show()

# sc.pl.tsne(clustering_adata, color=adataScaled.var_names,use_raw=False, return_fig = True, color_map=cmap, show = False)
# plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.tiff')
# plt.savefig('../figures/neighborhood_exploration/nhood_marker_feature_plot_bluewhite_noscaling.svg', format = 'svg')
# sc.pl.scatter(sub_obj, basis = 'neighbor_umap', color = sub_obj.var_names)


In [ ]:

hm = sc.pl.matrixplot(sub_obj, groupby = 'spatial_clustering', layer='zscore', var_names=sub_obj.var_names, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap.svg', format='svg')

hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=protein_features, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_protein.svg', format='svg')

hm = sc.pl.matrixplot(sub_obj, groupby='spatial_clustering', var_names=morph_features, standard_scale='var', return_fig=True)
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.tiff', format='tiff')
hm.savefig('../figures/neighborhood_exploration/nhood_clusters_heatmap_morph.svg', format='svg')

#KILLME




        
# test = custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING.tiff')
# fjdlf

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_ALT_SCALING.svg')

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=protein_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_protein_ALT_SCALING.svg')

custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_ALT_SCALING.tiff')
custom_scaled_heatmap(sub_obj, group_by='spatial_clustering', layer = 'meta', var_names=morph_features, saveto='../figures/neighborhood_exploration/nhood_clusters_heatmap_morph_ALT_SCALING.svg')









In [ ]:

ax = sc.pl.scatter(sub_obj, color=["spatial_clustering"], groups=["1", "36"], show=False, basis = 'tsne_PROTEIN')

# We can change the 'NA' in the legend that represents all cells outside of the
# specified groups
legend_texts = ax.get_legend().get_texts()
# Find legend object whose text is "NA" and change it
for legend_text in legend_texts:
    if legend_text.get_text() == "NA":
        legend_text.set_text("other cell types")
        
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_only_3clusters.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_umap_only_3clusters.svg', format='svg')


In [ ]:

# what is an appropriate cutoff for tmem?

cd163_positive_tmem119_negative_ids = ['3155_1053', '3155_2811', '3155_538', '3196_5178', '3196_5305', '3196_172', '3196_6796']

only_spec_cells = sub_obj.obs[sub_obj.obs.id.isin(cd163_positive_tmem119_negative_ids)]
print('Cd163 scores (all are expressing cd163)')
print(only_spec_cells['CD163'])

print('TMEM119 scores (all are not expressing tmem119)')
print(only_spec_cells['TMEM119'])


In [ ]:
# what percentage of BLM GM AD cells express TMEM119, CD163, or both?

lower_cd163_bound = np.percentile(sub_obj.obs['CD163'], 10)

sub_obj.obs['CD163 > 0'] = sub_obj.obs['CD163'] > 120
sub_obj.obs['TMEM119 > 0'] = sub_obj.obs['TMEM119'] > 263

sc.pl.scatter(sub_obj, basis = 'tsne_PROTEIN', color = 'CD163 > 0')
sc.pl.scatter(sub_obj, basis = 'tsne_PROTEIN', color = 'TMEM119 > 0', show=False)

plt.savefig('../figures/neighborhood_exploration/TMEM_percent.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/TMEM_percent.svg', format='svg')


only_BLM_sub_obj = sub_obj[sub_obj.obs.protein_leiden == 'BLM']

sc.pl.scatter(only_BLM_sub_obj, basis = 'tsne_PROTEIN', color = 'TMEM119 > 0', show=False)

plt.savefig('../figures/neighborhood_exploration/TMEM_percent_onlyBLM.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/TMEM_percent_onlyBLM.svg', format='svg')

print('minimum percentage of cells in BLM that absolutely do not express TMEM119')
print(collections.Counter(only_BLM_sub_obj.obs['TMEM119 > 0']))

print((sum(only_BLM_sub_obj.obs['TMEM119 > 0'] == False)/len(only_BLM_sub_obj.obs['TMEM119 > 0'])) * 100)

only_BLM_sub_obj.obs['TMEM119 > 0'] = only_BLM_sub_obj.obs['TMEM119 > 0'].astype('category')

sc.pl.violin(only_BLM_sub_obj, 'A-Beta', groupby = 'TMEM119 > 0', layer='minmax')


In [ ]:
cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}

sub_obj_withcontrol = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter')]
sub_obj_withcontrol.obs['combined_column'] = sub_obj_withcontrol.obs['AD_CNT'].astype(str) + sub_obj_withcontrol.obs['beats_abeta_thresh'].astype(str)

translator_dict = {
    'CNTTrue':'Control',
    'CNTFalse':'Control',
    'ADTrue':'AD + high A-beta',
    'ADFalse':'AD + low A-beta'

}


sub_obj_withcontrol.obs['combined_column_translated'] = [translator_dict[i] for i in sub_obj_withcontrol.obs['combined_column']]


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': sub_obj_withcontrol.obs['combined_column_translated'],
    'Category2': sub_obj_withcontrol.obs['leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. Protein Expression clustering')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.savefig('../figures/neighborhood_exploration/adlevels_morph_barchart.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/adlevels_morph_barchart.svg', format='svg')

print(normalized_data)


# also split by population

for protein_cluster in sub_obj_withcontrol.obs.protein_leiden.unique():
    
    

    # Sample data as a pandas DataFrame
    data = pd.DataFrame({
        'Category1': sub_obj_withcontrol[sub_obj_withcontrol.obs.protein_leiden == protein_cluster].obs['combined_column_translated'],
        'Category2': sub_obj_withcontrol[sub_obj_withcontrol.obs.protein_leiden == protein_cluster].obs['leiden']
    })

    # Group the data by Category1 and count occurrences of Category2
    grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

    # Normalize the data to make it proportional
    normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

    # Create the proportional stacked bar chart
    ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

    # Set labels and title
    plt.xlabel('Proportion')
    plt.ylabel('Protein expression clusters')
    plt.title('Morphology vs. Protein Expression clustering_' + protein_cluster)
    
    

    # Show the legend
    plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

    print('---------------')
    print(protein_cluster)
    print(normalized_data)



In [ ]:
# order mini clusters by abeta neighbors

only_abeta_scores = miniclus.to_df(layer='zscore')['A-Beta']
abeta_score_df = pd.DataFrame({'cluster': miniclus.obs.spatial_clustering, 'abeta':only_abeta_scores})
abeta_score_df_bulked = abeta_score_df.groupby('cluster').mean()
abeta_score_df_bulked = abeta_score_df_bulked.sort_values(by = 'abeta', ascending=False)

#miniclus.obs['spatial_clustering'] = pd.Categorical(miniclus.obs['spatial_clustering'], categories=list(abeta_score_df_bulked.index))


sc.pl.matrixplot(miniclus, groupby='spatial_clustering', var_names=miniclus.var_names, layer='zscore', standard_scale='var')
sc.pl.matrixplot(miniclus, groupby='spatial_clustering', var_names=miniclus.var_names, layer='zscore')

sc.pl.matrixplot(miniclus, groupby='spatial_clustering', var_names=protein_features, standard_scale='var')
sc.pl.matrixplot(miniclus, groupby='spatial_clustering', var_names=morph_features, standard_scale='var')



In [ ]:
cMapDict_morph = {
    'Rounded':'#aa40fc',
    'Intermediate':'#8c564b',
    'Ramified 1':'#aec7e8',
    'Ramified 2':'palevioletred',
    'Ramified 3':'#ffbb78'
}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': miniclus.obs['spatial_clustering'],
    'Category2': miniclus.obs['leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)
normalized_data = normalized_data.loc[normalized_data.index[::-1]]

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict_morph)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. sub-clusters')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.savefig('../figures/neighborhood_exploration/nhood_clusters_morph_barchart.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_morph_barchart.svg', format='svg')






In [ ]:
# is there a correlation between ram 2 and abeta percent? 




for morph_type in normalized_data.columns:
    print(morph_type)
    
    
    data1 = abeta_score_df_bulked['abeta']
    data2 = normalized_data.loc[abeta_score_df_bulked.index][morph_type]
    
    coefficients1 = np.polyfit(data1, data2, 1)
    poly1 = np.poly1d(coefficients1)

    plt.scatter(data1, data2)
    plt.xlabel('Abeta zscore')
    plt.ylabel(morph_type + ' proportion')
    plt.plot(data1, poly1(data1), color='blue', linestyle='-', label='Line of Best Fit (Dataset 1)')

    
    plt.show()
    
    print(data1.corr(data2, method='spearman'))
    print('-------------------')

In [ ]:
cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}

sub_obj_withcontrol = merged_adata_sub[(merged_adata_sub.obs.parent == 'Grey Matter')]
sub_obj_withcontrol.obs['combined_column'] = sub_obj_withcontrol.obs['AD_CNT'].astype(str) + sub_obj_withcontrol.obs['beats_abeta_thresh'].astype(str)

translator_dict = {
    'CNTTrue':'Control',
    'CNTFalse':'Control',
    'ADTrue':'AD + high A-beta',
    'ADFalse':'AD + low A-beta'

}


sub_obj_withcontrol.obs['combined_column_translated'] = [translator_dict[i] for i in sub_obj_withcontrol.obs['combined_column']]


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': sub_obj_withcontrol.obs['combined_column_translated'],
    'Category2': sub_obj_withcontrol.obs['protein_leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Morphology vs. Protein Expression clustering')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.savefig('../figures/neighborhood_exploration/adlevels_protein_barchart.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/adlevels_protein_barchart.svg', format='svg')

normalized_data

In [ ]:
cMapDict = {'Microglia 1': '#e377c2',
 'Microglia 2': '#b5bd61',
 'Monocytes': '#d62728',
 'PVM': '#17becf',
 'BLM': '#ff7f0e',
 'default': '#8c564b'
}


import pandas as pd
import matplotlib.pyplot as plt

# Sample data as a pandas DataFrame
data = pd.DataFrame({
    'Category1': miniclus.obs['spatial_clustering'],
    'Category2': miniclus.obs['protein_leiden']
})

# Group the data by Category1 and count occurrences of Category2
grouped_data = data.groupby('Category1')['Category2'].value_counts().unstack(fill_value=0)

# Normalize the data to make it proportional
normalized_data = grouped_data.div(grouped_data.sum(axis=1), axis=0)
normalized_data = normalized_data.loc[normalized_data.index[::-1]]

# Create the proportional stacked bar chart
ax = normalized_data.plot(kind='barh', stacked=True, figsize=(8, 4), color = cMapDict)

# Set labels and title
plt.xlabel('Proportion')
plt.ylabel('Protein expression clusters')
plt.title('Protein vs. sub-clusters')

# Show the legend
plt.legend(title='Morph clusters', bbox_to_anchor=(1.05, 1), loc='upper left')


plt.savefig('../figures/neighborhood_exploration/nhood_clusters_protein_barchart.tiff', format='tiff')
plt.savefig('../figures/neighborhood_exploration/nhood_clusters_protein_barchart.svg', format='svg')

In [ ]:

for protein_type in normalized_data.columns:
    print(protein_type)
    
    
    data1 = abeta_score_df_bulked['abeta']
    data2 = normalized_data.loc[abeta_score_df_bulked.index][protein_type]
    
    coefficients1 = np.polyfit(data1, data2, 1)
    poly1 = np.poly1d(coefficients1)

    plt.scatter(data1, data2)
    plt.xlabel('Abeta zscore')
    plt.ylabel(protein_type + ' proportion')
    plt.plot(data1, poly1(data1), color='blue', linestyle='-', label='Line of Best Fit (Dataset 1)')

    plt.show()
    
    print(data1.corr(data2, method='spearman'))
    print('-------------------')





#### save object with neighborhood clusters

In [ ]:
#sub_obj.write_h5ad('sub_obj_with_spatial_clustering.h5ad')